# p003: 0.937からの次の比較候補

haideptry **V2** の推論ソースを保持し、全4 CoAt系統・生確率・統合方式を追加検査する未実行候補です。
作者Public **0.943** は当方の実測値ではありません。0.945/0.955への到達は未確認です。
過去の採点時Input版は不明であり、今回固定した版・hashの組合せとして比較します。

新しいprivate NotebookにImportし、**GPU T4 x2 / Internet OFF** を選択します。
下記14 Inputsだけを指定版で追加し、Save Version → Save & Run All。元のp002は保持します。
すべてのセルを新しいセッションで順番に実行してください。
最後に **P003 READY FOR MANUAL SUBMISSION** と **P003_READY.json** を確認し、
**submission.csv** を手動提出します。停止した場合は提出せずログを確認してください。
表示用の少数test成功は、隠しtestの9時間完走や精度向上の証拠ではありません。

この候補はA5 BF16を原版のまま保持します。別添のA5精度診断による変更を混ぜません。
演算のFP32再試行は原版どおり記録し、途中失敗による定数埋め・モデル省略は拒否します。
ラベルによる学習・係数選択・自動アップロード・自動提出は行いません。

## Inputs

- [dreaddevelopment/raptor-knee-maxspan](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-maxspan/versions/1) — Version 1
- [dreaddevelopment/raptor-knee-native384](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-native384/versions/1) — Version 1
- [dreaddevelopment/raptor-knee-native384dense](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-native384dense/versions/1) — Version 1
- [mattiaangeli/knee-mri-fold-weights](https://www.kaggle.com/datasets/mattiaangeli/knee-mri-fold-weights/versions/2) — Version 2
- [mattiaangeli/opencv-python-headless-4120088-x86](https://www.kaggle.com/datasets/mattiaangeli/opencv-python-headless-4120088-x86/versions/1) — Version 1
- [marwanmath/resnet-50-radimagenet-marwan](https://www.kaggle.com/datasets/marwanmath/resnet-50-radimagenet-marwan/versions/1) — Version 1
- [mattiaangeli/rsna-knee-coat-resgated-ep10-top3](https://www.kaggle.com/datasets/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/versions/3) — Version 3
- [mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2](https://www.kaggle.com/datasets/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/versions/2) — Version 2
- [mattiaangeli/rsna-knee-coatnet-global96-top3](https://www.kaggle.com/datasets/mattiaangeli/rsna-knee-coatnet-global96-top3/versions/1) — Version 1
- [antoinegg1/rsna-knee-e9-radimagenet-heads-v15](https://www.kaggle.com/datasets/antoinegg1/rsna-knee-e9-radimagenet-heads-v15/versions/3) — Version 3
- [pilkwang/rsna-knee-weights](https://www.kaggle.com/datasets/pilkwang/rsna-knee-weights/versions/1) — Version 1
- [sofiaanjenje/rsna-knee-e13-train](https://www.kaggle.com/code/sofiaanjenje/rsna-knee-e13-train) — Version 1
- [metaresearch/dinov2/PyTorch/small/1](https://www.kaggle.com/models/metaresearch/dinov2/frameworks/PyTorch/variations/small/versions/1) — Version 1
- [rsna-knee-abnormality-detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/data) — Version None


In [ ]:
import builtins as _p003_builtins
import types as _p003_types
import sys as _p003_sys
if hasattr(_p003_builtins, '_rsna_p003_guard'):
    raise RuntimeError('Restart with a fresh session')
_p003_module = _p003_types.ModuleType('rsna_p003_guard')
_p003_sys.modules['rsna_p003_guard'] = _p003_module
exec(compile('"""Strict execution envelope for the frozen haideptry V2 candidate.\n\nOnly the manually executed Kaggle notebook runs inference. This module does not\ndownload data, fit coefficients, train, upload, or submit anything.\n"""\n\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport linecache\nimport os\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nLABELS = [\n    "ACL",\n    "MCL",\n    "Medial Meniscus",\n    "Lateral Meniscus",\n    "Medial OA",\n    "Lateral OA",\n    "PF OA",\n    "Effusion",\n    "Synovitis",\n    "Baker\'s",\n    "Contusion",\n    "Fracture",\n]\nMEMBERS = ("resgated_top3", "global96_top3", "d4_swa3", "repairv1_top3")\nPROB_FILES = (\n    "coat_resgated_ep10_top3_predictions.npz",\n    "g96_input/coatnet_global96_top3_predictions.npz",\n    "d4_input/coatnet_d4_depthzone_swa3_predictions.npz",\n    "coatnet_repairv1_top3_predictions.npz",\n)\nRECEIPT_FILES = (\n    "_coat_arm_receipt.json",\n    "_coat_g96_receipt.json",\n    "d4_input/d4.receipt.json",\n    "_coat_repairv1_receipt.json",\n)\nRECEIPT_STATUS = (\n    "VALID_COAT_RESGATED_EP10_TOP3_RANK_SUBMISSION",\n    "VALID_COATNET_GLOBAL96_TOP3_T4X2_SUBMISSION",\n    "VALID_COATNET_D4_DEPTHZONE_SWA3_T4X2_SUBMISSION",\n    "VALID_COATNET_REPAIRV1_TOP3_PROBABILITY_RANK_SUBMISSION",\n)\nOUTER = dict.fromkeys(LABELS, 0.60)\nOUTER.update({"ACL": 0.75, "Medial Meniscus": 0.80, "Lateral Meniscus": 1.0, "Lateral OA": 0.75, "Fracture": 0.75})\nALLOWED_EVENTS = {\n    "dino_shared_path_parity",\n    "raptor_checkpoint",\n    "dino_fp16_nonfinite_retry_fp32",\n    "dino_legacy_fp16_nonfinite_retry_fp32",\n    "raptor_fp16_nonfinite_retry_fp32",\n    "a5_fp16_nonfinite_retry_fp32",\n    "rad_fp16_nonfinite_retry_fp32",\n    "rad_head_fp16_nonfinite_retry_fp32",\n    "cache_complete",\n    "scratch_allocation",\n    "scratch_fallback",\n    "a5_no_metadata",\n    "a5_empty_study",\n    "empty_study_rows",\n}\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(8 << 20), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef save_json(path, value):\n    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n", encoding="utf-8")\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef read_scores(path, ids):\n    import numpy as np\n\n    with Path(path).open(encoding="utf-8-sig", newline="") as stream:\n        rows = list(csv.reader(stream))\n    if not rows or rows[0] != ["StudyInstanceUID", *LABELS]:\n        raise ValueError(f"Wrong label order: {path}")\n    if any(len(row) != 13 for row in rows[1:]) or [row[0] for row in rows[1:]] != ids:\n        raise ValueError(f"Wrong UID order/coverage: {path}")\n    return probabilities(np.asarray([row[1:] for row in rows[1:]], dtype=np.float64), (len(ids), 12))\n\n\ndef probabilities(value, shape):\n    import numpy as np\n\n    value = np.asarray(value)\n    if value.shape != shape or not np.isfinite(value).all() or (value < 0).any() or (value > 1).any():\n        raise ValueError(f"Invalid raw predictions: expected {shape}, got {value.shape}")\n    return value\n\n\ndef aligned_npz(path, ids, key="values", labels_required=True):\n    import numpy as np\n\n    with np.load(path, allow_pickle=False) as data:\n        uids = data["study_uids"].astype(str).tolist()\n        if len(uids) != len(ids) or len(set(uids)) != len(uids) or set(uids) != set(ids):\n            raise ValueError(f"Raw UID coverage mismatch: {path}")\n        if (labels_required or "labels" in data) and data["labels"].astype(str).tolist() != LABELS:\n            raise ValueError(f"Raw label order mismatch: {path}")\n        if key == "probability_mean" and key not in data:\n            raw = np.asarray(data["raw_probabilities"], dtype=np.float64)\n            probabilities(raw, (3, len(ids), 12))\n            value = raw.mean(axis=0)\n        else:\n            value = np.asarray(data[key], dtype=np.float64)\n    value = probabilities(value, (len(ids), 12))\n    order = {uid: index for index, uid in enumerate(uids)}\n    return value[[order[uid] for uid in ids]]\n\n\ndef rank_pct(values):\n    """Average ties over the complete cohort, never separately by GPU/chunk."""\n    import numpy as np\n\n    values = np.asarray(values, dtype=np.float64)\n    if values.ndim != 2 or not len(values) or not np.isfinite(values).all():\n        raise ValueError("Rank input must be a nonempty finite matrix")\n    result = np.empty_like(values)\n    n = len(values)\n    for col in range(values.shape[1]):\n        order = np.argsort(values[:, col], kind="stable")\n        start = 0\n        while start < n:\n            end = start + 1\n            while end < n and values[order[end], col] == values[order[start], col]:\n                end += 1\n            result[order[start:end], col] = ((start + 1 + end) / 2) / n\n            start = end\n    return result\n\n\ndef replay_final(parent, raptor, coats):\n    """Frozen V2 coefficients; callers must align the full cohort first."""\n    import numpy as np\n\n    if len(coats) != 4:\n        raise ValueError("All four CoAt families are required")\n    shape = parent.shape\n    for value in (parent, raptor, *coats):\n        probabilities(value, shape)\n    family = rank_pct(sum(coats) / 4.0)\n    hybrid = 0.6 * rank_pct(raptor) + 0.4 * family\n    weights = np.asarray([OUTER[label] for label in LABELS])\n    return rank_pct((1 - weights) * rank_pct(parent) + weights * hybrid), family\n\n\ndef check_blend_receipt(receipt, n):\n    expected = {\n        "members": list(MEMBERS),\n        "study_count": n,\n        "inner_rerank": False,\n        "private_alpha": 0.4,\n        "public_raptor_alpha": 0.6,\n        "within_coat": dict.fromkeys(MEMBERS, 0.25),\n        "finding_specific_weights": False,\n        "family_reduction": "rank_of_member_probability_mean",\n    }\n    for key, value in expected.items():\n        if receipt.get(key) != value or (isinstance(value, bool) and receipt.get(key) is not value):\n            raise ValueError(f"CoAt composition changed: {key}")\n\n\nclass P003Guard:\n    def __init__(self, contract, work="/kaggle/working", input_root="/kaggle/input"):\n        self.contract = contract\n        self.work, self.input_root = Path(work), Path(input_root)\n        self.started = time.monotonic()\n        self.completed, self.timings = [], []\n        self.ids, self.mounts, self.input_files = [], {}, []\n        self.ready = False\n        self.owns_outputs = False\n\n    def prepare_files(self):\n        self.work.mkdir(parents=True, exist_ok=True)\n        # Source rsna_initialize removes its old output. Refuse before it can do so.\n        notebook_runtime_files = {".virtual_documents", "__notebook__.ipynb", "__notebook_source__.ipynb"}\n        if any(path.name not in notebook_runtime_files for path in self.work.iterdir()):\n            raise RuntimeError("Use a fresh Kaggle saved session with an empty /kaggle/working")\n        for entry in self.contract["inputs"]:\n            roots = set()\n            for relative in entry["mounts"]:\n                path = (self.input_root / relative).resolve()\n                path.relative_to(self.input_root.resolve())\n                if path.is_dir():\n                    roots.add(path)\n            if len(roots) != 1:\n                raise RuntimeError(f"Attach one pinned Input: {entry[\'ref\']} version {entry[\'version\']}")\n            root = roots.pop()\n            self.mounts[entry["key"]] = root\n            for item in entry["files"]:\n                path = (root / item["path"]).resolve()\n                path.relative_to(root)\n                if not path.is_file() or (item.get("bytes") is not None and path.stat().st_size != item["bytes"]):\n                    raise ValueError(f"Missing/wrong-size asset: {entry[\'key\']}/{item[\'path\']}")\n                digest = sha256(path)\n                if item.get("sha256") and digest != item["sha256"]:\n                    raise ValueError(f"Input SHA-256 mismatch: {entry[\'key\']}/{item[\'path\']}")\n                self.input_files.append(\n                    {\n                        "input": entry["key"],\n                        "path": item["path"],\n                        "sha256": digest,\n                        "expected_hash_verified": bool(item.get("sha256")),\n                    }\n                )\n        # Reject undeclared mounts that could shadow the source\'s filename resolver.\n        declared = set(self.mounts.values())\n        pending = [self.input_root.resolve()]\n        while pending:\n            for child in pending.pop().iterdir():\n                if not child.is_dir():\n                    continue\n                resolved = child.resolve()\n                if resolved in declared:\n                    continue\n                if any(mount.is_relative_to(resolved) for mount in declared):\n                    pending.append(resolved)\n                else:\n                    raise ValueError(f"Remove undeclared Input: {child}")\n        competition = self.mounts["competition"]\n        for name in ("train.csv", "test.csv", "test_series.csv", "sample_submission.csv"):\n            if not (competition / name).is_file():\n                raise FileNotFoundError(name)\n        if not (competition / "test_series").is_dir():\n            raise RuntimeError("Missing test_series; refuse train fallback")\n        with (competition / "test.csv").open(encoding="utf-8-sig", newline="") as f:\n            self.ids = [row["StudyInstanceUID"] for row in csv.DictReader(f)]\n        if not self.ids or any(not uid for uid in self.ids) or len(set(self.ids)) != len(self.ids):\n            raise ValueError("Invalid test UID set")\n        read_scores(competition / "sample_submission.csv", self.ids)\n        self.owns_outputs = True\n\n    def prepare(self):\n        self.prepare_files()\n        if Path.cwd().resolve() != self.work.resolve():\n            raise RuntimeError("Execute in /kaggle/working")\n        for name in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE", "HF_DATASETS_OFFLINE", "PIP_NO_INDEX"):\n            os.environ[name] = "1"\n        # Preserve the source\'s cohort-size scheduling branch; reject inherited overrides.\n        if "RSNA_PARALLEL_COAT_READERS" in os.environ or "SLOT_SCHEME" in os.environ:\n            raise RuntimeError("Remove scheduling/slot overrides; this is the frozen V2 recipe")\n        wheel = self.mounts["knee_mri_fold_weights"] / "timm-1.0.22-py3-none-any.whl"\n        try:\n            installed = importlib.metadata.version("timm")\n        except importlib.metadata.PackageNotFoundError:\n            installed = None\n        if installed != "1.0.22":\n            if "timm" in sys.modules:\n                raise RuntimeError("Restart before pinning timm")\n            subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", str(wheel)], check=True)\n        import torch\n\n        if torch.cuda.device_count() != 2 or any("T4" not in torch.cuda.get_device_name(i) for i in range(2)):\n            raise RuntimeError("Select GPU T4 x2")\n        self.ready = True\n        save_json(\n            self.work / "P003_PREFLIGHT.json",\n            {\n                "status": "passed",\n                "inputs": self.input_files,\n                "candidate_id": self.contract["candidate_id"],\n                "studies": len(self.ids),\n                "environment": {\n                    "python": sys.version,\n                    "torch": torch.__version__,\n                    "cuda": torch.version.cuda,\n                    "timm": importlib.metadata.version("timm"),\n                    "gpu": [torch.cuda.get_device_name(i) for i in range(2)],\n                },\n                "historical_author_inputs_verified": False,\n            },\n        )\n\n    def check_events(self, namespace):\n        bad = [\n            event\n            for event in namespace.get("_RSNA_AUDIT", {}).get("events", [])\n            if event.get("kind") not in ALLOWED_EVENTS\n        ]\n        if bad:\n            raise ValueError(f"Source reported degraded/unknown events: {bad[:8]}")\n\n    def check_dino(self, namespace):\n        if namespace.get("_DINOV2_MATCHED_MEMBERS") != 20:\n            raise ValueError("DINO must complete 20 members")\n        parity = [\n            event["member"]\n            for event in namespace["_RSNA_AUDIT"]["events"]\n            if event["kind"] == "dino_shared_path_parity"\n        ]\n        expected = self.contract["dino_member_ids"]\n        if len(parity) != 20 or set(parity) != set(expected):\n            raise ValueError("DINO shared/full forward parity did not cover all 20 members")\n        for member in expected:\n            aligned_npz(self.work / "diagnostics" / f"dino_{member}.npz", self.ids)\n\n    def check_a5(self):\n        import numpy as np\n\n        with np.load(self.work / "speed_a5_raw.npz", allow_pickle=False) as data:\n            if data["study_uids"].astype(str).tolist() != self.ids:\n                raise ValueError("A5 UID order changed")\n            raw = data["raw_probabilities"]\n            applicable = data["applicable"]\n            if raw.shape != (5, len(self.ids), 12) or applicable.shape != (len(self.ids),) or applicable.dtype != bool:\n                raise ValueError("A5 shape/applicability contract failed")\n            probabilities(raw[:, applicable], (5, int(applicable.sum()), 12))\n            if not np.isnan(raw[:, ~applicable]).all():\n                raise ValueError("A5 unavailable rows must remain missing")\n        aligned_npz(self.work / "diagnostics/a5_rank_mean.npz", self.ids)\n\n    def check_coats(self, namespace):\n        import numpy as np\n\n        logs = ("resgated_runtime.log", "global96_runtime.log", "d4_input/d4.log", "repairv1_runtime.log")\n        for name in logs:\n            path = self.work / name\n            with path.open(encoding="utf-8", errors="replace") as stream:\n                if any("[dense-fallback]" in line for line in stream):\n                    raise ValueError(f"CoAt input preparation changed after failure: {name}")\n        check_blend_receipt(read_json(self.work / "_coat_raptor_blend_receipt.json"), len(self.ids))\n        for name, status, epochs, models in zip(\n            RECEIPT_FILES, RECEIPT_STATUS, ([4, 6, 8], [16, 23, 18], None, [12, 7, 11]), (3, 3, 1, 3)\n        ):\n            receipt = read_json(self.work / name)\n            if (\n                receipt.get("status") != status\n                or receipt.get("fallback_studies") != 0\n                or receipt.get("models") != models\n            ):\n                raise ValueError(f"Incomplete CoAt family: {name}")\n            if epochs and [item["epoch"] for item in receipt["checkpoints"]] != epochs:\n                raise ValueError(f"Wrong checkpoint epochs: {name}")\n            for shard in receipt.get("shards", []):\n                if shard.get("preparation_warnings", 0) != 0 or shard.get("fallback_studies", 0) != 0:\n                    raise ValueError(f"Degraded preparation in CoAt shard: {name}")\n        expected = {(name, uid) for name in ("maxspan-v5", "native384dense-v10", "native384-v8") for uid in self.ids}\n        if namespace.get("_ke_input_ids") != expected:\n            raise ValueError("Incomplete Raptor preparation UID coverage")\n        if namespace.get("_coatnet_weight") != OUTER:\n            raise ValueError("Outer weights changed")\n        parent = aligned_npz(self.work / "btk_v32_rest_input.npz", self.ids)\n        raptor = read_scores(self.work / "raptor_input_before_coat.csv", self.ids)\n        coats = [aligned_npz(self.work / name, self.ids, "probability_mean", False) for name in PROB_FILES]\n        _, family = replay_final(parent, raptor, coats)\n        expected_hybrid = 0.6 * rank_pct(raptor) + 0.4 * family\n        # Source serializes the hybrid CSV and reads it with pandas before ranking.\n        # Validate that roundtrip, then use its actual floating values for ties.\n        actual_hybrid = probabilities(np.asarray(namespace["_blend_cr"]), (len(self.ids), 12))\n        actual_parent = probabilities(np.asarray(namespace["_blend_tr"]), (len(self.ids), 12))\n        if not np.allclose(actual_hybrid, expected_hybrid, rtol=0, atol=1e-12):\n            raise ValueError("Inner hybrid changed beyond CSV roundtrip precision")\n        if not np.allclose(actual_parent, rank_pct(parent), rtol=0, atol=1e-12):\n            raise ValueError("Parent full-cohort ranks changed")\n        weights = np.asarray([OUTER[label] for label in LABELS])\n        final = rank_pct((1 - weights) * actual_parent + weights * actual_hybrid)\n        if not np.allclose(read_scores(self.work / "_coat_family_rank.csv", self.ids), family, rtol=0, atol=1e-12):\n            raise ValueError("CoAt reduction is not rank of the full-cohort probability mean")\n        if not np.allclose(read_scores(self.work / "_pipeline_stage.csv", self.ids), final, rtol=0, atol=1e-12):\n            raise ValueError("Final predictions do not match the frozen V2 recipe")\n\n    def reject(self, reason):\n        self.ready = False\n        if self.owns_outputs:\n            final = self.work / "submission.csv"\n            if final.exists():\n                final.rename(self.work / "P003_REJECTED_submission.csv.disabled")\n            save_json(\n                self.work / "P003_FAILED.json",\n                {"status": "failed", "reason": reason, "completed_cells": self.completed, "timings": self.timings},\n            )\n\n    def run_cell(self, number, source, namespace):\n        order = self.contract["cell_order"]\n        started = time.monotonic()\n        try:\n            if not self.ready or self.completed != order[: len(self.completed)] or number != order[len(self.completed)]:\n                raise RuntimeError("Run every cell once, in order, in a fresh session")\n            if hashlib.sha256(source.encode()).hexdigest() != self.contract["source_cells"][str(number)]:\n                raise ValueError("Frozen source changed")\n            filename = f"<p003-source-{number}>"\n            linecache.cache[filename] = (len(source), None, source.splitlines(keepends=True), filename)\n            exec(compile(source, filename, "exec", dont_inherit=True), namespace)\n            self.check_events(namespace)\n            if number == 17:\n                self.check_dino(namespace)\n            if number == 22:\n                self.check_a5()\n            if number == 25 and namespace.get("V18_CALIBRATOR_APPLIED") is not True:\n                raise ValueError("Fixed Rad calibrator missing")\n            if number == 27:\n                self.check_coats(namespace)\n            self.completed.append(number)\n        except BaseException as exc:\n            self.reject(f"Cell {number}: {type(exc).__name__}: {exc}")\n            raise\n        finally:\n            self.timings.append({"cell": number, "seconds": time.monotonic() - started})\n            if self.owns_outputs:\n                save_json(self.work / "P003_TIMINGS.json", self.timings)\n\n    def finish(self, namespace):\n        try:\n            if not self.ready or self.completed != self.contract["cell_order"]:\n                raise ValueError("Incomplete notebook")\n            self.check_events(namespace)\n            read_scores(self.work / "submission.csv", self.ids)\n            receipt = read_json(self.work / "btkd_v559_complete.json")\n            digest = sha256(self.work / "submission.csv")\n            if receipt.get("status") != "COMPLETE" or receipt.get("submission_sha256") != digest:\n                raise ValueError("Final receipt/hash mismatch")\n            if (\n                receipt.get("coat_family_members") != 4\n                or receipt.get("family_reduction") != "rank_of_member_probability_mean"\n            ):\n                raise ValueError("Final CoAt receipt incomplete")\n            elapsed = time.monotonic() - self.started\n            if elapsed >= 9 * 3600:\n                raise ValueError("Full notebook exceeded nine hours")\n            result = {\n                "status": "passed",\n                "candidate_id": self.contract["candidate_id"],\n                "studies": len(self.ids),\n                "submission_sha256": digest,\n                "elapsed_seconds": elapsed,\n                "timings": self.timings,\n                "source_notebook_sha256": self.contract["source_notebook_sha256"],\n                "public_score": None,\n                "score_reproduction_claimed": False,\n                "note": "Saved-run completion does not establish hidden-test runtime or score.",\n            }\n            save_json(self.work / "P003_READY.json", result)\n            print(f"P003 READY FOR MANUAL SUBMISSION | {len(self.ids)} studies | {digest}")\n            return result\n        except BaseException as exc:\n            self.reject(f"Final gate: {type(exc).__name__}: {exc}")\n            raise\n', '<p003-guard>', 'exec'), _p003_module.__dict__)
_p003_guard = _p003_module.P003Guard({'candidate_id': 'p003-haideptry-v2-current-pins-20261005', 'inputs': [{'key': 'raptor_knee_maxspan', 'kind': 'dataset', 'ref': 'dreaddevelopment/raptor-knee-maxspan', 'version': 1, 'url': 'https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-maxspan/versions/1', 'mounts': ['datasets/dreaddevelopment/raptor-knee-maxspan', 'raptor-knee-maxspan'], 'license_declaration': 'CC0: Public Domain', 'files': [{'path': 'raptor_ft_coatnet_v5_full_swa.pt', 'bytes': 292831426, 'sha256': '20d45e0732a0e9120025d0172a4d2dd0170900bc250b06742dfb7ae08a63786a', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}]}, {'key': 'raptor_knee_native384', 'kind': 'dataset', 'ref': 'dreaddevelopment/raptor-knee-native384', 'version': 1, 'url': 'https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-native384/versions/1', 'mounts': ['datasets/dreaddevelopment/raptor-knee-native384', 'raptor-knee-native384'], 'license_declaration': 'CC0: Public Domain', 'files': [{'path': 'raptor_ft_coatnet_v8_full_swa.pt', 'bytes': 292831426, 'sha256': '3db77388662d516bb0ce790dafd24e36f4991f6089af26daf546817fe1c21862', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}]}, {'key': 'raptor_knee_native384dense', 'kind': 'dataset', 'ref': 'dreaddevelopment/raptor-knee-native384dense', 'version': 1, 'url': 'https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-native384dense/versions/1', 'mounts': ['datasets/dreaddevelopment/raptor-knee-native384dense', 'raptor-knee-native384dense'], 'license_declaration': 'CC0: Public Domain', 'files': [{'path': 'raptor_ft_coatnet_v10_full.pt', 'bytes': 292829892, 'sha256': '5add8816a47e040efca89bb44480e1cbd01f92db1253831ff7ceb6c65ee02b1d', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}]}, {'key': 'knee_mri_fold_weights', 'kind': 'dataset', 'ref': 'mattiaangeli/knee-mri-fold-weights', 'version': 2, 'url': 'https://www.kaggle.com/datasets/mattiaangeli/knee-mri-fold-weights/versions/2', 'mounts': ['datasets/mattiaangeli/knee-mri-fold-weights', 'knee-mri-fold-weights'], 'license_declaration': 'CC0: Public Domain', 'files': [{'path': 'm_f0.pt', 'bytes': 94000039, 'sha256': 'b447a665a92ed65e0f4cacf6800940ba7a6d585d0c36154ce18dee60ec48d5cd', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_f1.pt', 'bytes': 94000039, 'sha256': '30bbe5f9afa24af57fd69bad293a4b9fe3ca2e1d737381bc06e469c5ed380e76', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_f2.pt', 'bytes': 94000039, 'sha256': '5bdc4f72471e7c5db01f2f919ba39212087cbb5f2f76b619e4ba38446800b782', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_f3.pt', 'bytes': 94000039, 'sha256': 'f757ddc9531f903c6cd0729cadc7051fcdca03b281c89c39cc7957ab44fc2243', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_f4.pt', 'bytes': 94000039, 'sha256': '066490a5c17b3887d1bfda1427f45b14cab416e7b1dcc16afc9a003cdcc08aad', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'timm-1.0.22-py3-none-any.whl', 'bytes': 2530238, 'sha256': '888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0', 'role': 'offline_dependency', 'hash_evidence': 'https://pypi.org/pypi/timm/1.0.22/json の同名wheel SHA-256。Kaggle V2一覧と名前・bytes一致。Kaggle実行で一致必須。', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}]}, {'key': 'opencv_python_headless_4120088_x86', 'kind': 'dataset', 'ref': 'mattiaangeli/opencv-python-headless-4120088-x86', 'version': 1, 'url': 'https://www.kaggle.com/datasets/mattiaangeli/opencv-python-headless-4120088-x86/versions/1', 'mounts': ['datasets/mattiaangeli/opencv-python-headless-4120088-x86', 'opencv-python-headless-4120088-x86'], 'license_declaration': 'Unknown', 'files': [{'path': 'opencv_python_headless-4.12.0.88-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl', 'bytes': 54023419, 'sha256': '236c8df54a90f4d02076e6f9c1cc763d794542e886c576a6fee46ec8ff75a7a9', 'role': 'offline_dependency', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}]}, {'key': 'resnet_50_radimagenet_marwan', 'kind': 'dataset', 'ref': 'marwanmath/resnet-50-radimagenet-marwan', 'version': 1, 'url': 'https://www.kaggle.com/datasets/marwanmath/resnet-50-radimagenet-marwan/versions/1', 'mounts': ['datasets/marwanmath/resnet-50-radimagenet-marwan', 'resnet-50-radimagenet-marwan'], 'license_declaration': 'CC BY-NC-SA 4.0', 'files': [{'path': 'ResNet50.pt', 'bytes': 94345733, 'sha256': '08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}]}, {'key': 'rsna_knee_coat_resgated_ep10_top3', 'kind': 'dataset', 'ref': 'mattiaangeli/rsna-knee-coat-resgated-ep10-top3', 'version': 3, 'url': 'https://www.kaggle.com/datasets/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/versions/3', 'mounts': ['datasets/mattiaangeli/rsna-knee-coat-resgated-ep10-top3', 'rsna-knee-coat-resgated-ep10-top3'], 'license_declaration': 'CC0: Public Domain', 'files': [{'path': 'coat_resgated_ep10_top3_manifest.json', 'bytes': 7409, 'sha256': '98511a8fdeb9da0e6e70c78d013ff636e1476f31c80b5dc134d294b18c3f284e', 'role': 'publisher_manifest', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'locally_verified_small_manifest', 'payload_verified_locally': True, 'required': True}, {'path': 'cnx_dicom_geometry.py', 'bytes': 42634, 'sha256': '1e6e46ca40828aec8e68d93dea3676c68ee4803a10b349ad092861006e8301d4', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coat_resgated_ep10_gold58_geometry_contract.json', 'bytes': 27508, 'sha256': 'e4160563f96bdd7513605628294c23dc93aa03e47278f4f6668a951ede522796', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coat_resgated_ep10_t4_gold58_rank_portability.json', 'bytes': 7890, 'sha256': 'a11f5d37d877727f5d1d348ee2d5bfbcc144859c0459ced6b4a0d37eb76f2f21', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coat_resgated_ep10_t4_gold58_rank_portability_predictions.npz', 'bytes': 26635, 'sha256': 'a5ec6b5c56ea25efb850d3b2eab34462587850e4f28beb8345558be0aae5e4c4', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coat_resgated_gold_e04.pt', 'bytes': 293989431, 'sha256': 'cefaf007415b5e45ce2a720d5e211a183a4e90f160710ba1f72b1ce7c46239a2', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coat_resgated_gold_e06.pt', 'bytes': 293989431, 'sha256': '56eca1f61651362ecfae8aecdd9327125713b95653344b8ad62ef8cff17d08f4', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coat_resgated_gold_e08.pt', 'bytes': 293989367, 'sha256': '8d3159102046c127dc0d084fb69c9304ffcbd4b98d1333f9811d4d613fea1b12', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_q2l_bagmatch.py', 'bytes': 21977, 'sha256': 'a3303a724e95cf3d9f13b67ef52529adf854b0cbe580b896ef42382cc740d932', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_resgated_ep10_top3_inference.py', 'bytes': 49679, 'sha256': 'b11e58f8d7cabe9e264ac70b01e811dc6a846aa01248ace1f0e6536d0d4094d9', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': True, 'required': True}, {'path': 'coatnet_residual_gated_k32.py', 'bytes': 16639, 'sha256': '83735f535235bfd575767b9ac9b12ca0a77a6b11adf14343ca5c2466257f3374', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'gold58_top3_predictions.npz', 'bytes': 19046, 'sha256': '9b6541a643a5f9eccabc2b9414784ee3f82c3f0049c330ec237e2f0835ddeb32', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'raptor_k6_surgical.py', 'bytes': 29665, 'sha256': 'cc525db3bb67f0fde9db8dcfbbd33ff5bf5200d0c3c92251b368101314ae647f', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'raptor_light224.py', 'bytes': 43176, 'sha256': 'd8f533df336f097910cd8c9ed7ae2248783554c9c797b661ee89406b5041cde5', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'raptor_light_inference.py', 'bytes': 112310, 'sha256': '3ce34aba32b44509bda7fe37a9b9b62c68028ea9a4d1364a5b4db3d97640b318', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'smart_crop224.py', 'bytes': 14344, 'sha256': '31b66d1a687fb9de3d01c10f31baf8b728903b6ed9e9effedec6fc8e77c4e424', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'trainer_a6000_epochs01_06.py', 'bytes': 58679, 'sha256': 'bc4bd65893b99214b06ebd19055cf0db85427ed1bbe6c901ba78aec82f037b9e', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'trainer_h100_epochs07_10.py', 'bytes': 61532, 'sha256': '88cdd6d343b43e94f3539bbcdeff0c77eed6359c3c95457aff8d2593c7b4e4a7', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'training_history_through_epoch10.json', 'bytes': 46434, 'sha256': '45fc2fd5983b434cce633d3f90e5c363bda267c02fa583ab7f04d29c7c45ca49', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'triplet_physical_crop.py', 'bytes': 7784, 'sha256': 'b84826fede9fa4195774e208e829ad60f189cfee90dce8a2c9528349b18717cf', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coat-resgated-ep10-top3/3/coat_resgated_ep10_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}]}, {'key': 'rsna_knee_coatnet_d4_depthzone_swa3_b2', 'kind': 'dataset', 'ref': 'mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2', 'version': 2, 'url': 'https://www.kaggle.com/datasets/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/versions/2', 'mounts': ['datasets/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2', 'rsna-knee-coatnet-d4-depthzone-swa3-b2'], 'license_declaration': 'CC0: Public Domain', 'files': [{'path': 'coatnet_pairfilm_manifest.json', 'bytes': 14155, 'sha256': '7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d', 'role': 'publisher_manifest', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'locally_verified_small_manifest', 'payload_verified_locally': True, 'required': True}, {'path': 'audit_coatnet_d4_serving_equivalence.py', 'bytes': 12178, 'sha256': 'd1aa41f568f06691a368923eafb9067fe9f590deed1a03e8aa7de0798c8c4424', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'cnx_dicom_geometry.py', 'bytes': 42634, 'sha256': '1e6e46ca40828aec8e68d93dea3676c68ee4803a10b349ad092861006e8301d4', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_d4_depthzone_swa3_adapter.pt', 'bytes': 43258, 'sha256': 'b7c48b19997bc7ecb7e997f8dcf491ec5701ff917d1dac4b369eb85ac97ae89a', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_d4_depthzone_swa_inference.py', 'bytes': 10937, 'sha256': 'fedf2053d5cdd5ad6fb3793c33bdb226a42cf9eddfb3667f91b33e34670b7a1f', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': True, 'required': True}, {'path': 'coatnet_d4_parent_rank8_swa2.pt', 'bytes': 303275707, 'sha256': '5de38e333e3184d7b52fb068183f0cc198d6cbabb6ffbe36b26338622e82d297', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_fsxfilm_inference_model.py', 'bytes': 25914, 'sha256': '290688a0b063e03a7c4f7eefebeb7923dfac795ef2c342498e6803b48f056bcf', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_fsxfilm_swa_inference.py', 'bytes': 37044, 'sha256': 'c05738825546649d69c8eefb17696bcc4881699d36d6a4ec59ddec566aa704fe', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_global96_baseline_top3_inference.py', 'bytes': 56308, 'sha256': '7c6a8df50e269b9aa69077b64f7296acb4ad5bf3792ec2523b645b2a42778acb', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_global96_inference_model.py', 'bytes': 11214, 'sha256': 'd2ca34d5b1538ce42642efa0d34937f0d8a98719130dd76cc5bfa8944a2288ae', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_pairfilm_swa_inference.py', 'bytes': 20184, 'sha256': '60d016489aaca485ea94a33048238cd37ab284174c118c2e668006ded3cd0dd9', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_slice_depth_adapters.py', 'bytes': 38168, 'sha256': 'fb24c03b7554f4acf02cfa40dfd7492e72bf1e9fde6ae8e19ff3868727cfa37c', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'd4_gold58_reference.npz', 'bytes': 9626, 'sha256': 'ef4c66c212cec2ed7b8b3b9da44e2a0e7d8bcd1a85567563c32eef70dacce6d6', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'd4_train_serving_equivalence.json', 'bytes': 2851, 'sha256': '58c0e36e9dd530f11b2e8625c4a907be9762aa49791c02abc83432216e4840c6', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_CODEX_SLICE_CENTER_DEPTH_5_ABLATIONS_20260909.md', 'bytes': 12362, 'sha256': '92a50ec523d4a183f043f7b286a680ea011ca6a86f69a02b33593393a60c9dc7', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_anchored_adaptive_patch_pool.py', 'bytes': 21303, 'sha256': '0ae316a4c0b970bdb6386769eb47d25cd0cb4e765293cf1354e3dbfd9848ae11', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_coatnet_finding_slot_cross_attention_lr.py', 'bytes': 34570, 'sha256': '69d197571af8e971c618bc004839f07162a27636761c2b22b6f2585bd4ea5a59', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_coatnet_joint_slot_finding_mha.py', 'bytes': 26122, 'sha256': 'b899e9e020305c963ebeabe905e79802a006806761fbb08da569ff88e70fbcc3', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_coatnet_q2l_bagmatch.py', 'bytes': 21977, 'sha256': 'a3303a724e95cf3d9f13b67ef52529adf854b0cbe580b896ef42382cc740d932', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_coatnet_residual_gated_k32.py', 'bytes': 43181, 'sha256': '23bbf2625ae23a12c49bef8295d298aca1cff233bff7fe4bf34e540849e8f57a', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_coatnet_slice_depth_adapters.py', 'bytes': 38168, 'sha256': 'fb24c03b7554f4acf02cfa40dfd7492e72bf1e9fde6ae8e19ff3868727cfa37c', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_convnext_cross_slot_finding_head.py', 'bytes': 18263, 'sha256': 'a8dde3cb4df2158a73db5ace7faefbafe5b08b34ebabf8384842df4e43f78fcb', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_convnext_head_ablation_k32.py', 'bytes': 35161, 'sha256': 'ea59169fa4297b9e1760cc21a61335864843a98af55b11e933fd0349890dcaa1', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_finding_slot_cross_attention_lr.py', 'bytes': 29181, 'sha256': 'e0e609f6e268fdd8134b08ae93f433f25ecdb79b57699886f7ed1e7f24d5846e', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_raptor_global_stack_slot_coverage.py', 'bytes': 9402, 'sha256': '2e00a97ed4b83f9a1dcf96a24d0fecf923468da5aea1ae10ef93474aa448de22', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_raptor_global_stack_smart336.py', 'bytes': 28204, 'sha256': 'c73d84cfb385215c6c987bf9dcde73d1fe47e25806538dee0da04c407466b802', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_raptor_k6_surgical.py', 'bytes': 29665, 'sha256': 'cc525db3bb67f0fde9db8dcfbbd33ff5bf5200d0c3c92251b368101314ae647f', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_raptor_light224.py', 'bytes': 43176, 'sha256': 'd8f533df336f097910cd8c9ed7ae2248783554c9c797b661ee89406b5041cde5', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_test_coatnet_slice_depth_adapters.py', 'bytes': 9682, 'sha256': 'e2d59850c36e2d80e9bc663f2f042a3b0d608902257cb6ed67af3181edd05b5e', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_train_coatnet_adaptive_patch_guard.py', 'bytes': 52367, 'sha256': '19d8a52917c056b15567d1c1431e903860e919220e4efe2edafb3ad91e988fce', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_train_coatnet_global96_joint_mha_k12_effb24_e24_full_gold_swa.py', 'bytes': 241473, 'sha256': 'f3cbfb85e1824c5172ab53d2497559492b1d97ae0cbfabd96a84817d4539245d', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_train_coatnet_slice_depth_ablation.py', 'bytes': 47062, 'sha256': '826aec545a22c708e08b6489615c9dc2664e5bee9795dcd2857b1747a43459ad', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'executed_triplet_physical_crop.py', 'bytes': 7784, 'sha256': 'b84826fede9fa4195774e208e829ad60f189cfee90dce8a2c9528349b18717cf', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'finding_slot_cross_attention_lr.py', 'bytes': 29181, 'sha256': 'e0e609f6e268fdd8134b08ae93f433f25ecdb79b57699886f7ed1e7f24d5846e', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'inherited_coatnet_fsxfilm_swa_train_submission_parity_gold58.json', 'bytes': 42936, 'sha256': '105dfb703bf29d0bcbfe6236589f582a91ef60ce336685ccd522315bbd996a5a', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'inherited_coatnet_fsxfilm_train36_cache_reference.json', 'bytes': 4824, 'sha256': '2b143a323d586840e5e852c3d18f550db27859df1e6ad44accaa1d21b929999d', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'inherited_coatnet_global96_train_submission_parity_gold58.json', 'bytes': 49078, 'sha256': 'c96290d0bea1511f7feb662b40c370fc21369747daef12748a24485db8e4c382', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'raptor_global_stack_slot_coverage.py', 'bytes': 9402, 'sha256': '2e00a97ed4b83f9a1dcf96a24d0fecf923468da5aea1ae10ef93474aa448de22', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'raptor_global_stack_smart336.py', 'bytes': 28204, 'sha256': 'c73d84cfb385215c6c987bf9dcde73d1fe47e25806538dee0da04c407466b802', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'raptor_light224.py', 'bytes': 43176, 'sha256': 'd8f533df336f097910cd8c9ed7ae2248783554c9c797b661ee89406b5041cde5', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'raptor_light_inference.py', 'bytes': 112836, 'sha256': '2bd1f9f85d6895442825d6440ef9e2cffe6ad002f2f1bff484a35769443e4328', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'smart_crop224.py', 'bytes': 14344, 'sha256': '31b66d1a687fb9de3d01c10f31baf8b728903b6ed9e9effedec6fc8e77c4e424', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'timm-1.0.22-py3-none-any.whl', 'bytes': 2530238, 'sha256': '888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0', 'role': 'offline_dependency', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'training_completion_d4.json', 'bytes': 4557, 'sha256': 'fdb25f9e72fe2e8a76d3ec9b7d1c6d971ee3524f6c395a54f9273abca9ff9e9b', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': True, 'required': True}, {'path': 'training_history_d4.json', 'bytes': 47630, 'sha256': '23b25b198ff79cd7e24e88e15d62622b74d85604451ffc20c1058b5b45529d50', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'triplet_physical_crop.py', 'bytes': 7784, 'sha256': 'b84826fede9fa4195774e208e829ad60f189cfee90dce8a2c9528349b18717cf', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/coatnet_pairfilm_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/coatnet_repairv1_top3_manifest.json', 'bytes': 10483, 'sha256': '99288a9a42f36ba544d7b0ad2fe645e3b1fc9c507c23802c16ca0d2e5085f684', 'role': 'publisher_manifest', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'locally_verified_small_manifest', 'payload_verified_locally': True, 'required': True}, {'path': 'repairv1/COATNET_REPAIRV1_SUBMISSION_GATES_20260904.md', 'bytes': 6175, 'sha256': 'f206514f349f49a74d7a3b092ee192c87845fdc1f21b42d402e6a24edfb3ec73', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/COATNET_REPAIRV1_T4_RENDERER_SELECTION_20260904.md', 'bytes': 1920, 'sha256': '7ae46fa0a90cb700b00b73cff7e0317549d7674ef6d0b311b06c29bf35a422f3', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/cnx_dicom_geometry.py', 'bytes': 42634, 'sha256': '1e6e46ca40828aec8e68d93dea3676c68ee4803a10b349ad092861006e8301d4', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/coatnet_q2l_bagmatch.py', 'bytes': 21977, 'sha256': 'a3303a724e95cf3d9f13b67ef52529adf854b0cbe580b896ef42382cc740d932', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/coatnet_repairv1_full_gold58_t4_cv412_audit.json', 'bytes': 22712, 'sha256': '97359a7ad9dd8a8ee6a4499850b6e8aaf0fc440e7a40de38dba2b9a60470d752', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/coatnet_repairv1_full_gold58_t4_cv412_audit_source.py', 'bytes': 19222, 'sha256': '6f60df836df5ff7f0eb61f0747e6a7e25688cb1abd174c94b4ee0b6ec7b370a9', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/coatnet_repairv1_full_gold58_t4_cv412_predictions.npz', 'bytes': 26157, 'sha256': '3786c3f45cc86edbc4e46c37bd92ccd66bc84eed9cca32b404b10d05ae4fc39e', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/coatnet_repairv1_gold58_top3_predictions.npz', 'bytes': 20411, 'sha256': 'd47364a441bcfa23fd9c1778f49432b33bb45713b34e2e4e3d0da04791d1671e', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/coatnet_repairv1_gold_e07.pt', 'bytes': 292939604, 'sha256': '4f36e9aff09986fcf4f130203e3fbf391b95e0a466ed9c6f656dafd01c349cdf', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/coatnet_repairv1_gold_e11.pt', 'bytes': 292939604, 'sha256': '2ec451b03e4c1da5048aee91d4358fa2efa2900619d9dfd5b75dadce659e2f21', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/coatnet_repairv1_gold_e12.pt', 'bytes': 292939668, 'sha256': '45f34ec036a6b2dfb75239343f1dc33bc815bacd8c2d996235035d2602b5b2e7', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/coatnet_repairv1_top3_inference.py', 'bytes': 54159, 'sha256': '4f3e5cca50c048955fed1d6dcaf57bc3ccf7c45e143d75267550a3f53596aa69', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': True, 'required': True}, {'path': 'repairv1/coatnet_repairv1_train_submission_parity_gold58.json', 'bytes': 34531, 'sha256': '424ce4c85159e40aa11797aa8ef35ed4b84c3f6e0bdbbbd1a17fdf8bda9079de', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/coatnet_residual_gated_k32.py', 'bytes': 37342, 'sha256': '4bc148bf21420076678aaad3187b0b457b45f6cebd6f5374e404232ab6536f36', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/historical_k12_t4_microbatch36.json', 'bytes': 3173, 'sha256': '501bf6fb7b46b125b7c57df2a93bf832be9330f79cb0d6987f6d151bea6a9ff5', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/historical_k12_t4_precision_audit.json', 'bytes': 3752, 'sha256': 'e4eb7ab5d98908818d78151f3664039e597542ec5ef30e235352a0724e555cd2', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/historical_k12_t4x2_receipt.json', 'bytes': 3843, 'sha256': 'ac6d5cdef99a3859de32aa12eeee55d8b4fd3c387c43a8b5522fa3488c899833', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/historical_resgated_top3_t4x2_receipt.json', 'bytes': 3709, 'sha256': '41d163dc88fd685c02c0222bd10b4e29353911016bd16516bf2b6bd1b89a8f32', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/raptor_k6_surgical.py', 'bytes': 29665, 'sha256': 'cc525db3bb67f0fde9db8dcfbbd33ff5bf5200d0c3c92251b368101314ae647f', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/raptor_light224.py', 'bytes': 43176, 'sha256': 'd8f533df336f097910cd8c9ed7ae2248783554c9c797b661ee89406b5041cde5', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/raptor_light_inference.py', 'bytes': 112310, 'sha256': '3ce34aba32b44509bda7fe37a9b9b62c68028ea9a4d1364a5b4db3d97640b318', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/runpod_retrieval_receipt.json', 'bytes': 770, 'sha256': '1982c9b86a955e37e433309f9640a2c53128c3d4c35c5d7ceaa351be4c11b3ff', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/smart_crop224.py', 'bytes': 14344, 'sha256': '31b66d1a687fb9de3d01c10f31baf8b728903b6ed9e9effedec6fc8e77c4e424', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/train_coatnet_repairv1_exact.py', 'bytes': 112594, 'sha256': '123f2eb49536fee60beb97c4dd6b12f90e28cb9a9d30ad2c1758a5ba444928c3', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/training_history_terminal_epoch18.json', 'bytes': 125886, 'sha256': 'fb825bb22a5c0e7a10ef4561247a5c9fabb14c02059cf854b44d81b06ede1a43', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'repairv1/triplet_physical_crop.py', 'bytes': 7784, 'sha256': 'b84826fede9fa4195774e208e829ad60f189cfee90dce8a2c9528349b18717cf', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2/2/repairv1/coatnet_repairv1_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}]}, {'key': 'rsna_knee_coatnet_global96_top3', 'kind': 'dataset', 'ref': 'mattiaangeli/rsna-knee-coatnet-global96-top3', 'version': 1, 'url': 'https://www.kaggle.com/datasets/mattiaangeli/rsna-knee-coatnet-global96-top3/versions/1', 'mounts': ['datasets/mattiaangeli/rsna-knee-coatnet-global96-top3', 'rsna-knee-coatnet-global96-top3'], 'license_declaration': 'CC0: Public Domain', 'files': [{'path': 'coatnet_global96_top3_manifest.json', 'bytes': 7486, 'sha256': '015f09030e76f86274cadae40777b3c3cf5c19f8c835d5ad88e15db41903779e', 'role': 'publisher_manifest', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'locally_verified_small_manifest', 'payload_verified_locally': True, 'required': True}, {'path': 'cnx_dicom_geometry.py', 'bytes': 42634, 'sha256': '1e6e46ca40828aec8e68d93dea3676c68ee4803a10b349ad092861006e8301d4', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_global96_baseline_top3_inference.py', 'bytes': 55963, 'sha256': '8beec87f648f298d348679e8812a0ca278e6c040cba9f797dbbc7efa0ed5d4b1', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': True, 'required': True}, {'path': 'coatnet_global96_gold58_reference.npz', 'bytes': 20511, 'sha256': '05651c1ec0980f31fea95d95c116e8e7d7d86eeaf95e43cae80639e36fe40ddc', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_global96_gold_e16.pt', 'bytes': 300400663, 'sha256': 'f824d16bd506f442cc5a37b054663963e5461a59a0c42ad480f3fe515dd9408f', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_global96_gold_e18.pt', 'bytes': 300400663, 'sha256': '629f0d10ebc1946a25577b8dc0fa1d83d65d8a607961d146b53b7a36dc2c5c2e', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_global96_gold_e23.pt', 'bytes': 300400663, 'sha256': 'c177a8a9240c7d55465fea83bb27b1ab400274722095b2895b689092a804bfdb', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_global96_inference_model.py', 'bytes': 11214, 'sha256': 'd2ca34d5b1538ce42642efa0d34937f0d8a98719130dd76cc5bfa8944a2288ae', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'coatnet_global96_train_submission_parity_gold58.json', 'bytes': 49078, 'sha256': 'c96290d0bea1511f7feb662b40c370fc21369747daef12748a24485db8e4c382', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'minimal_model_equivalence.json', 'bytes': 215, 'sha256': '90bf1d03596af0696fbe95415c8909ac5f75a9852f3457975b27e04a6f3192e0', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'raptor_global_stack_smart336.py', 'bytes': 26372, 'sha256': '96d9f2475438771bcdd937a7747fb54b62ecb4ece74f9f0f0b5d6b71db99d7c0', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'raptor_light224.py', 'bytes': 43176, 'sha256': 'd8f533df336f097910cd8c9ed7ae2248783554c9c797b661ee89406b5041cde5', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'raptor_light_inference.py', 'bytes': 112310, 'sha256': '3ce34aba32b44509bda7fe37a9b9b62c68028ea9a4d1364a5b4db3d97640b318', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'smart_crop224.py', 'bytes': 14344, 'sha256': '31b66d1a687fb9de3d01c10f31baf8b728903b6ed9e9effedec6fc8e77c4e424', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'timm-1.0.22-py3-none-any.whl', 'bytes': 2530238, 'sha256': '888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0', 'role': 'offline_dependency', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'train_coatnet_global96_exact.py', 'bytes': 162797, 'sha256': '413e67c6ead41765a0662888fd9a40176df1e0a0eec02dd1c6200e5fe0dbaa1a', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'training_history_global96_e24.json', 'bytes': 253853, 'sha256': '3795b033ce2a26c11bad0bf655ec36f7076c32400c330999419dc0fba472970f', 'role': 'provenance_only_hash_check', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}, {'path': 'triplet_physical_crop.py', 'bytes': 7784, 'sha256': 'b84826fede9fa4195774e208e829ad60f189cfee90dce8a2c9528349b18717cf', 'role': 'runtime_source', 'hash_evidence': 'artifacts/research/20261005-p003-investigate/source-documents/mattiaangeli/rsna-knee-coatnet-global96-top3/1/coatnet_global96_top3_manifest.json', 'hash_evidence_kind': 'public_manifest_declaration', 'payload_verified_locally': False, 'required': True}]}, {'key': 'rsna_knee_e9_radimagenet_heads_v15', 'kind': 'dataset', 'ref': 'antoinegg1/rsna-knee-e9-radimagenet-heads-v15', 'version': 3, 'url': 'https://www.kaggle.com/datasets/antoinegg1/rsna-knee-e9-radimagenet-heads-v15/versions/3', 'mounts': ['datasets/antoinegg1/rsna-knee-e9-radimagenet-heads-v15', 'rsna-knee-e9-radimagenet-heads-v15'], 'license_declaration': 'CC BY-NC-SA 4.0', 'files': [{'path': 'v52_radimagenet_heads.pt', 'bytes': 63525150, 'sha256': '0f465649799ecfbccaac1767844639e7ced44e1bc9babde6e4bac7c5d9b89eaa', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}]}, {'key': 'rsna_knee_weights', 'kind': 'dataset', 'ref': 'pilkwang/rsna-knee-weights', 'version': 1, 'url': 'https://www.kaggle.com/datasets/pilkwang/rsna-knee-weights/versions/1', 'mounts': ['datasets/pilkwang/rsna-knee-weights', 'rsna-knee-weights'], 'license_declaration': 'CC0: Public Domain', 'files': [{'path': 'm_013dc75703.pt', 'bytes': 89293562, 'sha256': '36d513243ebb22f2e8c5ded08cdb36932537bd700c49497a158ab7b3883d09b5', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_1e553ba481.pt', 'bytes': 89286714, 'sha256': '72b293b7199bedc038d26fa5453360752541714f5b023b8910e7e90dd435f8e4', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_2dce50ebd9.pt', 'bytes': 89295866, 'sha256': '0b8d57e7d6e160f95f9525d9a46a643691478fe63d25429cd4755e8d5567fcd4', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_2f7e785e82.pt', 'bytes': 89293818, 'sha256': '7090a090d38bb82dac4aad78b98f0cb3a54960ff5a009adc2a6718478b5fa32c', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_3d1f4c48fc.pt', 'bytes': 89293818, 'sha256': 'c913e392c89e406123efbaeea5fa3048101621947eebd77c206d7ec5682c7e35', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_44128e3ff3.pt', 'bytes': 89295866, 'sha256': '4b49cac5102e66cbf96210c29b17e35501b4cf8b69d058a8e791199aea25e89a', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_44bc3c6f14.pt', 'bytes': 89295866, 'sha256': '2cbdee353893131b73148b446f1e9b3fb4ee61595ab6145e854968125f47f20e', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_5f8ba4c5ea.pt', 'bytes': 89306554, 'sha256': '6267820f18b5ee7ab4672f7f8868949a86b1d5801224820927b9555304261fde', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_72081758ce.pt', 'bytes': 89296634, 'sha256': 'e895398d69fab80eff3a93be93e1c5673d87291abd1fb421e6210c82847b7423', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_7f8321affc.pt', 'bytes': 89296634, 'sha256': 'ee66cea7f13b09fb8b710decce196ef0a8dc072dc0a9bccd65554f9014dc40c2', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_84079fe8cb.pt', 'bytes': 89296634, 'sha256': 'ad3e21e73ae400b5c536936d2acbe8344487345b5358cdfb655758eceef41958', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_8476b29285.pt', 'bytes': 89286970, 'sha256': 'ea81c3eabc916286b6d73a26645b1feefb1b6511ff6e84e1a1559933a27786ae', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_8baecb9d1b.pt', 'bytes': 89293818, 'sha256': '321de45924ceed510d51a23cc6d462591e02825a6f5ff06beac48adc570c7621', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_91f171fe6f.pt', 'bytes': 89286970, 'sha256': 'fb40a77c1f19b70cbff241f1570adabc5517f0b11fabae30c5ca142bbfc7af19', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_b7c37cc80e.pt', 'bytes': 89296634, 'sha256': 'f54e85607e55ea8da5cc4582d6d9a3fec1ee3852f57a11f63a27db88f5824c2d', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_d47313baef.pt', 'bytes': 89306554, 'sha256': 'f79e7202e94cd15f057becb0f7f86123ae90472f625397b95a6a00d6ff34fa1f', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_e5427d6c21.pt', 'bytes': 89306234, 'sha256': '7223cd50ae5e7bdc8d3ffe888ab207bdbb2256d06e33f96e7a709eca5d86ab33', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_ea3fbf6edf.pt', 'bytes': 89306554, 'sha256': '5558228ee3099311a5ab9ad573b0861773428cf1ed537bd6f388d866be292d7c', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_ed427aa10a.pt', 'bytes': 89295866, 'sha256': 'ee8fe9a48061f0f6215bf7766dc0b705fbac2afe753e57828ab8842ee7f683b3', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'm_f335c812fe.pt', 'bytes': 89286970, 'sha256': 'dfc23ad8dbdf636ad72a19dbbf6cfa5434379d30d4d94caba8b33a56425b92c3', 'role': 'checkpoint', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}, {'path': 'manifest.json', 'bytes': 42477, 'sha256': '496949a3a3e789bc1f4ccff595205c911e471c5b5ef669366a2dd0a58e125844', 'role': 'publisher_manifest', 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/rsna-knee-weights/manifest.json', 'hash_evidence_kind': 'prior_public_manifest_expected_identity', 'payload_verified_locally': False, 'required': True}]}, {'key': 'rad_e13_notebook', 'kind': 'notebook_output', 'ref': 'sofiaanjenje/rsna-knee-e13-train', 'version': 1, 'url': 'https://www.kaggle.com/code/sofiaanjenje/rsna-knee-e13-train', 'mounts': ['notebooks/sofiaanjenje/rsna-knee-e13-train', 'rsna-knee-e13-train'], 'license_declaration': 'Upstream public Notebook license and RadImageNet terms; see previous source audit.', 'files': [{'path': 'rsna_rad_e11/v52_e11_heads.pt', 'bytes': 63534726, 'sha256': 'ad9f19af73bfdf4e49263c0e45060dc3cb239e1195039b26dc8c0a3a6bcd1a8a', 'role': 'rad_e13_five_heads', 'required': True, 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'payload_verified_locally': False, 'verify_payload_at_kaggle_preflight': True, 'hash_evidence_kind': 'public_p002_manifest_and_p003_source_agree'}]}, {'key': 'dinov2_base', 'kind': 'model', 'ref': 'metaresearch/dinov2/PyTorch/small/1', 'version': 1, 'url': 'https://www.kaggle.com/models/metaresearch/dinov2/frameworks/PyTorch/variations/small/versions/1', 'files': [{'path': 'config.json', 'bytes': 547, 'sha256': '1809f83e3bdb1609a501a610ad4a742f4fd8ae44d72ca4aa0df52d1f2ac8628d', 'role': 'dinov2_base_local_asset', 'required': True, 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'payload_verified_locally': False, 'verify_payload_at_kaggle_preflight': True}, {'path': 'pytorch_model.bin', 'bytes': 88297097, 'sha256': '1051e25b2ed69ddad24f3c41e7b6eed6e7f7d012103ea227e47eb82e87dc2050', 'role': 'dinov2_base_local_asset', 'required': True, 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'payload_verified_locally': False, 'verify_payload_at_kaggle_preflight': True}, {'path': 'preprocessor_config.json', 'bytes': 436, 'sha256': '14e780d86fa1861f8751f868d7f45425b5feb55c38ca26f152ca5097ab30f828', 'role': 'dinov2_base_local_asset', 'required': True, 'hash_evidence': 'artifacts/research/20261005-p002-inputs/source-documents/tonylica/rsna-knee-bend-dinov3-0917-repro-assets/5/bundle_manifest.json', 'payload_verified_locally': False, 'verify_payload_at_kaggle_preflight': True}], 'mounts': ['models/metaresearch/dinov2/PyTorch/small/1', 'models/metaresearch/dinov2/pytorch/small/1', 'dinov2/PyTorch/small/1', 'dinov2/pytorch/small/1']}, {'key': 'competition', 'kind': 'competition', 'ref': 'rsna-knee-abnormality-detection', 'version': None, 'url': 'https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/data', 'files': [{'path': 'train.csv', 'bytes': None, 'sha256': None, 'role': 'capacity_planning_row_count_only_no_label_selection', 'required': True}, {'path': 'test.csv', 'bytes': None, 'sha256': None, 'role': 'test_study_identity', 'required': True}, {'path': 'test_series.csv', 'bytes': None, 'sha256': None, 'role': 'test_series_geometry', 'required': True}, {'path': 'sample_submission.csv', 'bytes': None, 'sha256': None, 'role': 'submission_target_order', 'required': True}], 'mounts': ['rsna-knee-abnormality-detection', 'competitions/rsna-knee-abnormality-detection']}], 'source_notebook_sha256': 'b5f1e12ae3f8171aae716e53c779d0bc218b8631884238c2a182c937ade75c1c', 'source_cells': {'2': '25e69f00b3cebe2fdca50e18fa911f1c287b9f45836dd4cd80a459956033b20d', '3': '5815eed58685ad17392c9c5b13dfb0fb43fb54ec32b7834f4c2c66439cd3ea5a', '4': '32c4445b136e86d744ed5e2c465ad60cbc6fa781f06511280bf38c3dd634f2b7', '6': '61f911b3b024f870718f1eb123e051f2cd9f3718b4473a62d4b211fd151ea555', '7': 'a773890aca786656f025af0f008fd97bcd0736eb143698d71cd16448bb8fce76', '8': 'a554f7d0f44a197e371a0ce3ce734466c2658bae27c09d6ffd7855cb4210f1a5', '9': '0d2a6d71963ae849b3e39ea27ee739a769e33d1e0779b4ff2be9b0e7f50821fa', '10': 'ba17bd3298fc57f07ef581794ee274b5d4990b62a999086881ed6f57bcd882aa', '11': 'a1334e4b2174f1512d3dcd7f46ba257d698b83a6e399b8c1e6c2fa0901f29b8a', '12': '6878773f8ffa7883670b9bfceed886f80465514cea3ab30057c7378406531507', '13': '17434f44700a37e344f33040cf12618806370b438c0f2b2f8fe3caa8b42af065', '14': '0cee81943e6bfbb987380cf9c88b2ed961acde926d96211154c4b2946f6e849e', '15': 'a816bbcb91c3f8645f847d921a7e387df9ba7301c546ed151299a050c6e1e56c', '16': '9006955666c6c5f89bd87d606b83d5fba2d446c42168e223491e11b8b8f95102', '17': 'cd7a8762ef90ccf507eaf4588e350402a4f8b1f0f58b7f6687b135eee684d357', '19': '6fffd33cf27daf23afcae0616f5fac9ffc7e869e08cd7827f3d4f141276f2611', '20': 'f1fa6aa2130ce82d2efb84793f47c50c97b71e3e61c68ffc5840c4431e7a5f5e', '21': 'a0f7ec9a6a26310a246fef4d3349a1af1e853e99cd2ecbc2b819d96aa9071a1e', '22': 'c6126a20146a0853e11afe37c522b9e149495d0fe9e9450394a858c52532cab3', '23': '5c31acb1019dd9de97103d9a68cb0fc0db6aa1032fb00af5b483b3a67ed1b08b', '25': '8a5b323bada53d4ddfa913f8257042292ac6b92d31bbe2f2590c8a92b1bd1ab5', '27': 'e885d5241bdd5e5f4d7871033229a0a3dcef6e42ebc1363a61f3529238ecd747', '29': 'e2bf22f86a89808e0c3a1869d7daf735bfcd789e8fd8012ee938112c3f779e48'}, 'cell_order': [2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 23, 25, 27, 29], 'dino_member_ids': ['44128e3ff3', '1e553ba481', 'e5427d6c21', '7f8321affc', '013dc75703', '2dce50ebd9', '91f171fe6f', 'd47313baef', '72081758ce', '8baecb9d1b', '44bc3c6f14', '8476b29285', '5f8ba4c5ea', 'b7c37cc80e', '2f7e785e82', 'ed427aa10a', 'f335c812fe', 'ea3fbf6edf', '84079fe8cb', '3d1f4c48fc']})
_p003_builtins._rsna_p003_guard = _p003_guard
_p003_guard.prepare()


## Runtime contract and artifact checks

The first cells pin checkpoint hashes, dependency versions, memory limits, and
the D4 serving contract before expensive inference begins. Any missing or
changed required asset stops the run instead of silently changing predictions.


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(2, 'import os\nfor _thread_env in (\'OMP_NUM_THREADS\', \'OPENBLAS_NUM_THREADS\', \'MKL_NUM_THREADS\'):\n    os.environ.setdefault(_thread_env, \'4\')\nos.environ.setdefault(\'CUDNN_CONV_WSCAP_DBG\', \'1024\')\n\ndef _d4_check_runtime(rt, artifact_root):\n    import hashlib\n    import json\n    from pathlib import Path\n    import torch\n    root = Path(artifact_root)\n    manifest_path = root / \'coatnet_pairfilm_manifest.json\'\n    digest = hashlib.sha256(manifest_path.read_bytes()).hexdigest()\n    if digest != \'7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d\':\n        raise RuntimeError(\'D4 reference manifest identity changed\')\n    expected = json.loads(manifest_path.read_text())\n    manifest, paths = rt.validate_artifact_root(root, verify_hashes=True)\n    if manifest != expected or len(paths) != 1 or rt.N_MODELS != 1:\n        raise RuntimeError(\'D4 must load exactly the reference SWA model\')\n    if Path(paths[0]).name != \'coatnet_d4_parent_rank8_swa2.pt\':\n        raise RuntimeError(\'D4 must use its reference rank-8 parent checkpoint\')\n    if manifest[\'slice_depth\'][\'adapter_sha256\'] != \'b7c48b19997bc7ecb7e997f8dcf491ec5701ff917d1dac4b369eb85ac97ae89a\':\n        raise RuntimeError(\'D4 reference adapter identity changed\')\n    if rt.GPU_BATCH_STUDIES != 2 or rt.BACKBONE_MICRO_IMAGES != 8:\n        raise RuntimeError(\'D4 reference batching changed\')\n    if rt.CUDNN_BENCHMARK is not False:\n        raise RuntimeError(\'D4 reference convolution policy changed\')\n    if rt.base.cv2.__version__ != \'4.12.0\' or rt.parent.timm.__version__ != \'1.0.22\':\n        raise RuntimeError(\'D4 reference dependency versions changed\')\n    if torch.cuda.device_count() != 2:\n        raise RuntimeError(\'D4 requires exactly two T4 GPUs\')\n    names = [torch.cuda.get_device_name(i) for i in range(2)]\n    if not all((\'T4\' in name for name in names)):\n        raise RuntimeError(f\'D4 reference device mismatch: {names}\')\n    return (manifest, paths)\n\ndef _d4_check_outputs(rt, manifest, receipt, output, competition):\n    import hashlib\n    from pathlib import Path\n    import numpy as np\n    import pandas as pd\n\n    def need(condition, message):\n        if not condition:\n            raise RuntimeError(\'D4 release check: \' + message)\n    need(receipt[\'status\'] == rt.SUBMISSION_STATUS, \'unsuccessful status\')\n    expected_fields = {\'models\': 1, \'fallback_studies\': 0, \'dicom_preparations_per_study\': 1, \'model_passes_per_study\': 1, \'rank_after_probability_average\': True, \'canonical_sagittal\': True, \'common_physical_triplet_warp\': False, \'resolution\': 384, \'slice_depth_arm\': \'d4_zones\', \'slice_depth_state_elements\': 3255, \'slice_depth_parent_checkpoint_sha256\': \'5de38e333e3184d7b52fb068183f0cc198d6cbabb6ffbe36b26338622e82d297\', \'fsx_attention_heads\': 2, \'fsx_attention_width\': 128, \'fsx_serving_delta_cap\': 0.3, \'fsx_training_delta_cap\': 0.5, \'head\': \'rank8_swa2_plus_d4_three_zone_expanded_fsx\', \'slice_depth_swa_member_epochs\': [11, 9, 5], \'slice_depth_checkpoint_sha256\': manifest[\'slice_depth\'][\'adapter_sha256\']}\n    for key, value in expected_fields.items():\n        need(receipt[key] == value, key)\n    need(1 <= int(receipt[\'maximum_eval_windows\']) <= 94, \'window limit\')\n    checkpoints = receipt[\'checkpoints\']\n    need(len(checkpoints) == 1, \'parent checkpoint count\')\n    need(checkpoints[0][\'sha256\'] == \'5de38e333e3184d7b52fb068183f0cc198d6cbabb6ffbe36b26338622e82d297\', \'parent checkpoint hash\')\n    need(checkpoints[0][\'member_epochs\'] == [15, 16], \'parent SWA member epochs\')\n    need(receipt[\'slice_depth_checkpoint_sha256\'] == \'b7c48b19997bc7ecb7e997f8dcf491ec5701ff917d1dac4b369eb85ac97ae89a\', \'adapter checkpoint hash\')\n    processes = receipt[\'processes\']\n    need(len(processes) == 2 and all((int(p[\'returncode\']) == 0 for p in processes)), \'worker process failure\')\n    shards = receipt[\'shards\']\n    need(len(shards) == 2, \'shard count\')\n    for s in shards:\n        need(s[\'device\'] == \'cuda\', \'non-CUDA shard\')\n        need(int(s[\'fallback_studies\']) == 0 and int(s[\'preparation_warnings\']) == 0, \'failed study/preparation\')\n        need(int(s[\'models_resident\']) == 1, \'resident model count\')\n        need(int(s[\'peak_reserved_bytes\']) < int(15.5 * 1024 ** 3), \'reference memory gate\')\n        need(s[\'specialized_worker_loader_contract\'] == manifest[\'worker_loader_contract\'], \'specialized loader\')\n    output = Path(output)\n    prediction_path = output.parent / rt.PREDICTIONS_NAME\n    with np.load(prediction_path, allow_pickle=False) as payload:\n        uids = payload[\'study_uids\'].astype(str).tolist()\n        raw = np.asarray(payload[\'raw_probabilities\'], dtype=np.float32)\n        mean = np.asarray(payload[\'probability_mean\'], dtype=np.float32)\n        ranked = np.asarray(payload[\'submission_percentile_rank\'], dtype=np.float64)\n    ids = pd.read_csv(Path(competition) / \'test.csv\', dtype={\'StudyInstanceUID\': str}).StudyInstanceUID.tolist()\n    need(len(ids) > 0 and len(ids) == len(set(ids)), \'test identity\')\n    need(uids == ids and int(receipt[\'studies\']) == len(ids), \'prediction identity/count\')\n    need(raw.shape == (1, len(ids), 12), \'raw probability shape\')\n    need(np.isfinite(raw).all() and ((raw >= 0) & (raw <= 1)).all(), \'raw probabilities\')\n    need(np.array_equal(raw.astype(np.float64).mean(0).astype(np.float32), mean), \'probability mean\')\n    need(np.array_equal(rt.percentile_rank64(mean), ranked), \'global percentile ranks\')\n    frame = pd.read_csv(output, dtype={\'StudyInstanceUID\': str}, float_precision=\'round_trip\')\n    need(frame.columns.tolist() == [\'StudyInstanceUID\', *rt.base.LABELS], \'CSV labels\')\n    need(frame.StudyInstanceUID.tolist() == ids, \'CSV identity\')\n    need(np.array_equal(frame.iloc[:, 1:].to_numpy(np.float64), ranked), \'CSV numeric round trip\')\n    actual_hash = hashlib.sha256(output.read_bytes()).hexdigest()\n    need(actual_hash == receipt[\'output_sha256\'], \'CSV hash\')\n    receipt[\'reference_wrapper_checks\'] = {\'export_sha256\': \'7f8410fdf1b60cd3831d50f517bb25798a4e4a3404c59dc269193d5fe7cce729\', \'validated_manifest_sha256\': \'7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d\', \'probability_mean_exact\': True, \'ranks_exact\': True, \'csv_roundtrip_exact\': True, \'source_and_model_hashes_validated\': True, \'input_identity_equals_original_grid\': False}\n    return receipt\n\ndef _run_required_child(command, environment, logfile):\n    import os\n    import signal\n    import subprocess\n    import time\n    from pathlib import Path\n    remaining = TIME_BUDGET - (time.time() - T0)\n    if remaining <= 0:\n        raise TimeoutError(\'No time remains for a required model branch\')\n    path = Path(logfile)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\'w\') as log_handle:\n        process = subprocess.Popen(command, env=environment, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True)\n        try:\n            returncode = process.wait(timeout=remaining)\n        except BaseException:\n            try:\n                os.killpg(process.pid, signal.SIGTERM)\n            except ProcessLookupError:\n                pass\n            try:\n                process.wait(timeout=10)\n            except subprocess.TimeoutExpired:\n                try:\n                    os.killpg(process.pid, signal.SIGKILL)\n                except ProcessLookupError:\n                    pass\n                process.wait()\n            raise\n    if returncode != 0:\n        with path.open(\'rb\') as handle:\n            handle.seek(max(0, path.stat().st_size - 5000))\n            tail = handle.read().decode(\'utf-8\', errors=\'replace\')\n        raise RuntimeError(f\'Required model branch failed ({returncode}); {path}\\n{tail}\')\n    return returncode\nASSET_ROOTS = [\'/kaggle/input/datasets/dreaddevelopment/raptor-knee-maxspan\', \'/kaggle/input/raptor-knee-maxspan\', \'/kaggle/input/datasets/dreaddevelopment/raptor-knee-native384\', \'/kaggle/input/raptor-knee-native384\', \'/kaggle/input/datasets/dreaddevelopment/raptor-knee-native384dense\', \'/kaggle/input/raptor-knee-native384dense\', \'/kaggle/input/datasets/mattiaangeli/knee-mri-fold-weights\', \'/kaggle/input/knee-mri-fold-weights\', \'/kaggle/input/datasets/mattiaangeli/opencv-python-headless-4120088-x86\', \'/kaggle/input/opencv-python-headless-4120088-x86\', \'/kaggle/input/datasets/marwanmath/resnet-50-radimagenet-marwan\', \'/kaggle/input/resnet-50-radimagenet-marwan\', \'/kaggle/input/datasets/mattiaangeli/rsna-knee-coat-resgated-ep10-top3\', \'/kaggle/input/rsna-knee-coat-resgated-ep10-top3\', \'/kaggle/input/datasets/antoinegg1/rsna-knee-e11-diverse-heads-v20\', \'/kaggle/input/rsna-knee-e11-diverse-heads-v20\', \'/kaggle/input/datasets/antoinegg1/rsna-knee-e9-radimagenet-heads-v15\', \'/kaggle/input/rsna-knee-e9-radimagenet-heads-v15\', \'/kaggle/input/datasets/pilkwang/rsna-knee-llm-labels\', \'/kaggle/input/rsna-knee-llm-labels\', \'/kaggle/input/datasets/prvsiyan/rsna-knee-v52-radimagenet-heads-20260812\', \'/kaggle/input/rsna-knee-v52-radimagenet-heads-20260812\', \'/kaggle/input/datasets/pilkwang/rsna-knee-weights\', \'/kaggle/input/rsna-knee-weights\', \'/kaggle/input/datasets/mattiaangeli/rsna-knee-coatnet-d4-depthzone-swa3-b2\', \'/kaggle/input/rsna-knee-coatnet-d4-depthzone-swa3-b2\', \'/kaggle/input/datasets/mattiaangeli/rsna-knee-coatnet-global96-top3\', \'/kaggle/input/rsna-knee-coatnet-global96-top3\', \'/kaggle/input/notebooks/sofiaanjenje/rsna-knee-e11-train\', \'/kaggle/input/rsna-knee-e11-train\', \'/kaggle/input/notebooks/sofiaanjenje/rsna-knee-e13-train\', \'/kaggle/input/rsna-knee-e13-train\', \'/kaggle/input/models/metaresearch/dinov2/pytorch/small/1\', \'/kaggle/input/dinov2/pytorch/small/1\']\n\ndef _asset_walk():\n    from pathlib import Path\n    seen = set()\n    for root in ASSET_ROOTS:\n        root = Path(root).resolve()\n        if not root.is_dir():\n            continue\n        level = [root]\n        for _depth in range(6):\n            following = []\n            for directory in sorted(level):\n                if str(directory) in seen:\n                    continue\n                seen.add(str(directory))\n                children = sorted(directory.iterdir())\n                dirs = [p for p in children if p.is_dir() and p.name not in (\'train_series\', \'test_series\', \'train_images\', \'test_images\')]\n                files = [p.name for p in children if p.is_file()]\n                yield (str(directory), [p.name for p in dirs], files)\n                following.extend(dirs)\n            level = following\n\ndef _asset_find_asset(name, digest=None):\n    import fnmatch, hashlib\n    from pathlib import Path\n    hits = []\n    for root, _, files in _asset_walk():\n        for file in files:\n            if fnmatch.fnmatchcase(file, name):\n                p = Path(root) / file\n                if digest:\n                    h = hashlib.sha256()\n                    with p.open(\'rb\') as handle:\n                        for block in iter(lambda: handle.read(8 << 20), b\'\'):\n                            h.update(block)\n                    if h.hexdigest() != digest:\n                        continue\n                if p.resolve() not in [h.resolve() for h in hits]:\n                    hits.append(p)\n    if len(hits) != 1:\n        raise RuntimeError(f\'expected one pinned asset {name}, got {hits}\')\n    return hits[0]\n\ndef _asset_half_rank_mix(*members):\n    if len(members) < 2:\n        raise ValueError(\'CoAt family rank mix needs at least two members\')\n    first = members[0]\n    for other in members[1:]:\n        if other.shape != first.shape or not other.index.equals(first.index):\n            raise ValueError(\'CoAt family alignment changed\')\n    for member in members:\n        if member.isna().any().any():\n            raise ValueError(\'missing CoAt family prediction\')\n    units = members[0].rank(method=\'average\')\n    for other in members[1:]:\n        units = units + other.rank(method=\'average\')\n    return units / (float(len(members)) * len(units))\n\ndef _asset_write_coat_runtime(artifact, output, helper_source):\n    from pathlib import Path\n    import hashlib\n    source = Path(artifact) / \'coatnet_resgated_ep10_top3_inference.py\'\n    text = source.read_text()\n    if hashlib.sha256(source.read_bytes()).hexdigest() != \'b11e58f8d7cabe9e264ac70b01e811dc6a846aa01248ace1f0e6536d0d4094d9\':\n        raise RuntimeError(\'unexpected parent CoAt runtime\')\n    edits = {\'if not 1 <= windows <= MAX_EVAL_WINDOWS:\': \'if not 1 <= windows <= 94:\', \'or int(counts.max()) > MAX_EVAL_WINDOWS:\': \'or int(counts.max()) > 94:\', \'"runtime_contract": RUNTIME_CONTRACT,\': \'"runtime_contract": RUNTIME_CONTRACT, "input_max_windows": 94,\', \'"eval_grid": "saved_training_gold_unique_center_slot_budgets",\': \'"eval_grid": "input_unique_native_centers_six_slot_v1",\'}\n    for old, new in edits.items():\n        if old not in text:\n            raise RuntimeError(\'CoAt patch target missing: \' + old)\n        text = text.replace(old, new)\n    text = text.replace(\'"gold58_rank_ensemble_auc": manifest["selection"]\', \'"historical_parent_gold58_rank_ensemble_auc": manifest["selection"]\')\n    text = text.replace(\'"t4_gold58_rank_ensemble_auc": manifest["t4_numerical_portability"]\', \'"historical_parent_t4_gold58_rank_ensemble_auc": manifest["t4_numerical_portability"]\')\n    marker = \'if __name__ == "__main__":\'\n    if text.count(marker) != 1:\n        raise RuntimeError(\'CoAt entrypoint changed\')\n    patch = helper_source + (\'\\n_original_train_faithful_eval_specs = train_faithful_eval_specs\\n\'\n                             \'def _tolerant_coat_specs(study):\\n    try:\\n        return _dense_coat_specs(study)\\n\'\n                             \'    except Exception as exc:\\n        print("[dense-fallback] " + str(getattr(study, "study_uid", "?")) + ": " + type(exc).__name__ + ": " + str(exc), flush=True)\\n\'\n                             \'        return _original_train_faithful_eval_specs(study)\\n\'\n                             \'train_faithful_eval_specs = _tolerant_coat_specs\\nRUNTIME_CONTRACT = "resgated_ep10_input_nativecenters_v1"\\n\')\n    text = text.replace(marker, patch + \'\\n\' + marker)\n    compile(text, str(output), \'exec\')\n    Path(output).write_text(text)\n    return hashlib.sha256(text.encode()).hexdigest()\n\ndef _asset_patch_d4(rt, grid_source):\n    import ast\n    import hashlib\n    import inspect\n    import json\n    import os\n    import textwrap\n    from pathlib import Path\n    audited = rt.parent.audited\n    original = audited.prepare_global96_bag\n    original_shard = rt._PARENT_INFER_SHARD\n    if getattr(original, \'_input_patch_installed\', False):\n        raise RuntimeError(\'D4 input preparation was already patched\')\n    src = textwrap.dedent(inspect.getsource(original))\n    tree = ast.parse(src)\n    fn = next((n for n in tree.body if isinstance(n, ast.FunctionDef)), None)\n    if fn is None or fn.name != \'prepare_global96_bag\':\n        raise RuntimeError(\'D4 preparation function identity changed\')\n    if \'study_uid\' not in inspect.signature(original).parameters:\n        raise RuntimeError(\'D4 preparation study-UID contract changed\')\n    candidates = [n for n in ast.walk(fn) if isinstance(n, ast.Assign) and any((isinstance(t, ast.Name) and t.id == \'specs\' for t in n.targets)) and isinstance(n.value, ast.Call) and isinstance(n.value.func, ast.Attribute) and isinstance(n.value.func.value, ast.Name) and (n.value.func.value.id == \'global_stack\') and (n.value.func.attr == \'sample_global_windows\')]\n    if len(candidates) != 1 or candidates[0] not in fn.body:\n        raise RuntimeError(\'D4 expected one top-level Global96 specs constructor\')\n    call = candidates[0]\n    keywords = {k.arg: k.value for k in call.value.keywords}\n    if not (isinstance(keywords.get(\'count\'), ast.Constant) and keywords[\'count\'].value is None and isinstance(keywords.get(\'train\'), ast.Constant) and (keywords[\'train\'].value is False)):\n        raise RuntimeError(\'D4 original evaluation must use count=None, train=False\')\n    exec(compile(grid_source, \'<input-grid>\', \'exec\'), audited.__dict__)\n    audit_dir = Path(os.environ[\'RSNA_D4_INPUT_AUDIT_DIR\'])\n    audit_dir.mkdir(parents=True, exist_ok=True)\n\n    def record_input(study_uid, stack, audit):\n        import numpy as np\n        arrays = (\'global_indices\', \'nominal_slots\', \'canonical_depths\', \'nominal_steps\', \'source_series_rows\', \'source_slot_ids\')\n        digest = hashlib.sha256()\n        for name in arrays:\n            value = np.ascontiguousarray(getattr(stack, name))\n            digest.update(name.encode())\n            digest.update(str(value.dtype).encode())\n            digest.update(str(value.shape).encode())\n            digest.update(value.tobytes())\n        record = {\'study_uid\': str(study_uid), \'pid\': os.getpid(), \'positions\': int(len(stack.global_indices)), \'windows_expected\': int(len(stack.global_indices) - 2), \'slot_widths\': list(map(int, stack.slot_widths)), \'stack_sidecars_sha256\': digest.hexdigest(), \'slots\': audit}\n        with (audit_dir / f\'input_{os.getpid()}.jsonl\').open(\'a\') as handle:\n            handle.write(json.dumps(record, sort_keys=True, allow_nan=False) + \'\\n\')\n    audited.__dict__[\'_record_input\'] = record_input\n    injection = ast.parse(\'try:\\n    stack, input_audit = _dense_stack(stack, study.offsets, study.valid, study.source_depth)\\n\'\n                          \'except Exception as _dense_exc:\\n    print("[dense-fallback] " + str(study_uid) + ": " + type(_dense_exc).__name__ + ": " + str(_dense_exc), flush=True)\\n\'\n                          \'    input_audit = [{"fallback": type(_dense_exc).__name__ + ": " + str(_dense_exc)}]\\n\'\n                          \'_record_input(study_uid, stack, input_audit)\\n\').body\n    where = fn.body.index(call)\n    fn.body[where:where] = injection\n    ast.fix_missing_locations(tree)\n    exec(compile(tree, \'<D4-input-preparation-only>\', \'exec\'), audited.__dict__)\n    patched = audited.prepare_global96_bag\n    patched._input_patch_installed = True\n    for module in (rt, rt.parent):\n        if getattr(module, \'prepare_global96_bag\', None) is original:\n            module.prepare_global96_bag = patched\n    if rt._PARENT_INFER_SHARD is not original_shard:\n        raise RuntimeError(\'D4 parent shard must remain the official callable\')\n    files = {}\n    for name, module in [(\'d4_runtime\', rt), (\'parent_runtime\', rt.parent), (\'audited_runtime\', audited)]:\n        path = Path(module.__file__)\n        if path.is_file():\n            files[name] = {\'file\': path.name, \'sha256\': hashlib.sha256(path.read_bytes()).hexdigest()}\n    receipt = {\'contract\': \'d4-original-runtime-input-preparation-only-v2\', \'reference_script_version_id\': 348764100, \'reference_export_available\': True, \'original_prepare_source_sha256\': hashlib.sha256(src.encode()).hexdigest(), \'patched_prepare_source_sha256\': hashlib.sha256(ast.unparse(tree).encode()).hexdigest(), \'model_loader_modified\': False, \'zone_head_modified\': False, \'parent_infer_shard_replaced\': False, \'grid_source_sha256\': hashlib.sha256(grid_source.encode()).hexdigest(), \'artifact_source_files\': files}\n    (audit_dir / f\'patch_{os.getpid()}.json\').write_text(json.dumps(receipt, indent=2))\n    return receipt\n\ndef _asset_run_d4(artifact, environment_dir, competition, output):\n    import hashlib, inspect, json, os, sys\n    from pathlib import Path\n    import pandas as pd\n    artifact, output = (Path(artifact), Path(output))\n    manifest = artifact / \'coatnet_pairfilm_manifest.json\'\n    if hashlib.sha256(manifest.read_bytes()).hexdigest() != \'7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d\':\n        raise RuntimeError(\'D4 SWA3 manifest changed\')\n    grid = \'import numpy as np\\nfrom raptor_global_stack_smart336 import FixedGlobalStack, INFERRED96_WIDTHS\\n\' + \'\\n\'.join((inspect.getsource(f) for f in (_dense_quotas, _dense_stack)))\n    audit_dir = output.parent / \'study_input_receipts\'\n    audit_dir.mkdir(parents=True, exist_ok=True)\n    for stale in audit_dir.glob(\'*.json*\'):\n        stale.unlink()\n    output.unlink(missing_ok=True)\n    output.with_suffix(\'.receipt.json\').unlink(missing_ok=True)\n    wrapper = output.parent / \'d4_worker_entry.py\'\n    code = f\'import sys\\nsys.path.insert(0,{str(environment_dir)!r})\\nsys.path.insert(0,{str(artifact)!r})\\nfrom pathlib import Path\\nimport json\\nimport coatnet_d4_depthzone_swa_inference as rt\\n\' + inspect.getsource(_asset_patch_d4) + \'\\n\' + inspect.getsource(_d4_check_runtime) + \'\\n\' + inspect.getsource(_d4_check_outputs) + \'\\n\' + f"if __name__ == \'__main__\' and \'--worker-output\' not in sys.argv:\\n    _checked_manifest, _checked_paths = _d4_check_runtime(rt, Path({str(artifact)!r}))\\n" + f\'_asset_patch_d4(rt,{grid!r})\\n\' + \'rt.parent.audited.__file__ = __file__\\n\' + "if __name__ == \'__main__\':\\n" + "    if \'--worker-output\' in sys.argv:\\n        raise SystemExit(rt.main())\\n" + \'    manifest, paths = _checked_manifest, _checked_paths\\n\' + \'    r = rt.run_submission(\\n\' + f\'        competition_root=Path({str(competition)!r}), artifact_root=Path({str(artifact)!r}),\\n\' + f\'        output_path=Path({str(output)!r}), gpu_batch_studies=2, backbone_micro_images=8)\\n\' + f\'    r = _d4_check_outputs(rt, manifest, r, Path({str(output)!r}), Path({str(competition)!r}))\\n\' + f"    Path({str(output.with_suffix(\'.receipt.json\'))!r}).write_text(json.dumps(r,indent=2,allow_nan=False))\\n"\n    compile(code, str(wrapper), \'exec\')\n    wrapper.write_text(code)\n    env = dict(os.environ)\n    env[\'RSNA_D4_INPUT_AUDIT_DIR\'] = str(audit_dir)\n    env[\'PYTHONPATH\'] = f\'{environment_dir}:/kaggle/working/_coat_env:{artifact}:\' + env.get(\'PYTHONPATH\', \'\')\n    env.pop(\'CUDNN_CONV_WSCAP_DBG\', None)\n    _run_required_child([sys.executable, str(wrapper)], env, output.with_suffix(\'.log\'))\n    receipt = json.loads(output.with_suffix(\'.receipt.json\').read_text())\n    expected = pd.read_csv(Path(competition) / \'test.csv\', dtype={\'StudyInstanceUID\': str}).StudyInstanceUID.tolist()\n    records = []\n    for path in sorted(audit_dir.glob(\'input_*.jsonl\')):\n        records.extend((json.loads(line) for line in path.read_text().splitlines() if line.strip()))\n    ids = [r[\'study_uid\'] for r in records]\n    if len(ids) != len(expected) or len(ids) != len(set(ids)) or set(ids) != set(expected):\n        raise RuntimeError(\'D4 input hook must execute exactly once for every test study\')\n    if not all((1 <= r[\'windows_expected\'] <= 94 and r[\'positions\'] == r[\'windows_expected\'] + 2 for r in records)):\n        raise RuntimeError(\'D4 input record cardinality drift\')\n    receipt[\'input_coverage\'] = {\'studies\': len(records), \'folder\': str(audit_dir), \'min_windows\': min((r[\'windows_expected\'] for r in records)), \'max_windows\': max((r[\'windows_expected\'] for r in records)), \'original_model_loader_retained\': True, \'original_shard_scheduler_retained\': True}\n    output.with_suffix(\'.receipt.json\').write_text(json.dumps(receipt, indent=2, allow_nan=False))\n    return receipt\n_speed_original_asset_walk = _asset_walk\n_speed_original_find_asset = _asset_find_asset\n\'Catalogue named immutable attachments once; no competition-tree search.\'\nimport threading as _speed_threading\n_speed_asset_lock = _speed_threading.RLock()\n_speed_asset_catalogue = None\n_speed_asset_hits = {}\n\ndef _asset_walk():\n    global _speed_asset_catalogue\n    with _speed_asset_lock:\n        if _speed_asset_catalogue is None:\n            _speed_asset_catalogue = tuple(((root, tuple(dirs), tuple(files)) for root, dirs, files in _speed_original_asset_walk()))\n        catalogue = _speed_asset_catalogue\n    for root, dirs, files in catalogue:\n        yield (root, list(dirs), list(files))\n\ndef _asset_find_asset(name, digest=None):\n    key = (str(name), digest)\n    with _speed_asset_lock:\n        if key not in _speed_asset_hits:\n            _speed_asset_hits[key] = _speed_original_find_asset(name, digest)\n        return _speed_asset_hits[key]\n', globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(3, "import numpy as np\n\ndef _dense_allocate(capacities, proportions, target):\n    capacities = np.asarray(capacities, dtype=np.int64)\n    proportions = np.asarray(proportions, dtype=np.float64)\n    if capacities.ndim != 1 or capacities.shape != proportions.shape or np.any(capacities < 0) or (not np.isfinite(proportions).all()) or np.any(proportions <= 0) or (int(target) < 1):\n        raise ValueError('invalid capacity-aware sampling allocation')\n    ideal = int(target) * proportions / proportions.sum()\n    initial = np.floor(ideal).astype(np.int64)\n    for i in np.argsort(-(ideal - initial), kind='stable')[:int(target) - int(initial.sum())]:\n        initial[i] += 1\n    quotas = np.minimum(initial, capacities)\n    wanted = min(int(target), int(capacities.sum()))\n    deficit = np.zeros(len(quotas), dtype=np.float64)\n    while int(quotas.sum()) < wanted:\n        donors = quotas < capacities\n        weights = np.where(donors, proportions, 0.0)\n        deficit[donors] += weights[donors] / weights.sum()\n        chosen = int(np.argmax(np.where(donors, deficit, -np.inf)))\n        quotas[chosen] += 1\n        deficit[chosen] -= 1\n    assert int(quotas.sum()) == wanted and np.all(quotas <= capacities)\n    return quotas\n\ndef _dense_unique_linspace(capacity, count):\n    if not 0 <= count <= capacity:\n        raise ValueError('cannot create unique picks beyond physical capacity')\n    result = np.linspace(0, capacity - 1, count).round().astype(np.int64)\n    if count and (result.min() < 0 or result.max() >= capacity):\n        raise AssertionError('out-of-range source')\n    assert len(np.unique(result)) == count\n    return result\n\ndef _dense_coat_specs(study):\n    import raptor_light224 as sampler\n    capacities = []\n    for row in study.series_rows:\n        if int(row) < 0:\n            capacities.append(0)\n        else:\n            centers, _, _, _ = sampler._candidate_centers(int(row), study.offsets, study.valid, study.source_depth)\n            capacities.append(len(centers))\n    if sum(capacities) < 1:\n        raise ValueError('no usable CoAt triplets')\n    quotas = _dense_allocate(capacities, (20, 12, 12, 8, 16, 8), 94)\n    specs = sampler.sample_eval_windows(study.series_rows, study.offsets, study.valid, study.source_depth, count=None, slot_budgets=np.maximum(quotas, 1).tolist())\n    if len(specs) != min(94, sum(capacities)):\n        raise RuntimeError('capacity-aware sampling window-count drift')\n    keys = [(s.series_row, s.center_local) for s in specs]\n    if len(keys) != len(set(keys)):\n        raise RuntimeError('capacity-aware sampling repeated a native center')\n    for s in specs:\n        start = int(study.offsets[s.series_row])\n        if not np.all(study.valid[start + np.asarray(s.local_indices)]):\n            raise RuntimeError('capacity-aware sampling invalid source channel')\n    return specs\nINFERRED96_WIDTHS = (27, 21, 18, 12, 18)\n\ndef _dense_quotas(capacities) -> np.ndarray:\n    capacities = np.asarray(capacities, dtype=np.int64)\n    base = np.asarray(INFERRED96_WIDTHS, dtype=np.int64)\n    if capacities.shape != (5,) or bool((capacities < 0).any()):\n        raise ValueError('expected five nonnegative source capacities')\n    quotas = np.minimum(base, capacities)\n    target = min(96, int(capacities.sum()))\n    deficit = np.zeros(5, dtype=np.float64)\n    while int(quotas.sum()) < target:\n        donors = quotas < capacities\n        weights = np.where(donors, base, 0).astype(np.float64)\n        deficit[donors] += weights[donors] / weights.sum()\n        selected = int(np.argmax(np.where(donors, deficit, -np.inf)))\n        quotas[selected] += 1\n        deficit[selected] -= 1\n    if int(quotas.sum()) != target or bool((quotas > capacities).any()):\n        raise RuntimeError('additional quota allocation changed')\n    return quotas\n\ndef _dense_stack(reference, offsets, valid, source_depth):\n    from types import SimpleNamespace\n    from raptor_global_stack_smart336 import FixedGlobalStack\n    if tuple(reference.slot_widths) != tuple(INFERRED96_WIDTHS):\n        raise ValueError('capacity-aware sampling is pinned to the Global96 reference layout')\n    pools = []\n    audit = []\n    begin = 0\n    for slot, width in enumerate(reference.slot_widths, 1):\n        end = begin + width\n        old = reference.global_indices[begin:end]\n        row = int(reference.source_series_rows[begin])\n        info = {'slot': slot, 'original_width': width, 'source_row': row}\n        audit.append(info)\n        if row < 0 or bool((old < 0).all()):\n            pools.append((np.empty(0, dtype=np.int64), np.empty(0, dtype=np.float32)))\n            begin = end\n            continue\n        start, stop = map(int, offsets[row:row + 2])\n        if not 0 <= start < stop <= len(valid) == len(source_depth):\n            raise ValueError('source offset drift')\n        depth = np.asarray(source_depth[start:stop], dtype=np.float32)\n        keep = np.asarray(valid[start:stop], dtype=bool) & np.isfinite(depth) & (depth >= 0.02) & (depth <= 0.98)\n        eligible = np.flatnonzero(keep).astype(np.int64) + start\n        reverse = bool(old[0] > old[-1] or (old[0] == old[-1] and np.float32(source_depth[old[0]]) != reference.canonical_depths[begin]))\n        if reverse:\n            eligible = eligible[::-1].copy()\n        depth = np.asarray(source_depth[eligible], dtype=np.float32)\n        if reverse:\n            depth = (1.0 - depth).astype(np.float32)\n        pools.append((eligible, depth))\n        begin = end\n    quotas = _dense_quotas([len(pool[0]) for pool in pools])\n    if int(quotas.sum()) < 3:\n        raise ValueError('study has fewer than three usable unique source slices')\n    parts = {key: [] for key in ('global_indices', 'nominal_slots', 'canonical_depths', 'nominal_steps', 'source_series_rows', 'source_slot_ids')}\n    for slot, ((eligible, depth), quota, info) in enumerate(zip(pools, quotas, audit), 1):\n        quota = int(quota)\n        info.update(eligible=len(eligible), allocated_width=quota, additional=max(0, quota - info['original_width']))\n        info['reason'] = 'missing_no_padding' if not quota else 'short_slot_all_unique' if len(eligible) < info['original_width'] else 'uniform_native_centers' if info['additional'] else 'uniform_native_centers'\n        positions = np.linspace(0, len(eligible) - 1, quota).round().astype(np.int64)\n        if len(positions) != quota or not bool((np.diff(positions) > 0).all()):\n            raise RuntimeError('capacity-aware sampling slot selection repeated a source')\n        mask = reference.nominal_slots == slot\n        source_slot = int(reference.source_slot_ids[mask][0])\n        parts['global_indices'].append(eligible[positions])\n        parts['canonical_depths'].append(depth[positions])\n        parts['nominal_slots'].append(np.full(quota, slot, dtype=np.int8))\n        parts['nominal_steps'].append(np.full(quota, 0.96 / max(quota - 1, 1), dtype=np.float32))\n        parts['source_series_rows'].append(np.full(quota, info['source_row'], dtype=np.int32))\n        parts['source_slot_ids'].append(np.full(quota, source_slot, dtype=np.int8))\n    result = FixedGlobalStack(**{key: np.concatenate(value) for key, value in parts.items()}, slot_widths=tuple((int(value) for value in quotas)))\n    if len(np.unique(result.global_indices)) != len(result.global_indices) or bool((result.global_indices < 0).any()):\n        raise RuntimeError('capacity-aware sampling output contains a repeated/missing source')\n    return (result, audit)\n", globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(4, 'from __future__ import annotations\nimport time as _startup_time\n_STARTUP_T0 = _startup_time.perf_counter()\nimport os\nfor _v in (\'OMP_NUM_THREADS\', \'OPENBLAS_NUM_THREADS\', \'MKL_NUM_THREADS\'):\n    os.environ.setdefault(_v, \'4\')\nimport gc\nimport hashlib\nimport json\nimport re\nimport time\nimport traceback\nimport threading\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport pydicom\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\ndef _cuda_execution_probe(index):\n    dev = torch.device(f\'cuda:{index}\')\n    try:\n        major, minor = torch.cuda.get_device_capability(index)\n        probe = nn.Conv2d(3, 4, kernel_size=3, padding=1).eval().to(dev)\n        with torch.inference_mode():\n            out = probe(torch.zeros((1, 3, 16, 16), device=dev))\n            if tuple(out.shape) != (1, 4, 16, 16):\n                raise RuntimeError(f\'unexpected CUDA probe shape {tuple(out.shape)}\')\n        torch.cuda.synchronize(index)\n        print(f\'cuda:{index} probe PASS (compute {major}.{minor})\')\n        del probe, out\n        torch.cuda.empty_cache()\n        return True\n    except Exception as exc:\n        print(f\'cuda:{index} probe FAIL ({type(exc).__name__}: {exc}); rejecting device\')\n        try:\n            torch.cuda.empty_cache()\n        except Exception:\n            pass\n        return False\nDEVS = []\nif torch.cuda.is_available():\n    DEVS = [torch.device(f\'cuda:{i}\') for i in range(torch.cuda.device_count()) if _cuda_execution_probe(i)]\nif len(DEVS) != 2:\n    raise RuntimeError(\'This complete ensemble requires two working GPUs; no CPU/single-GPU partial fallback\')\nif any(\'T4\' not in torch.cuda.get_device_name(d) for d in DEVS):\n    raise RuntimeError(\'Select the reference T4 x2 accelerator\')\nprint(f\'devices: {[str(d) for d in DEVS]}\')\n_STARTUP_GPU_IMPORT_S = _startup_time.perf_counter() - _STARTUP_T0\nprint(f\'[startup] imports+GPU preflight: {_STARTUP_GPU_IMPORT_S:.2f}s\', flush=True)\nT0 = time.time()\nSEED = 2026\nnp.random.seed(SEED)\ntorch.manual_seed(SEED)\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\', \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\nCROP_MM = 130.0\nCACHE_IMG = 336\nGROUP = 3\nN_GROUP_MAX = 1\nCACHE_FRACTION = 0.45\nCACHE_BUDGET_MAX_GB = 24.0\nCACHE_BUDGET_GB = 12.0\nTEST_SHARE = 0.3\nHDR_THREADS = 16\nPIX_THREADS = 12\nORDER_THREADS = 32\nORDER_BUDGET_S = 5400\nRUNS = [{\'name\': \'r224\', \'img\': 224}, {\'name\': \'r336\', \'img\': 336}]\nEPOCHS = 10\nBATCH_STUDIES = 8\nAUG_ROT_DEG = 8.0\nAUG_SCALE = 0.08\nAUG_SHIFT = 0.05\nAUG_INTENSITY = 0.1\nLAT_MIN_OFFSET_MM = 20.0\nSLICE_BAND = (0.2, 0.8)\nRULES_NATIVE = {\'order\': \'normal\', \'lat\': \'centre\', \'slot_fallback\': False, \'decode_fill\': \'nearest\'}\nRULES_LEGACY = {\'order\': \'dominant_axis\', \'lat\': \'corner_x\', \'slot_fallback\': True, \'decode_fill\': \'zero\'}\nRULES = dict(RULES_NATIVE)\nLEGACY_LAT_OFFSET_MM = 5.0\nLR_HEAD = 0.001\nLR_BACKBONE = 8e-06\nUNFREEZE_LAST = 6\nWEIGHT_DECAY = 0.02\nEVAL_BATCH = 8\nTIME_BUDGET = 8.0 * 3600\nSLOTS_RECOVERED = [(\'SAG_FLUID_FS\', \'Sagittal\', True, True), (\'COR_FLUID_FS\', \'Coronal\', True, True), (\'AX_FLUID_FS\', \'Axial\', True, True), (\'SAG_FLUID_NOFS\', \'Sagittal\', True, False), (\'COR_T1\', \'Coronal\', False, False), (\'SAG_T1\', \'Sagittal\', False, False)]\nSLOTS_PUBLIC = [(\'SAG_FLUID\', \'Sagittal\', None, True), (\'COR_FLUID\', \'Coronal\', None, True), (\'AX_FLUID\', \'Axial\', None, True), (\'SAG_STRUCT\', \'Sagittal\', None, False), (\'COR_STRUCT\', \'Coronal\', None, False), (\'AX_STRUCT\', \'Axial\', None, False)]\nSLOT_SCHEME = os.environ.get(\'SLOT_SCHEME\', \'recovered\')\nSLOTS = SLOTS_PUBLIC if SLOT_SCHEME == \'public\' else SLOTS_RECOVERED\nN_SLOT = len(SLOTS)\nPOOL_PARTS = {\'cls_mean\': 2, \'cls_mean_focal\': 3}\nSLOT_PRIOR_TABLE = {\'ACL\': (0, 3, 5), \'MCL\': (1, 4), \'Medial Meniscus\': (0, 1, 3, 4), \'Lateral Meniscus\': (0, 1, 3, 4), \'Medial OA\': (1, 4, 5), \'Lateral OA\': (1, 4, 5), \'PF OA\': (0, 2, 5), \'Effusion\': (0, 2), \'Synovitis\': (0, 2), "Baker\'s": (0,), \'Contusion\': (0, 1, 2), \'Fracture\': (0, 1, 2, 4, 5)}\nSLOT_PRIOR_STRENGTH = 0.55\nFATSAT_OPTS = {\'FS\', \'FATSAT\', \'FAT_SAT\', \'FSAT\'}\n_SEP = re.compile(\'[_\\\\-.]\')\n_FATSAT_RX = re.compile(\'\\\\bfs\\\\b|fatsat|fat sat|\\\\bstir\\\\b|\\\\bspair\\\\b|\\\\bspir\\\\b|\\\\bwe\\\\b|water excit|\\\\btirm\\\\b|\\\\bsting\\\\b|\\\\bfatsup\\\\b|smart fat|\\\\bwater\\\\b\')  # 2026-09-17: GE Dixon water images (\'SMART FAT\', \'Water: SMART FAT\', seq FSEfw) are fat-suppressed; 37/4407 training studies had no fat-sat series detected before\n_T1_RX = re.compile(\'\\\\bt1\\\\b|\\\\bt1w\\\\b\')\n_T2_RX = re.compile(\'\\\\bt2\\\\b|\\\\bt2w\\\\b\')\n_PD_RX = re.compile(\'\\\\bpd\\\\b|\\\\bpdw\\\\b|proton|\\\\bdp\\\\b|dens\')\n\n# Runtime integrity: no partial ensemble is ever published as submission.csv.\nimport os, json, hashlib, time, threading, tempfile\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\n_RSNA_AUDIT_LOCK = threading.RLock()\n_RSNA_AUDIT = {\'contract\': \'btkd_speedy_v558_input_uniform\', \'events\': [], \'phases\': {}}\n_RSNA_ORDER_MEMO = {}\n_RSNA_HEADERS_MEMO = {}\n_RSNA_CACHE_FILES = []\n_RSNA_TEST_IDS = None\n_RSNA_LABELS = [\'ACL\',\'MCL\',\'Medial Meniscus\',\'Lateral Meniscus\',\'Medial OA\',\'Lateral OA\',\'PF OA\',\'Effusion\',\'Synovitis\',"Baker\'s",\'Contusion\',\'Fracture\']\n\ndef rsna_sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(8 << 20), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\ndef rsna_json(path, value):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(\'.\' + path.name + \'.tmp\')\n    with tmp.open(\'w\') as f:\n        json.dump(value, f, indent=2, sort_keys=True, allow_nan=False, default=str)\n        f.write(\'\\n\'); f.flush(); os.fsync(f.fileno())\n    os.replace(tmp, path)\n\ndef rsna_event(kind, **details):\n    with _RSNA_AUDIT_LOCK:\n        _RSNA_AUDIT[\'events\'].append({\'kind\':kind, **details})\n\ndef rsna_finite(values, tag, probability=False):\n    x = np.asarray(values)\n    if not np.issubdtype(x.dtype, np.number) or not np.isfinite(x).all():\n        raise RuntimeError(f\'{tag}: nonfinite/non-numeric values BEFORE ranking\')\n    if probability and x.size and (x.min() < 0 or x.max() > 1):\n        raise RuntimeError(f\'{tag}: probability outside [0,1]\')\n    return x\n\ndef rsna_rank01(values):\n    """Parent endpoint scale; tied values share their average rank."""\n    x = np.asarray(rsna_finite(values, \'rank01\'), dtype=np.float64)\n    if x.ndim != 2 or not len(x):\n        raise ValueError(\'rank01 requires a nonempty [study,finding] matrix\')\n    if len(x) == 1:\n        return np.full_like(x, .5)\n    return (pd.DataFrame(x).rank(method=\'average\').to_numpy() - 1) / (len(x)-1)\n\ndef rsna_rankpct(values):\n    x = np.asarray(rsna_finite(values, \'rankpct\'), dtype=np.float64)\n    if x.ndim != 2 or not len(x):\n        raise ValueError(\'rankpct requires a nonempty matrix\')\n    return pd.DataFrame(x).rank(method=\'average\', pct=True).to_numpy(np.float64)\n\ndef rsna_frame(frame, ids, labels, tag):\n    ids = [str(u) for u in ids]; labels = list(labels)\n    if len(ids) != len(set(ids)):\n        raise RuntimeError(f\'{tag}: duplicate expected UID\')\n    if frame.columns.tolist() != [\'StudyInstanceUID\', *labels]:\n        raise RuntimeError(f\'{tag}: label order/schema mismatch\')\n    frame = frame.copy(); frame[\'StudyInstanceUID\'] = frame[\'StudyInstanceUID\'].astype(str)\n    if frame.StudyInstanceUID.duplicated().any() or set(frame.StudyInstanceUID) != set(ids):\n        raise RuntimeError(f\'{tag}: missing/extra/duplicate UID\')\n    frame = frame.set_index(\'StudyInstanceUID\').loc[ids].reset_index()\n    rsna_finite(frame[labels].to_numpy(), tag, probability=True)\n    return frame\n\ndef rsna_save_predictions(tag, ids, values, labels=None):\n    x = rsna_finite(values, tag)\n    target = Path(\'/kaggle/working/diagnostics\'); target.mkdir(parents=True, exist_ok=True)\n    path = target/(tag+\'.npz\')\n    np.savez_compressed(path, study_uids=np.asarray(ids,dtype=str), labels=np.asarray(labels or _RSNA_LABELS,dtype=str), values=x)\n    with _RSNA_AUDIT_LOCK:\n        _RSNA_AUDIT[\'phases\'][tag]={\'path\':str(path),\'sha256\':rsna_sha(path),\'shape\':list(x.shape)}\n\ndef rsna_strict_load(model, state, tag):\n    """A random frozen encoder is never an acceptable missing-state fallback."""\n    expected = model.state_dict()\n    missing = sorted(set(expected)-set(state)); extra = sorted(set(state)-set(expected))\n    bad_shape = [k for k in set(expected)&set(state) if tuple(expected[k].shape)!=tuple(state[k].shape)]\n    if missing or extra or bad_shape:\n        raise RuntimeError(f\'{tag}: incomplete/mismatched checkpoint; missing={missing[:20]}, extra={extra[:20]}, shapes={bad_shape[:20]}. A partial encoder needs its exact pinned source weights; random initialization is forbidden.\')\n    for k,t in state.items():\n        if t.is_floating_point() and not bool(t.isfinite().all()):\n            raise RuntimeError(f\'{tag}: nonfinite checkpoint tensor {k}\')\n    return model.load_state_dict(state, strict=True)\n\ndef rsna_deadline(tag):\n    if time.time() - T0 > TIME_BUDGET:\n        raise TimeoutError(f\'{tag}: full-run budget exceeded; no incomplete output may be submitted\')\n\ndef rsna_array(shape, tag):\n    """Large temporary pixels belong in scratch space, not saved notebook outputs."""\n    size = int(np.prod(shape))\n    if size <= float(os.environ.get(\'RSNA_CACHE_RAM_GIB\', \'1\')) * (1 << 30):\n        return np.zeros(shape, np.uint8)\n    directory = Path(os.environ.get(\'RSNA_PIXEL_SCRATCH\', \'/kaggle/temp/rsna_pixels_v559\'))\n    try:\n        directory.mkdir(parents=True, exist_ok=True)\n    except OSError as scratch_error:  # /kaggle/temp is not guaranteed; /tmp is non-persisted scratch with ~1 TiB free\n        rsna_event(\'scratch_fallback\', tag=tag, requested=str(directory), error=f\'{type(scratch_error).__name__}: {scratch_error}\')\n        directory = Path(\'/tmp/rsna_pixels_v559\')\n        directory.mkdir(parents=True, exist_ok=True)\n    import shutil\n    free = shutil.disk_usage(directory).free\n    rsna_event(\'scratch_allocation\', tag=tag, bytes=size, directory=str(directory), free_bytes=int(free))\n    if free < size + (1 << 30) and str(directory) != \'/tmp/rsna_pixels_v559\':\n        rsna_event(\'scratch_fallback\', tag=tag, requested=str(directory), error=f\'insufficient free space ({free} bytes)\')\n        directory = Path(\'/tmp/rsna_pixels_v559\'); directory.mkdir(parents=True, exist_ok=True); free = shutil.disk_usage(directory).free\n    if free < size + (1 << 30):\n        raise RuntimeError(f\'{tag}: not enough scratch disk for complete cache ({size} bytes; {free} free)\')\n    fd, path = tempfile.mkstemp(prefix=\'pixels_\', suffix=\'.npy\', dir=directory)\n    os.close(fd)\n    try:\n        a = np.lib.format.open_memmap(path, mode=\'w+\', dtype=np.uint8, shape=tuple(shape))\n        a[:] = 0\n    except BaseException:\n        Path(path).unlink(missing_ok=True)\n        raise\n    _RSNA_CACHE_FILES.append(path)\n    return a\n\ndef rsna_release_pixels(array):\n    if isinstance(array,np.memmap):\n        path=str(array.filename)\n        array.flush()\n        # POSIX unlink releases the file after the last view is closed.\n        Path(path).unlink(missing_ok=True)\n        if path in _RSNA_CACHE_FILES: _RSNA_CACHE_FILES.remove(path)\n\ndef rsna_initialize(ids):\n    global _RSNA_TEST_IDS\n    _RSNA_TEST_IDS=[str(u) for u in ids]\n    if not _RSNA_TEST_IDS or len(set(_RSNA_TEST_IDS))!=len(_RSNA_TEST_IDS):\n        raise RuntimeError(\'empty or duplicate competition study IDs\')\n    work=Path(\'/kaggle/working\'); work.mkdir(parents=True,exist_ok=True)\n    os.chdir(work)\n    for f in [\'submission.csv\',\'_pipeline_stage.csv\',\'btkd_v559_complete.json\',\'btkd_v559_complete.json\']:\n        (work/f).unlink(missing_ok=True)\n    (work/\'diagnostics\').mkdir(exist_ok=True)\n    _RSNA_AUDIT[\'cohort\']=len(_RSNA_TEST_IDS)\n    _RSNA_AUDIT[\'environment\']={\'python\':__import__(\'sys\').version,\'torch\':str(torch.__version__),\'cuda\':torch.version.cuda,\'gpu_names\':[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}\n    rsna_json(work/\'diagnostics/runtime_started.json\',_RSNA_AUDIT)\n\ndef rsna_asset_preflight():\n    """Locate required additions before spending time on the public parent."""\n    names=[\'raptor_ft_coatnet_v5_full_swa.pt\',\'raptor_ft_coatnet_v10_full.pt\',\'raptor_ft_coatnet_v8_full_swa.pt\']\n    receipt={name:str(_asset_find_asset(name)) for name in names}\n    for name,digest in [\n        (\'coat_resgated_ep10_top3_manifest.json\',\'98511a8fdeb9da0e6e70c78d013ff636e1476f31c80b5dc134d294b18c3f284e\'),\n        (\'coatnet_pairfilm_manifest.json\',\'7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d\'),\n        (\'opencv_python_headless-4.12.0.88-*.whl\',\'236c8df54a90f4d02076e6f9c1cc763d794542e886c576a6fee46ec8ff75a7a9\')]:\n        receipt[name]={\'path\':str(_asset_find_asset(name,digest)),\'sha256\':digest}\n    roots=[Path(p) for p in [\'/kaggle/input/datasets/mattiaangeli/knee-mri-fold-weights\',\'/kaggle/input/knee-mri-fold-weights\'] if (Path(p)/\'m_f0.pt\').is_file()]\n    roots=list({str(p.resolve()):p for p in roots}.values())\n    if len(roots)!=1:raise RuntimeError(f\'A5: expected one complete attached root, found {roots}\')\n    missing=[str(roots[0]/f\'m_f{i}.pt\') for i in range(5) if not (roots[0]/f\'m_f{i}.pt\').is_file()]\n    if missing:raise FileNotFoundError(f\'A5 missing folds: {missing}\')\n    receipt[\'A5_root\']=str(roots[0]);receipt[\'tensor_loading_and_fingerprints\']=\'validated by mandatory component loaders before predictions\'\n    d4_manifest_path = _asset_find_asset(\'coatnet_pairfilm_manifest.json\',\n        \'7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d\')\n    timm_wheel = d4_manifest_path.parent / \'timm-1.0.22-py3-none-any.whl\'\n    if not timm_wheel.is_file() or rsna_sha(timm_wheel) != \'888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0\':\n        raise RuntimeError(\'D4 pinned timm wheel missing or changed\')\n    receipt[\'D4_timm_wheel_sha256\'] = rsna_sha(timm_wheel)\n    rsna_json(\'/kaggle/working/diagnostics/asset_preflight.json\',receipt)\n\n\ndef rsna_phase(stage, status=\'START\', **extra):\n    """Local phase evidence; this does not expose Kaggle\'s hidden rerun logs."""\n    import json, os, resource, time\n    from pathlib import Path\n    work = Path(\'/kaggle/working/diagnostics\')\n    work.mkdir(parents=True, exist_ok=True)\n    current = {\'stage\': stage, \'status\': status,\n               \'elapsed_seconds\': time.time()-T0,\n               \'max_rss_kib\': int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss), **extra}\n    for path in [\'/sys/fs/cgroup/memory.current\', \'/sys/fs/cgroup/memory.max\']:\n        try: current[path.rsplit(\'/\', 1)[-1]] = Path(path).read_text().strip()\n        except OSError: pass\n    if torch.cuda.is_initialized():\n        current[\'gpus\'] = [{\'id\': i, \'allocated_bytes\': torch.cuda.memory_allocated(i),\n                            \'reserved_bytes\': torch.cuda.memory_reserved(i)}\n                           for i in range(torch.cuda.device_count())]\n    print(\'[phase] \'+json.dumps(current, sort_keys=True), flush=True)\n    rsna_json(work/\'current_phase.json\', current)\n    with (work/\'phase_events.jsonl\').open(\'a\') as f:\n        f.write(json.dumps(current, sort_keys=True)+\'\\n\')\n    return current\n\n# Record uncaught cell errors locally; Kaggle may still withhold hidden-run logs.\ndef _rsna_capture_cell_error(result):\n    error = getattr(result, \'error_in_exec\', None) or getattr(result, \'error_before_exec\', None)\n    if error is None:\n        return\n    try:\n        path = Path(\'/kaggle/working/diagnostics/current_phase.json\')\n        previous = json.loads(path.read_text()) if path.is_file() else {}\n        rsna_phase(previous.get(\'stage\', \'unknown\'), \'FAILED\', error_type=type(error).__name__, error=str(error))\n    except Exception as logging_error:\n        print(\'[diagnostic-write-failed]\', type(logging_error).__name__, flush=True)\ntry:\n    _rsna_ip = get_ipython()\n    if _rsna_ip is not None:\n        _rsna_ip.events.register(\'post_run_cell\', _rsna_capture_cell_error)\nexcept NameError:\n    pass\n', globals())


## Stage 1 — DINO ensemble

Each MRI study is mapped into anatomical slots. Twenty member-specific tails
share the verified frozen DINO prefix, preserving all members while avoiding
the largest repeated transformer computation.


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(6, "def log(msg):\n    print(f'[{time.time() - T0:7.1f}s] {msg}', flush=True)\n\ndef _one_direct(tag, candidates, required):\n    matches = []\n    for candidate in map(Path, candidates):\n        if candidate.is_dir() and all((candidate / item).exists() for item in required):\n            matches.append(candidate)\n    unique = []\n    for match in matches:\n        if str(match.resolve()) not in {str(item.resolve()) for item in unique}:\n            unique.append(match)\n    if len(unique) != 1:\n        raise FileNotFoundError(\n            f'{tag}: expected exactly one direct mounted artifact, found '\n            f'{[str(item) for item in unique]}; checked {[str(Path(item)) for item in candidates]}'\n        )\n    return unique[0]\n\ndef find_root():\n    return _one_direct('competition', [\n        os.environ.get('RSNA_COMP_ROOT', '/kaggle/input/competitions/rsna-knee-abnormality-detection'),\n        '/kaggle/input/rsna-knee-abnormality-detection',\n    ], ['test.csv', 'test_series.csv', 'test_series'])\n\ndef find_dinov2(variant='small'):\n    if variant != 'small':\n        raise ValueError(f'unpinned DINOv2 variant: {variant}')\n    return _one_direct('DINOv2-S model', [\n        '/kaggle/input/models/metaresearch/dinov2/PyTorch/small/1',\n        '/kaggle/input/models/metaresearch/dinov2/pytorch/small/1',\n        '/kaggle/input/dinov2/PyTorch/small/1',\n        '/kaggle/input/dinov2/pytorch/small/1',\n    ], ['config.json', 'pytorch_model.bin'])\nROOT = find_root()\nDINOV2_SOURCE = find_dinov2('small')\nlog(f'input root: {ROOT}')\n_STARTUP_PATH_S = _startup_time.perf_counter() - _STARTUP_T0 - _STARTUP_GPU_IMPORT_S\nprint(f'[startup] competition-root preflight: {_STARTUP_PATH_S:.2f}s', flush=True)\nIMG = CACHE_IMG\n\ndef available_gb():\n    try:\n        with open('/proc/meminfo') as fh:\n            info = {k.strip(): v for k, v in (l.split(':', 1) for l in fh if ':' in l)}\n        return int(info['MemAvailable'].split()[0]) / 1024 ** 2\n    except Exception:\n        return CACHE_BUDGET_GB / CACHE_FRACTION\n\ndef plan_cache(n_study, n_test=0):\n    avail = available_gb()\n    budget = min(avail * CACHE_FRACTION, CACHE_BUDGET_MAX_GB)\n    n_total = n_study + max(n_test, int(TEST_SHARE * n_study))\n    per_slice = n_total * N_SLOT * IMG * IMG\n    afford = int(budget * 1024 ** 3 // max(per_slice, 1))\n    groups = max(1, min(N_GROUP_MAX, afford // GROUP))\n    log(f'memory: {avail:.1f} GB available, {budget:.1f} GB to the cache; sizing for {n_study} train + {n_total - n_study} test studies -> {groups} group(s) of {GROUP} = {groups * GROUP} slices per slot' + (f' (wanted {N_GROUP_MAX})' if groups < N_GROUP_MAX else ''))\n    return groups\nN_GROUP = plan_cache(len(pd.read_csv(ROOT / 'train.csv')), len(pd.read_csv(ROOT / 'test.csv')))\nCACHE_SLICES = GROUP * N_GROUP\nlog(f'cache layout: {N_GROUP} groups x {GROUP} slices = {CACHE_SLICES} per slot')\nrsna_initialize(pd.read_csv(ROOT/'test.csv',dtype={'StudyInstanceUID':str}).StudyInstanceUID.tolist())\nos.environ['RSNA_COMP_ROOT']=str(ROOT)\nrsna_asset_preflight()\n\n# Resolve unsupported A5 metadata before long model inference, without inventing labels.\n_meta = pd.read_csv(ROOT/'test_series.csv', dtype={'StudyInstanceUID':str})\n_required = {'StudyInstanceUID','Anatomical_Plane','Fat_Suppression'}\nif not _required.issubset(_meta.columns):\n    raise RuntimeError('Missing required series metadata columns: '+str(sorted(_required-set(_meta.columns))))\n_a5_matches = _meta.Anatomical_Plane.isin(['Sagittal','Coronal','Axial']) & _meta.Fat_Suppression.isin([0,1])\n_a5_ids = set(_meta.loc[_a5_matches,'StudyInstanceUID'])\n_a5_unavailable = [u for u in _RSNA_TEST_IDS if u not in _a5_ids]\nrsna_json('/kaggle/working/diagnostics/a5_metadata_applicability.json', {\n    'studies':len(_RSNA_TEST_IDS), 'no_matching_acquisition_uids':_a5_unavailable,\n    'policy':'base-only contribution for explicit A5 absence; model failures remain fatal'})\nrsna_phase('input_metadata', 'COMPLETE', a5_without_matching_acquisition=len(_a5_unavailable),\n           missing_fluid_flags=int(_meta.Fluid_Sensitive.isna().sum()) if 'Fluid_Sensitive' in _meta else len(_meta))\ndel _meta, _a5_matches, _a5_ids, _a5_unavailable\n", globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(7, "HDR_TAGS = ['SeriesDescription', 'SequenceName', 'ScanOptions', 'ScanningSequence', 'RepetitionTime', 'EchoTime', 'Laterality', 'PixelSpacing', 'Rows', 'Columns', 'RescaleSlope', 'RescaleIntercept', 'ImagePositionPatient', 'ImageOrientationPatient']\n\ndef _hdr_vec(s, n):\n    if not isinstance(s, str):\n        return None\n    try:\n        v = [float(x) for x in s.split('|')]\n    except ValueError:\n        return None\n    return np.array(v) if len(v) >= n else None\n\ndef side_from_geometry(h):\n    cx = {}\n    for r in h.itertuples(index=False):\n        ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)\n        iop = _hdr_vec(getattr(r, 'ImageOrientationPatient', None), 6)\n        ps = _hdr_vec(getattr(r, 'PixelSpacing', None), 2)\n        rows, cols = (getattr(r, 'Rows', None), getattr(r, 'Columns', None))\n        if ipp is None or iop is None or ps is None or (not rows) or (not cols):\n            continue\n        try:\n            c = ipp[:3] + iop[:3] * ps[1] * float(cols) / 2 + iop[3:6] * ps[0] * float(rows) / 2\n        except (TypeError, ValueError):\n            continue\n        cx.setdefault(r.StudyInstanceUID, []).append(float(c[0]))\n    out = {}\n    for st, xs in cx.items():\n        m = float(np.median(xs))\n        out[st] = None if abs(m) < LAT_MIN_OFFSET_MM else 'R' if m < 0 else 'L'\n    return out\n\ndef side_from_corner_x(h):\n    out = {}\n    for st, g in h.groupby('StudyInstanceUID'):\n        xs = []\n        for r in g.itertuples(index=False):\n            ipp = _hdr_vec(getattr(r, 'ImagePositionPatient', None), 3)\n            if ipp is not None and np.isfinite(ipp).all():\n                xs.append(float(ipp[0]))\n        if not xs:\n            out[st] = None\n            continue\n        x = float(np.median(xs))\n        out[st] = None if abs(x) < LEGACY_LAT_OFFSET_MM else 'R' if x < 0 else 'L'\n    return out\n\ndef lat_of(h, tag=''):\n    geo = side_from_corner_x(h) if RULES['lat'] == 'corner_x' else side_from_geometry(h)\n    d, n_tag, n_geo, n_none, n_disagree = ({}, 0, 0, 0, 0)\n    for st, g in h.groupby('StudyInstanceUID'):\n        v = [str(x).strip().upper() for x in g['Laterality'].dropna()]\n        if RULES['lat'] == 'corner_x' and 'ImageLaterality' in g.columns:\n            v += [str(x).strip().upper() for x in g['ImageLaterality'].dropna()]\n        v = [x[0] for x in v if x and x[0] in ('L', 'R')]\n        side = v[0] if v else None\n        if side is not None:\n            n_tag += 1\n            if geo.get(st) is not None and geo[st] != side:\n                n_disagree += 1\n        else:\n            side = geo.get(st)\n            n_geo += side is not None\n            n_none += side is None\n        d[st] = side\n    log(f'{tag}laterality: {n_tag} from the tag, {n_geo} from geometry, {n_none} unresolved; tag and geometry disagree on {n_disagree} ({n_disagree / max(n_tag, 1):.1%} of the tagged)')\n    return d\n\ndef probe(item):\n    split, study, series, path = item\n    row = {'split': split, 'StudyInstanceUID': study, 'SeriesInstanceUID': series, 'dir': path}\n    try:\n        files = sorted((e.name for e in os.scandir(path) if e.name.endswith('.dcm')))\n        row['files'] = files\n        row['n_slices'] = len(files)\n        if not files:\n            return row\n        ds = pydicom.dcmread(os.path.join(path, files[len(files) // 2]), stop_before_pixels=True, force=True)\n        for t in HDR_TAGS:\n            v = getattr(ds, t, None)\n            if v is None:\n                row[t] = None\n            elif isinstance(v, (list, tuple)) or type(v).__name__ == 'MultiValue':\n                row[t] = '|'.join((str(x) for x in v))\n            else:\n                row[t] = str(v)\n    except Exception as exc:\n        row['err'] = str(exc)[:120]\n    return row\n\ndef walk(split):\n    key=(str(ROOT.resolve()),str(split))\n    if key in _RSNA_HEADERS_MEMO:\n        return _RSNA_HEADERS_MEMO[key].copy(deep=True)\n    base = ROOT / split\n    items = []\n    if not base.is_dir():\n        return pd.DataFrame(columns=['split', 'StudyInstanceUID', 'SeriesInstanceUID', 'dir', 'files', 'n_slices'] + HDR_TAGS)\n    for study in os.scandir(base):\n        if study.is_dir():\n            for series in os.scandir(study.path):\n                if series.is_dir():\n                    items.append((split, study.name, series.name, series.path))\n    with ThreadPoolExecutor(max_workers=HDR_THREADS) as pool:\n        rows = list(pool.map(probe, items))\n    result=pd.DataFrame(rows)\n    _RSNA_HEADERS_MEMO[key]=result.copy(deep=True)\n    return result\n\ndef annotate(df):\n    desc = df['SeriesDescription'].fillna('') + ' ' + df['SequenceName'].fillna('')\n    desc = desc.str.lower().str.replace(_SEP, ' ', regex=True)\n    opts = df['ScanOptions'].fillna('').str.upper().str.split('|')\n    opts_fs = opts.apply(lambda ts: any((t.strip() in FATSAT_OPTS for t in ts)))\n    df['fatsat'] = desc.str.contains(_FATSAT_RX) | opts_fs\n    tr = pd.to_numeric(df['RepetitionTime'], errors='coerce')\n    te = pd.to_numeric(df['EchoTime'], errors='coerce')\n    gre = df['ScanningSequence'].fillna('').str.upper().str.contains('GR')\n    t1, t2, pdw = (desc.str.contains(_T1_RX), desc.str.contains(_T2_RX), desc.str.contains(_PD_RX))\n    df['weight'] = np.where(t1 & ~t2 & ~pdw, 'T1', np.where(t2 & ~pdw, 'T2', np.where(pdw, 'PD', np.where(gre, 'GRE', np.where(tr < 800, 'T1', np.where(te > 60, 'T2', np.where(tr >= 800, 'PD', 'UNK')))))))\n    df['fluid'] = np.isin(df['weight'], ['PD', 'T2'])\n    df['px'] = pd.to_numeric(df['PixelSpacing'].fillna('').str.split('|').str[0].replace('', np.nan), errors='coerce')\n    return df\n", globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(8, "def pick_slots(series_df, plane_map):\n    series_df = series_df.copy()\n    series_df['plane'] = series_df['SeriesInstanceUID'].map(plane_map)\n    out = {}\n    for study, g in series_df.groupby('StudyInstanceUID'):\n        chosen = {}\n        for name, plane, fluid, fs in SLOTS:\n            sel = (g['plane'] == plane) & (g['fatsat'] == fs)\n            if fluid is not None:\n                sel &= g['fluid'] == fluid\n            cand = g[sel]\n            if len(cand) == 0 and RULES['slot_fallback'] and (fluid is False):\n                cand = g[(g['plane'] == plane) & ~g['fatsat']]\n            if len(cand):\n                chosen[name] = cand.sort_values('n_slices', ascending=False).iloc[0]\n        out[study] = chosen\n    return out\n", globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(9, 'ORDER_TAGS = [(32, 50), (32, 55), (32, 19)]\nDECODE_FAILED = []\nRAPTOR_HEADER_CACHE = {}\n\n\ndef _natural_key(name):\n    return tuple((int(x) if x.isdigit() else x.lower() for x in re.split(\'(\\\\d+)\', str(name))))\n\ndef _order_dominant_axis(rec):\n    files, d = (rec[\'files\'], rec[\'dir\'])\n    rows = []\n    for pos, f in enumerate(files):\n        ipp = inst = None\n        try:\n            ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=[\'ImagePositionPatient\', \'InstanceNumber\'])\n            raw = getattr(ds, \'ImagePositionPatient\', None)\n            if raw is not None and len(raw) >= 3:\n                c = np.asarray(raw[:3], dtype=np.float64)\n                if np.isfinite(c).all():\n                    ipp = c\n            n = getattr(ds, \'InstanceNumber\', None)\n            if n is not None:\n                inst = float(n)\n        except Exception:\n            pass\n        rows.append((f, ipp, inst, pos))\n    placed = [r for r in rows if r[1] is not None]\n    need = max(2, int(0.8 * len(rows)))\n    if len(placed) >= need:\n        xyz = np.stack([r[1] for r in placed])\n        axis = int(np.argmax(np.ptp(xyz, axis=0)))\n        spare = float(np.nanmedian(xyz[:, axis]))\n        rows.sort(key=lambda r: (float(r[1][axis]) if r[1] is not None else spare, r[2] if r[2] is not None else float(\'inf\'), r[3]))\n    elif sum((r[2] is not None for r in rows)) >= need:\n        rows.sort(key=lambda r: (r[2] if r[2] is not None else float(\'inf\'), r[3]))\n    else:\n        rows.sort(key=lambda r: _natural_key(r[0]))\n    return ([r[0] for r in rows], True)\n\ndef order_slices(rec):\n    if RULES[\'order\'] == \'dominant_axis\':\n        return _order_dominant_axis(rec)\n    files, d = (rec[\'files\'], rec[\'dir\'])\n    RAPTOR_HEADER_CACHE.pop(str(d), None)\n    keyed = []\n    raptor_records = []\n    raptor_safe = bool(files)\n    for f in files:\n        k = None\n        ds = None\n        file_raptor_safe = False\n        try:\n            try:\n                ds = pydicom.dcmread(os.path.join(d, f), stop_before_pixels=True)\n                file_raptor_safe = True\n            except Exception:\n                ds = pydicom.dcmread(os.path.join(d, f), force=True, stop_before_pixels=True, specific_tags=ORDER_TAGS)\n            iop = np.asarray(ds.ImageOrientationPatient, dtype=float)\n            ipp = np.asarray(ds.ImagePositionPatient, dtype=float)\n            k = float(np.dot(ipp, np.cross(iop[:3], iop[3:])))\n        except Exception:\n            file_raptor_safe = False\n            try:\n                k = float(ds.InstanceNumber)\n            except Exception:\n                k = None\n        if file_raptor_safe:\n            try:\n                file_raptor_safe = (\n                    iop.shape == (6,) and ipp.shape == (3,)\n                    and np.isfinite(iop).all() and np.isfinite(ipp).all()\n                    and np.isfinite(k)\n                )\n                spacing = getattr(ds, \'PixelSpacing\', None)\n                spacing = float(spacing[0]) if spacing is not None else 0.5\n                file_raptor_safe = file_raptor_safe and np.isfinite(spacing) and spacing > 0\n                if file_raptor_safe:\n                    raptor_records.append((k, os.path.join(d, f), spacing))\n            except Exception:\n                file_raptor_safe = False\n        raptor_safe = raptor_safe and file_raptor_safe\n        keyed.append((k, f))\n    if any((k is None for k, _ in keyed)):\n        return (sorted(files,key=_natural_key), False)\n    if raptor_safe and len(raptor_records) == len(files) and len({item[0] for item in raptor_records}) == len(files):\n        raptor_records.sort(key=lambda item: (item[0], item[1]))\n        RAPTOR_HEADER_CACHE[str(d)] = (\n            [(item[1], item[2]) for item in raptor_records],\n            float(np.median([item[2] for item in raptor_records])),\n        )\n    return ([f for _, f in sorted(keyed, key=lambda t: t[0])], True)\n\ndef read_slot(rec, n_slice=None, out_size=None):\n    """Same healthy pixels as BTKD; failed slices cannot erase good native planes."""\n    n_slice = GROUP if n_slice is None else int(n_slice)\n    out_size = IMG if out_size is None else int(out_size)\n    files, d, px = rec.get(\'ordered\') or rec[\'files\'], rec[\'dir\'], rec[\'px\']\n    if not files:\n        rsna_event(\'slot_no_files\', series=str(rec.get(\'SeriesInstanceUID\', d)), dir=str(d))\n        return None\n    lo, hi = int(SLICE_BAND[0]*(len(files)-1)), int(SLICE_BAND[1]*(len(files)-1))\n    idx = np.unique(np.linspace(lo,hi,n_slice).astype(int)) if hi>lo else np.array([len(files)//2])\n    while len(idx)<n_slice: idx=np.append(idx,idx[-1])\n    decoded, errors = {}, []\n    for i in sorted(set(int(v) for v in idx[:n_slice])):\n        path=os.path.join(d,files[i])\n        try:\n            ds=pydicom.dcmread(path,force=True)\n            a=ds.pixel_array.astype(np.float32)\n            a=a*float(getattr(ds,\'RescaleSlope\',1) or 1)+float(getattr(ds,\'RescaleIntercept\',0) or 0)\n            if a.ndim!=2 or not np.isfinite(a).all(): raise ValueError(\'expected finite 2-D pixels\')\n            decoded[i]=a\n        except Exception as exc:\n            decoded[i]=None; errors.append({\'path\':path,\'error\':f\'{type(exc).__name__}: {exc}\'})\n    planes=[decoded[int(i)] for i in idx[:n_slice]]\n    got=[i for i,p in enumerate(planes) if p is not None]\n    if errors:\n        DECODE_FAILED.append(rec.get(\'SeriesInstanceUID\',d))\n        rsna_event(\'partial_slice_decode\',series=str(rec.get(\'SeriesInstanceUID\',d)),errors=errors)\n    if not got:\n        rsna_event(\'slot_all_decode_failed\', series=str(rec.get(\'SeriesInstanceUID\', d)), dir=str(d), errors=errors[:5])\n        return None\n    from collections import Counter\n    shape=Counter(planes[i].shape for i in got).most_common(1)[0][0]\n    odd=[i for i in got if planes[i].shape!=shape]\n    if odd:\n        rsna_event(\'slot_shape_mismatch\', series=str(rec.get(\'SeriesInstanceUID\', d)), dir=str(d), majority=[int(v) for v in shape], dropped=len(odd))\n        for i in odd: planes[i]=None\n        got=[i for i in got if planes[i] is not None]\n    for i,p in enumerate(planes):\n        if p is None:\n            planes[i]=np.zeros(shape,np.float32) if RULES[\'decode_fill\']==\'zero\' else planes[min(got,key=lambda j:abs(j-i))]\n    vol=np.stack(planes)\n    if px and np.isfinite(px) and px>0:\n        want=int(round(CROP_MM/px)); h,w=shape\n        if 16<want<min(h,w):\n            cy,cx=h//2,w//2; half=want//2\n            vol=vol[:,max(0,cy-half):cy+half,max(0,cx-half):cx+half]\n    lo_v,hi_v=np.percentile(vol,[1,99])\n    vol=np.clip((vol-lo_v)/max(hi_v-lo_v,1e-6),0,1)\n    t=torch.from_numpy(np.ascontiguousarray(vol)).unsqueeze(0)\n    t=F.interpolate(t,size=(out_size,out_size),mode=\'bilinear\',align_corners=False)\n    return (t.squeeze(0)*255).round().clamp(0,255).to(torch.uint8)\n', globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(10, "def normalise_laterality(img, plane, lat):\n    if lat != 'R':\n        return img\n    if plane in ('Coronal', 'Axial'):\n        return torch.flip(img, dims=[-1])\n    return torch.flip(img, dims=[0])\n", globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(11, 'def build_cache(slot_map, plane_map, lat_map, tag):\n    studies = list(_RSNA_TEST_IDS) if _RSNA_TEST_IDS is not None else sorted(slot_map)\n    if set(studies)!=set(slot_map):\n        rsna_event(\'coverage_mismatch\',tag=tag,missing=[str(s) for s in studies if s not in slot_map][:20],extra=len(set(slot_map)-set(studies)))\n        slot_map={s:slot_map.get(s,{}) for s in studies}\n    sidx={s:i for i,s in enumerate(studies)}\n    cache=rsna_array((len(studies),N_SLOT,CACHE_SLICES,IMG,IMG),tag)\n    mask=np.zeros((len(studies),N_SLOT),np.float32)\n    jobs=[(st,k,plane,slot_map[st][name]) for st in studies for k,(name,plane,_,_) in enumerate(SLOTS) if name in slot_map[st]]\n    log(f\'{tag}: {len(studies)} studies, {len(jobs)} slot-series, {cache.nbytes/2**30:.2f} GiB pixels\')\n    missing=[]; reused=0\n    for job in jobs:\n        rec=job[3]\n        key=(str(Path(rec[\'dir\']).resolve()),RULES[\'order\'],tuple(sorted(rec[\'files\'])))\n        saved=_RSNA_ORDER_MEMO.get(key)\n        if saved is not None:\n            rec[\'ordered\']=list(saved[0]);reused+=1\n        else: missing.append((job,key))\n    def ordered(item):\n        job,key=item\n        try:\n            files,good=order_slices(job[3])\n            if len(files)!=len(job[3][\'files\']) or set(files)!=set(job[3][\'files\']):\n                raise ValueError(f\'ordering lost/added files ({len(files)} vs {len(job[3]["files"])})\')\n        except Exception as exc:\n            rsna_event(\'ordering_fallback\',tag=tag,series=str(job[3].get(\'SeriesInstanceUID\')),error=f\'{type(exc).__name__}: {exc}\')\n            return key,sorted(job[3][\'files\']),False\n        return key,files,bool(good)\n    with ThreadPoolExecutor(max_workers=ORDER_THREADS) as pool:\n        for start in range(0,len(missing),256):\n            rsna_deadline(tag+\' ordering\')\n            block=missing[start:start+256]\n            for (job,_),(key,files,good) in zip(block,pool.map(ordered,block)):\n                job[3][\'ordered\']=files;_RSNA_ORDER_MEMO[key]=(tuple(files),good)\n                if not good: rsna_event(\'deterministic_order_fallback\',series=str(job[3].get(\'SeriesInstanceUID\')))\n    done=0; unfilled=0\n    def _tolerant_read_slot(j):\n        try: return read_slot(j[3],CACHE_SLICES,IMG)\n        except Exception as exc:\n            rsna_event(\'slot_read_error\',tag=tag,study=str(j[0]),series=str(j[3].get(\'SeriesInstanceUID\')),error=f\'{type(exc).__name__}: {exc}\'); return None\n    with ThreadPoolExecutor(max_workers=PIX_THREADS) as pool:\n        for start in range(0,len(jobs),128):\n            rsna_deadline(tag+\' decoding\')\n            block=jobs[start:start+128]\n            for (st,k,plane,rec),img in zip(block,pool.map(_tolerant_read_slot,block)):\n                done+=1\n                if img is None:\n                    unfilled+=1; rsna_event(\'slot_unfilled\',tag=tag,study=str(st),series=str(rec.get(\'SeriesInstanceUID\')),slot=int(k)); continue\n                cache[sidx[st],k]=normalise_laterality(img,plane,lat_map.get(st)).numpy()\n                mask[sidx[st],k]=1\n    if unfilled: log(f\'{tag}: {unfilled} slot(s) left empty and masked (flagged)\')\n    _empty=[studies[i] for i in np.flatnonzero(mask.sum(1)==0)]\n    if _empty:\n        rsna_event(\'empty_study_rows\',tag=tag,count=len(_empty),studies=[str(s) for s in _empty])\n        log(f\'{tag}: {len(_empty)} study(ies) with no series matching this layout kept as all-zero masked rows (parent policy): {_empty[:5]}\')\n    if done!=len(jobs):\n        raise RuntimeError(f\'{tag}: incomplete cache; {done}/{len(jobs)} slots\')\n    if isinstance(cache,np.memmap): cache.flush()\n    rsna_event(\'cache_complete\',tag=tag,studies=len(studies),slot_jobs=len(jobs),filled=done,order_cache_hits=reused,bytes=int(cache.nbytes))\n    log(f\'{tag}: COMPLETE {done}/{len(jobs)}; reused ordering for {reused} series\')\n    return studies,cache,mask\n', globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(12, "class SlotHead(nn.Module):\n\n    def __init__(self, dim, n_slot, n_out, hidden=256, p=0.2, prior=False):\n        super().__init__()\n        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())\n        self.slot_emb = nn.Parameter(torch.randn(n_slot, hidden) * 0.02)\n        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)\n        self.drop = nn.Dropout(p)\n        self.out = nn.Linear(hidden, n_out)\n        self.hidden = hidden\n        p_ = torch.zeros(n_out, n_slot)\n        if prior and n_slot == len(SLOTS) and (n_out == len(TARGETS)):\n            for t, slots in SLOT_PRIOR_TABLE.items():\n                if t in TARGETS:\n                    p_[TARGETS.index(t), list(slots)] = SLOT_PRIOR_STRENGTH\n        self.prior = prior\n        if prior:\n            self.register_buffer('slot_prior', p_)\n\n    def forward(self, x, mask):\n        h = self.proj(x) + self.slot_emb\n        att = torch.einsum('bsh,oh->bos', h, self.query) / self.hidden ** 0.5\n        if self.prior:\n            att = att + self.slot_prior.unsqueeze(0)\n        att = att.masked_fill(mask.unsqueeze(1) < 0.5, -10000.0).softmax(-1)\n        ctx = self.drop(torch.einsum('bos,bsh->boh', att, h))\n        return (ctx * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias\n", globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(13, "class Model(nn.Module):\n\n    def __init__(self, backbone, dim, pool='cls_mean', prior=False):\n        super().__init__()\n        self.backbone = backbone\n        self.pool = pool\n        self.head = SlotHead(dim * POOL_PARTS[pool], N_SLOT, len(TARGETS), prior=prior)\n        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))\n        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))\n\n    def forward(self, imgs, mask, img_size=None):\n        B, S = imgs.shape[:2]\n        x = imgs.reshape(B * S, *imgs.shape[2:]).float().div_(255.0)\n        if img_size is not None and img_size != x.shape[-1]:\n            x = F.interpolate(x, size=(img_size, img_size), mode='bilinear', align_corners=False)\n        x = (x - self.mean) / self.std\n        out = self.backbone(pixel_values=x).last_hidden_state\n        patch = out[:, 1:]\n        parts = [out[:, 0], patch.mean(1)]\n        if self.pool == 'cls_mean_focal':\n            k = max(1, patch.shape[1] // 8)\n            parts.append(patch.topk(k, dim=1).values.mean(1))\n        feat = torch.cat(parts, dim=1).reshape(B, S, -1)\n        return self.head(feat, mask)\n", globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(14, 'def build_model(unfreeze_last, source=None, variant=\'small\', pool=\'cls_mean\', prior=False):\n    from transformers import AutoModel\n    p = source if source is not None else DINOV2_SOURCE\n    if p is None:\n        raise FileNotFoundError(\'DINOv2 weights not attached\')\n    bb = AutoModel.from_pretrained(str(p))\n    n_layer = len(bb.encoder.layer)\n    for prm in bb.parameters():\n        prm.requires_grad = False\n    for blk in bb.encoder.layer[max(0, n_layer - unfreeze_last):]:\n        for prm in blk.parameters():\n            prm.requires_grad = True\n    for prm in bb.layernorm.parameters():\n        prm.requires_grad = True\n    dim = bb.config.hidden_size\n    trainable = sum((p.numel() for p in bb.parameters() if p.requires_grad))\n    log(f\'backbone: {n_layer} blocks, last {unfreeze_last} trainable ({trainable / 1000000.0:.1f}M params), feature dim {dim * POOL_PARTS[pool]}\')\n    return Model(bb, dim, pool=pool, prior=prior)\n_speed_original_build_model = build_model\n"""Reuse a CPU architecture template; every member still strict-loads and fingerprints."""\nimport copy as _speed_copy\nimport threading as _speed_dino_threading\n\n_speed_dino_templates = {}\n_speed_dino_lock = _speed_dino_threading.RLock()\n\n\ndef build_model(unfreeze_last, source=None, variant=\'small\', pool=\'cls_mean\', prior=False):\n    key = (int(unfreeze_last), None if source is None else str(source), variant,\n           pool, bool(prior), N_SLOT, tuple(TARGETS))\n    with _speed_dino_lock:\n        if key not in _speed_dino_templates:\n            template = _speed_original_build_model(unfreeze_last, source, variant, pool, prior)\n            assert all(p.device.type == \'cpu\' for p in template.parameters())\n            _speed_dino_templates[key] = template\n        # No mutable state or GPU tensors shared between independently trained models.\n        return _speed_copy.deepcopy(_speed_dino_templates[key])\n', globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(15, 'SHARED_DINO_PREFIX_LAYERS = 6\nSHARED_DINO_PREFIX_SHA256 = \'0a55b893bde971c864ea5aeea443f075b17c084d13aa36f30bd1c7863f655cf9\'\ndef _shared_prefix_state_sha256(state):\n    digest = hashlib.sha256()\n    prefixes = (\n        \'backbone.embeddings.\',\n        *(\n            f\'backbone.encoder.layer.{index}.\'\n            for index in range(\n                SHARED_DINO_PREFIX_LAYERS\n            )\n        ),\n    )\n    keys = sorted(\n        key\n        for key in state\n        if key.startswith(prefixes)\n    )\n    if len(keys) != 113:\n        raise WeightsError(\n            \'shared DINOv2 prefix key-count drift: \'\n            f\'{len(keys)} != 113\'\n        )\n    for key in keys:\n        tensor = state[key].detach().cpu().contiguous()\n        digest.update(key.encode())\n        digest.update(str(tensor.dtype).encode())\n        digest.update(str(tuple(tensor.shape)).encode())\n        digest.update(tensor.numpy().tobytes())\n    return digest.hexdigest()\n\ndef _shared_prefix_eligible(members):\n    if not members or not DEVS:\n        return False\n    return all(\n        \'state\' not in member\n        and int(member[\'config\'][\'unfreeze_last\'])\n        == SHARED_DINO_PREFIX_LAYERS\n        and member[\'config\'][\'variant\'] == \'small\'\n        and member[\'config\'].get(\'pool\', \'cls_mean\')\n        in POOL_PARTS\n        for member in members\n    )\n\ndef _shared_dino_prefix(model, images, img_size):\n    batch, slots = images.shape[:2]\n    values = images.reshape(\n        batch * slots,\n        *images.shape[2:],\n    ).float().div_(255.0)\n    if (\n        img_size is not None\n        and img_size != values.shape[-1]\n    ):\n        values = F.interpolate(\n            values,\n            size=(img_size, img_size),\n            mode=\'bilinear\',\n            align_corners=False,\n        )\n    values = (\n        values - model.mean\n    ) / model.std\n    hidden = model.backbone.embeddings(values)\n    for layer in model.backbone.encoder.layer[\n        :SHARED_DINO_PREFIX_LAYERS\n    ]:\n        hidden = layer(hidden)\n        hidden = hidden[0] if isinstance(hidden, (tuple, list)) else hidden\n    return hidden\n\ndef _shared_dino_tail(\n    model,\n    hidden,\n    batch,\n    slots,\n    slot_mask,\n):\n    value = hidden\n    for layer in model.backbone.encoder.layer[\n        SHARED_DINO_PREFIX_LAYERS:\n    ]:\n        value = layer(value)\n        value = value[0] if isinstance(value, (tuple, list)) else value\n    value = model.backbone.layernorm(value)\n    patch = value[:, 1:]\n    parts = [\n        value[:, 0],\n        patch.mean(1),\n    ]\n    if model.pool == \'cls_mean_focal\':\n        count = max(\n            1,\n            patch.shape[1] // 8,\n        )\n        parts.append(\n            patch.topk(\n                count,\n                dim=1,\n            ).values.mean(1)\n        )\n    feature = torch.cat(\n        parts,\n        dim=1,\n    ).reshape(\n        batch,\n        slots,\n        -1,\n    )\n    return model.head(feature, slot_mask)\n\n@torch.no_grad()\ndef _predict_shared_dino_members(\n    entries,\n    cache,\n    mask,\n    idx,\n    dev,\n    img_size,\n    group,\n    starts,\n):\n    target_idx = {\n        target: index\n        for index, target in enumerate(TARGETS)\n    }\n    states = {}\n    for member, model, jitter in entries:\n        generator = torch.Generator(device=dev)\n        jitter_seed = (\n            SEED\n            + int(\n                hashlib.sha256(\n                    str(member[\'id\']).encode()\n                ).hexdigest()[:8],\n                16,\n            )\n        )\n        generator.manual_seed(\n            int(jitter_seed) % (2 ** 63 - 1)\n        )\n        states[member[\'id\']] = {\n            \'out\': [],\n            \'public\': [],\n            \'soft\': [],\n            \'generator\': generator,\n            \'jitter\': jitter,\n        }\n        model.eval()\n\n    reference = entries[0][1]\n    for batch_start in range(\n        0,\n        len(idx),\n        EVAL_BATCH,\n    ):\n        selected = idx[\n            batch_start:\n            batch_start + EVAL_BATCH\n        ]\n        slot_mask = torch.from_numpy(\n            mask[selected]\n        ).to(dev)\n        batch = len(selected)\n        slots = cache.shape[1]\n        member_windows = {\n            member[\'id\']: (\n                [],\n                [],\n                [],\n            )\n            for member, _, _ in entries\n        }\n\n        for start in starts:\n            rows = torch.from_numpy(\n                np.ascontiguousarray(\n                    cache[\n                        selected,\n                        :,\n                        start:start + group,\n                    ]\n                )\n            ).to(dev)\n            def _dino_shared_forward(_amp):\n                with torch.autocast(\'cuda\', enabled=_amp):\n                    _common = _shared_dino_prefix(reference, rows, img_size)\n                    return {member[\'id\']: _shared_dino_tail(model, _common, batch, slots, slot_mask).float() for member, model, _ in entries}\n            original_logits = _dino_shared_forward(dev.type == \'cuda\')\n            if dev.type == \'cuda\' and not all(bool(torch.isfinite(v).all()) for v in original_logits.values()):\n                rsna_event(\'dino_fp16_nonfinite_retry_fp32\', window_start=int(start))\n                original_logits = _dino_shared_forward(False)\n\n            for member, model, jitter in entries:\n                member_id = member[\'id\']\n                view_logits = [\n                    original_logits[member_id]\n                ]\n                if jitter:\n                    jittered = augment(\n                        rows,\n                        generator=states[\n                            member_id\n                        ][\'generator\'],\n                    )\n                    with torch.autocast(\n                        \'cuda\',\n                        enabled=dev.type == \'cuda\',\n                    ):\n                        jitter_hidden = (\n                            _shared_dino_prefix(\n                                reference,\n                                jittered,\n                                img_size,\n                            )\n                        )\n                        view_logits.append(\n                            _shared_dino_tail(\n                                model,\n                                jitter_hidden,\n                                batch,\n                                slots,\n                                slot_mask,\n                            ).float()\n                        )\n                    if dev.type == \'cuda\' and not bool(torch.isfinite(view_logits[-1]).all()):\n                        rsna_event(\'dino_jitter_fp16_nonfinite_dropped\', member=str(member_id))\n                        view_logits.pop()\n                view_probs = [\n                    torch.sigmoid(value)\n                    for value in view_logits\n                ]\n                probabilities, logits, originals = (\n                    member_windows[member_id]\n                )\n                logits.append(\n                    torch.stack(\n                        view_logits\n                    ).mean(0)\n                )\n                probabilities.append(\n                    torch.stack(\n                        view_probs\n                    ).mean(0)\n                )\n                originals.append(\n                    view_probs[0]\n                )\n\n        for member, _, _ in entries:\n            member_id = member[\'id\']\n            win_probs, win_logits, win_originals = (\n                member_windows[member_id]\n            )\n            probabilities = torch.stack(win_probs)\n            logits = torch.stack(win_logits)\n            originals = torch.stack(win_originals)\n            value = (\n                torch.sigmoid(logits.mean(0))\n                if TTA_POOL == \'logit\'\n                else probabilities.mean(0)\n            )\n            value = apply_target_window_pool(\n                value,\n                probabilities,\n                logits,\n                originals,\n                TTA_TARGET_POOL,\n                target_idx,\n            )\n            states[member_id][\'out\'].append(\n                value.cpu().numpy()\n            )\n            public_value = apply_target_window_pool(\n                originals.mean(0),\n                originals,\n                logits,\n                originals,\n                PUBLIC_FRONTIER_TARGET_POOL,\n                target_idx,\n            )\n            states[member_id][\'public\'].append(\n                public_value.cpu().numpy()\n            )\n            states[member_id][\'soft\'].append(\n                legacy_fold_soft_window_pool(\n                    originals,\n                    target_idx,\n                ).cpu().numpy()\n            )\n\n    return {\n        member[\'id\']: (\n            np.concatenate(states[member[\'id\']][\'out\']),\n            np.concatenate(states[member[\'id\']][\'public\']),\n            np.concatenate(states[member[\'id\']][\'soft\']),\n        )\n        for member, _, _ in entries\n    }\n\n@torch.inference_mode()\ndef _check_shared_dino_path(model,dev,img_size,tag):\n    model.eval()\n    g=torch.Generator().manual_seed(SEED)\n    im=torch.randint(0,256,(2,N_SLOT,GROUP,img_size,img_size),generator=g,dtype=torch.uint8).to(dev)\n    mask=torch.ones(2,N_SLOT,device=dev);mask[1,-1]=0\n    direct=model(im,mask,img_size).float()\n    split=_shared_dino_tail(model,_shared_dino_prefix(model,im,img_size),2,N_SLOT,mask).float()\n    if not torch.isfinite(split).all() or not torch.allclose(split,direct,rtol=1e-5,atol=2e-5):\n        raise RuntimeError(f\'{tag}: shared-prefix execution differs from full forward\')\n    rsna_event(\'dino_shared_path_parity\',member=tag,max_abs=float((split-direct).abs().max()))\n\ndef _run_shared_dino_group(\n    path,\n    members,\n    cache,\n    mask,\n    idx,\n    starts,\n):\n    ordered = sorted(\n        members,\n        key=lambda member: -(\n            member.get(\'holdout\') or 0\n        ),\n    )\n    # run_dinov2 retains only submission_public_0899.csv, whose member\n    # predictions are built exclusively from the unaugmented views.\n    # Jittering here affected only the intermediate submission.csv that the\n    # retained public-frontier file replaces at the end of this stage.\n    plans = [\n        (\n            member,\n            False,\n        )\n        for member in ordered\n    ]\n    assignments = [\n        plans[index::len(DEVS)]\n        for index in range(len(DEVS))\n    ]\n    results = {}\n    result_lock = threading.Lock()\n\n    def worker(dev, plans_for_device):\n        global _DINOV2_MATCHED_MEMBERS\n        loaded = []\n        for member, jitter in plans_for_device:\n            with BUILD_LOCK:\n                checkpoint = torch.load(\n                    Path(path) / member[\'file\'],\n                    map_location=\'cpu\',\n                    weights_only=False,\n                )\n                state = checkpoint[\'model\']\n                prefix_sha256 = (\n                    _shared_prefix_state_sha256(\n                        state\n                    )\n                )\n                if (\n                    prefix_sha256\n                    != SHARED_DINO_PREFIX_SHA256\n                ):\n                    raise WeightsError(\n                        f"{member[\'id\']}: frozen prefix "\n                        \'hash mismatch: \'\n                        f\'{prefix_sha256}\'\n                    )\n                model = build_model(\n                    int(\n                        member[\'config\'][\n                            \'unfreeze_last\'\n                        ]\n                    ),\n                    variant=member[\'config\'][\n                        \'variant\'\n                    ],\n                    pool=member[\'config\'].get(\n                        \'pool\',\n                        \'cls_mean\',\n                    ),\n                    prior=bool(\n                        member[\'config\'].get(\n                            \'prior\',\n                            False,\n                        )\n                    ),\n                ).to(dev)\n                model.load_state_dict(state)\n                check_fingerprint(\n                    model,\n                    dev,\n                    IMG,\n                    checkpoint.get(\'fingerprint\'),\n                    tag=f"{member[\'id\']}: ",\n                )\n                _check_shared_dino_path(model, dev, IMG, member[\'id\'])\n                _DINOV2_MATCHED_MEMBERS += 1\n                del checkpoint, state\n            loaded.append(\n                (member, model, jitter)\n            )\n\n        predicted = _predict_shared_dino_members(\n            loaded,\n            cache,\n            mask,\n            idx,\n            dev,\n            IMG,\n            GROUP,\n            starts,\n        )\n        with result_lock:\n            results.update(predicted)\n        del loaded\n        gc.collect()\n        with torch.cuda.device(dev):\n            torch.cuda.empty_cache()\n\n    with ThreadPoolExecutor(max_workers=len(DEVS)) as pool:\n        futures=[pool.submit(worker,dev,plan) for dev,plan in zip(DEVS,assignments)]\n        for future in futures: future.result()\n    if len(results) != len(ordered):\n        raise WeightsError(\n            \'shared DINOv2 worker did not return \'\n            f\'all members: {len(results)} / \'\n            f\'{len(ordered)}\'\n        )\n    log(\n        \'shared first 6 DINOv2 blocks for \'\n        f\'{len(ordered)} members; original views \'\n        f\'computed once per {len(DEVS)} device(s)\'\n    )\n    return [\n        (\n            member,\n            results[member[\'id\']],\n            jitter,\n        )\n        for member, jitter in plans\n    ]\n\nFINGERPRINT_TOL = 0.002\nEXPECTED_DINOV2_MEMBERS = 20\n_DINOV2_MATCHED_MEMBERS = 0\n\ndef fingerprint(model, dev, img_size, n_slot=None, group=None, seed=None):\n    n_slot = N_SLOT if n_slot is None else n_slot\n    group = GROUP if group is None else group\n    seed = SEED if seed is None else seed\n    g = torch.Generator().manual_seed(seed)\n    imgs = torch.randint(0, 256, (2, n_slot, group, img_size, img_size), generator=g, dtype=torch.uint8).to(dev)\n    mask = torch.ones(2, n_slot, device=dev)\n    mask[1, -1] = 0.0\n    was_training = model.training\n    model.eval()\n    with torch.no_grad():\n        out = model(imgs, mask, img_size).float().cpu().numpy()\n    if was_training:\n        model.train()\n    return out\n\ndef check_fingerprint(model, dev, img_size, expected, tol=FINGERPRINT_TOL, tag=\'\'):\n    got = fingerprint(model, dev, img_size)\n    exp = np.asarray(expected, np.float32)\n    if not np.isfinite(got).all() or not np.isfinite(exp).all():\n        raise WeightsError(f\'{tag}: nonfinite fingerprint\')\n    if got.shape != exp.shape:\n        raise WeightsError(f\'{tag}fingerprint shape {got.shape} != stored {exp.shape}: the architecture is not the one these weights were fitted to\')\n    d = float(np.abs(got - exp).max())\n    if d > tol:\n        raise WeightsError(f\'{tag}fingerprint differs by {d:.4g} (tolerance {tol:g}). The weights load but do not compute what they computed when fitted - preprocessing, resolution or architecture has moved between the two runs.\')\n    log(f\'{tag}fingerprint matches within {d:.2g}\')\n    return d\n\nclass WeightsError(RuntimeError):\n    pass\n\ndef find_weights(name=\'manifest.json\'):\n    import json\n    roots = [\n        Path(\'/kaggle/input/datasets/pilkwang/rsna-knee-weights\'),\n        Path(\'/kaggle/input/rsna-knee-weights\'),\n    ]\n    valid = []\n    for root in roots:\n        path = root / name\n        if not path.is_file():\n            continue\n        try:\n            man = json.loads(path.read_text())\n        except (OSError, ValueError) as exc:\n            raise WeightsError(f\'invalid pinned weights manifest at {path}: {exc}\') from exc\n        if not isinstance(man.get(\'members\'), list) or len(man[\'members\']) != EXPECTED_DINOV2_MEMBERS:\n            raise WeightsError(f\'{path} must list exactly {EXPECTED_DINOV2_MEMBERS} members\')\n        if len({m.get(\'id\') for m in man[\'members\']}) != EXPECTED_DINOV2_MEMBERS:\n            raise WeightsError(f\'{path} has duplicate or missing member ids\')\n        missing = [m[\'file\'] for m in man[\'members\'] if not (root / m[\'file\']).is_file()]\n        if missing:\n            raise WeightsError(\n                f\'{root} lists {len(man["members"])} members but misses {missing[0]!r}\'\n            )\n        valid.append(root)\n    if len(valid) != 1:\n        raise WeightsError(f\'expected one pinned rsna-knee-weights root, found {valid}\')\n    return valid[0]\nTTA_OVERLAP = True\nTTA_POOL = \'prob\'\nPUBLIC_FRONTIER_TARGET_POOL = {\'Fracture\': \'max\', \'Contusion\': \'max\', \'Medial Meniscus\': \'max\', \'Lateral Meniscus\': \'max\', \'ACL\': \'top2\', \'MCL\': \'top2\', "Baker\'s": \'max\'}\nTTA_TARGET_POOL = {**PUBLIC_FRONTIER_TARGET_POOL, \'Synovitis\': \'original_mean\'}\n# No-extra-pass diversity branch: smooth focal pooling is evaluated from\n# the same no-jitter public-member windows already used by the parent.\nLEGACY_FOLD_SOFTPOOL_BETA = {\n    \'ACL\': 6.0, \'MCL\': 6.0,\n    \'Medial Meniscus\': 8.0, \'Lateral Meniscus\': 8.0,\n    "Baker\'s": 8.0, \'Contusion\': 8.0, \'Fracture\': 10.0,\n}\nLEGACY_FOLD_SOFTPOOL_ALPHA = {\n    \'ACL\': 0.20, \'MCL\': 0.20,\n    \'Medial Meniscus\': 0.25, \'Lateral Meniscus\': 0.25,\n    "Baker\'s": 0.20, \'Contusion\': 0.20, \'Fracture\': 0.15,\n}\nLEGACY_MEMBER_WEIGHT_BY_TARGET = {\'Lateral Meniscus\': 15.0, \'Medial OA\': 2.5, \'Lateral OA\': 15.0, \'Contusion\': 5.0}\n\ndef window_starts(n_slice, group, overlap=None):\n    overlap = TTA_OVERLAP if overlap is None else overlap\n    if overlap and n_slice >= group:\n        return list(range(n_slice - group + 1))\n    return [g * group for g in range(max(n_slice // group, 1))]\n\ndef apply_target_window_pool(values, probs, logits, original_probs, mapping, target_idx):\n    for target, mode in mapping.items():\n        j = target_idx[target]\n        if mode == \'max\':\n            values[:, j] = probs[:, :, j].max(0).values\n        elif mode == \'mean\':\n            values[:, j] = probs[:, :, j].mean(0)\n        elif mode == \'logit_mean\':\n            values[:, j] = torch.sigmoid(logits[:, :, j].mean(0))\n        elif mode == \'original_mean\':\n            values[:, j] = original_probs[:, :, j].mean(0)\n        elif mode in (\'top2\', \'top3\'):\n            k = min(int(mode[3:]), probs.shape[0])\n            values[:, j] = probs[:, :, j].topk(k, dim=0).values.mean(0)\n        else:\n            raise ValueError(f\'unknown TTA pooling mode for {target}: {mode}\')\n    return values\n\ndef legacy_fold_soft_window_pool(original_probs, target_idx):\n    values = original_probs.mean(0).clone()\n    for target, beta in LEGACY_FOLD_SOFTPOOL_BETA.items():\n        j = target_idx[target]\n        x = original_probs[:, :, j]\n        weight = torch.softmax(float(beta) * x, dim=0)\n        values[:, j] = (weight * x).sum(0)\n    return values\n\n@torch.no_grad()\ndef predict_member(model, cache, mask, idx, dev, img_size, group=None, pool=None, starts=None, jitter=False, jitter_seed=SEED, return_public_frontier=False):\n    group = GROUP if group is None else group\n    pool = TTA_POOL if pool is None else pool\n    starts = window_starts(cache.shape[2], group) if starts is None else list(starts)\n    if not starts:\n        raise ValueError(\'predict_member was given no windows to average over\')\n    target_idx = {t: j for j, t in enumerate(TARGETS)}\n    unknown = (set(TTA_TARGET_POOL) | set(PUBLIC_FRONTIER_TARGET_POOL)) - set(target_idx)\n    if unknown:\n        raise ValueError(f\'unknown target(s) in TTA_TARGET_POOL: {unknown}\')\n    jitter_gen = torch.Generator(device=dev)\n    jitter_gen.manual_seed(int(jitter_seed) % (2 ** 63 - 1))\n    model.eval()\n    out, public_frontier_out, public_soft_out = ([], [], [])\n    for b in range(0, len(idx), EVAL_BATCH):\n        sel = idx[b:b + EVAL_BATCH]\n        m = torch.from_numpy(mask[sel]).to(dev)\n        win_probs, win_logits, win_original_probs = ([], [], [])\n        for st in starts:\n            rows = torch.from_numpy(np.ascontiguousarray(cache[sel, :, st:st + group])).to(dev)\n            views = [rows] + ([augment(rows, generator=jitter_gen)] if jitter else [])\n            view_probs, view_logits = ([], [])\n            for view in views:\n                with torch.autocast(\'cuda\', enabled=dev.type == \'cuda\'):\n                    z = model(view, m, img_size).float()\n                if dev.type == \'cuda\' and not bool(torch.isfinite(z).all()):\n                    rsna_event(\'dino_legacy_fp16_nonfinite_retry_fp32\')\n                    with torch.autocast(\'cuda\', enabled=False):\n                        z = model(view, m, img_size).float()\n                view_logits.append(z)\n                view_probs.append(torch.sigmoid(z))\n            win_logits.append(torch.stack(view_logits).mean(0))\n            win_probs.append(torch.stack(view_probs).mean(0))\n            win_original_probs.append(view_probs[0])\n        probs = torch.stack(win_probs)\n        logits = torch.stack(win_logits)\n        original_probs = torch.stack(win_original_probs)\n        v = torch.sigmoid(logits.mean(0)) if pool == \'logit\' else probs.mean(0)\n        v = apply_target_window_pool(v, probs, logits, original_probs, TTA_TARGET_POOL, target_idx)\n        out.append(v.cpu().numpy())\n        if return_public_frontier:\n            public_v = apply_target_window_pool(original_probs.mean(0), original_probs, logits, original_probs, PUBLIC_FRONTIER_TARGET_POOL, target_idx)\n            public_frontier_out.append(public_v.cpu().numpy())\n            public_soft = legacy_fold_soft_window_pool(original_probs, target_idx)\n            public_soft_out.append(public_soft.cpu().numpy())\n    primary = np.concatenate(out) if out else np.zeros((0, len(TARGETS)), np.float32)\n    if not return_public_frontier:\n        return primary\n    public_frontier = np.concatenate(public_frontier_out) if public_frontier_out else np.zeros((0, len(TARGETS)), np.float32)\n    public_soft = np.concatenate(public_soft_out) if public_soft_out else np.zeros((0, len(TARGETS)), np.float32)\n    return (primary, public_frontier, public_soft)\nBUILD_LOCK = threading.Lock()\nSTATE_LOCK = threading.Lock()\nLEGACY_BUNDLE_FILE = \'rsna_20260807_v1.pt\'\nLEGACY_WEIGHT = 0.5\n\ndef find_legacy_bundle():\n    candidates = [\n        Path(\'/kaggle/input/datasets/pilkwang/rsna-knee-weights\') / LEGACY_BUNDLE_FILE,\n        Path(\'/kaggle/input/rsna-knee-weights\') / LEGACY_BUNDLE_FILE,\n    ]\n    hits = [path for path in candidates if path.is_file()]\n    if len(hits) > 1:\n        raise WeightsError(f\'ambiguous legacy bundle: {hits}\')\n    return hits[0] if hits else None\n\ndef legacy_group_members():\n    return {}\n\ndef _run_member(path, m, dev, Cte, Mte, idx, starts, jitter):\n    t0 = time.time()\n    with BUILD_LOCK:\n        if \'state\' in m:\n            raise WeightsError(f"{m[\'id\']}: inline legacy state is forbidden")\n        ck = torch.load(Path(path) / m[\'file\'], map_location=\'cpu\', weights_only=False)\n        state, fp = (ck[\'model\'], ck.get(\'fingerprint\'))\n        if fp is None:\n            raise WeightsError(f"{m[\'id\']}: stored fingerprint is required")\n        model = build_model(int(m[\'config\'][\'unfreeze_last\']), variant=m[\'config\'][\'variant\'], pool=m[\'config\'].get(\'pool\', \'cls_mean\'), prior=bool(m[\'config\'].get(\'prior\', False))).to(dev)\n        model.load_state_dict(state)\n        check_fingerprint(model, dev, IMG, fp, tag=f"{m[\'id\']}: ")\n        global _DINOV2_MATCHED_MEMBERS\n        _DINOV2_MATCHED_MEMBERS += 1\n    t_ready = time.time()\n    jitter_seed = SEED + int(hashlib.sha256(str(m[\'id\']).encode()).hexdigest()[:8], 16)\n    public_member = \'state\' not in m\n    predicted = predict_member(model, Cte, Mte, idx, dev, IMG, starts=starts, jitter=jitter, jitter_seed=jitter_seed, return_public_frontier=public_member)\n    if public_member:\n        p, public_p, public_soft = predicted\n    else:\n        p, public_p, public_soft = (predicted, None, None)\n    t_done = time.time()\n    del model, state\n    gc.collect()\n    if dev.type == \'cuda\':\n        with torch.cuda.device(dev):\n            torch.cuda.empty_cache()\n    passes = len(starts) * (2 if jitter else 1)\n    return (p, public_p, public_soft, (t_ready - t0, (t_done - t_ready) / max(passes, 1)))\n\ndef _combine(per_member):\n    all_ids = sorted({s for m in per_member for s in m[\'ids\']})\n    pos = {s: i for i, s in enumerate(all_ids)}\n    acc = np.zeros((len(all_ids), len(TARGETS)), np.float64)\n    tot = np.zeros(len(TARGETS), np.float64)\n    for m in per_member:\n        target_weight = m.get(\'target_weight\')\n        w = np.asarray(target_weight if target_weight is not None else [float(m.get(\'weight\', 1.0))] * len(TARGETS), dtype=np.float64)\n        if w.shape != (len(TARGETS),) or np.any(w < 0):\n            raise ValueError(f"invalid target weights for {m.get(\'id\')}: {w}")\n        rsna_finite(m[\'pred\'],str(m[\'id\'])+\' retained raw\',probability=True)\n        r = pd.DataFrame(m[\'pred\']).rank(pct=True).to_numpy()\n        acc[[pos[s] for s in m[\'ids\']]] += r * w[None, :]\n        tot += w\n    if np.any(tot <= 0):\n        raise ValueError(f\'at least one target has no ensemble vote: {tot}\')\n    return (all_ids, acc / tot[None, :])\n\ndef combine_public_members_by_fold(per_member, pred_key=\'pred\'):\n    # Raw-average the four members within each fold, rank each fold,\n    # then give all five folds equal weight.\n    all_ids = sorted({study for member in per_member for study in member[\'ids\']})\n    position = {study: i for i, study in enumerate(all_ids)}\n    groups = {}\n    for i, member in enumerate(per_member):\n        fold = member.get(\'fold\')\n        key = f\'fold_{fold}\' if fold is not None else f\'member_{i}\'\n        groups.setdefault(key, []).append(member)\n    fold_ranks, diagnostics = ([], [])\n    for key, members_in_fold in sorted(groups.items()):\n        matrices = []\n        for member in members_in_fold:\n            values = np.full((len(all_ids), len(TARGETS)), np.nan, np.float64)\n            values[[position[study] for study in member[\'ids\']]] = np.asarray(member[pred_key], np.float64)\n            if np.isnan(values).any():\n                raise WeightsError(f"{member.get(\'id\')}: incomplete {pred_key} coverage")\n            matrices.append(values)\n        raw_fold_mean = np.mean(matrices, axis=0)\n        fold_ranks.append(pd.DataFrame(raw_fold_mean).rank(method=\'average\', pct=True).to_numpy(np.float64))\n        diagnostics.append({\'ensemble_group\': key, \'members\': len(members_in_fold)})\n    if len(fold_ranks) != 5:\n        raise WeightsError(f\'legacy branch requires five folds, found {len(fold_ranks)}\')\n    return all_ids, np.mean(fold_ranks, axis=0), pd.DataFrame(diagnostics)\n\ndef blend_legacy_frontier_and_soft(frontier_rank, soft_rank):\n    output = np.asarray(frontier_rank, np.float64).copy()\n    for j, target in enumerate(TARGETS):\n        alpha = float(LEGACY_FOLD_SOFTPOOL_ALPHA.get(target, 0.0))\n        if alpha:\n            output[:, j] = (1.0 - alpha) * frontier_rank[:, j] + alpha * soft_rank[:, j]\n    return output\n\ndef infer_from_package(path, dev=None):\n    man = json.loads((Path(path) / \'manifest.json\').read_text())\n    members = man[\'members\']\n    if len(members) != EXPECTED_DINOV2_MEMBERS:\n        raise WeightsError(f\'expected {EXPECTED_DINOV2_MEMBERS} DINOv2 members, found {len(members)}\')\n    log(f\'weights package: {len(members)} member(s) from {path}; {len(DEVS)} device(s)\')\n    test_df = pd.read_csv(ROOT / \'test.csv\')\n    test_series = pd.read_csv(ROOT / \'test_series.csv\')\n    plane_map = dict(zip(test_series[\'SeriesInstanceUID\'], test_series[\'Anatomical_Plane\']))\n    hte = annotate(walk(\'test_series\'))\n    log(f\'test header pass: {len(hte)} series\')\n    groups = {}\n    for m in members:\n        groups.setdefault(m[\'pixel_group\'], []).append(m)\n    groups.update(legacy_group_members())\n    per_member, public_frontier_members = ([], [])\n    failures = []\n    abort = threading.Event()\n    est = {\'fixed\': None, \'win\': None}\n\n    def bank(m, ids, pred, starts, jitter, public_pred=None, public_soft=None):\n        if not np.isfinite(pred).all():\n            rsna_event(\'dino_member_nonfinite_neutral\', member=str(m[\'id\']), rows=int((~np.isfinite(pred)).reshape(len(pred), -1).any(axis=1).sum()))\n            pred = np.where(np.isfinite(pred), pred, 0.5)\n        if public_pred is not None and not np.isfinite(public_pred).all():\n            rsna_event(\'dino_member_public_nonfinite_neutral\', member=str(m[\'id\']))\n            public_pred = np.where(np.isfinite(public_pred), public_pred, 0.5)\n        rsna_finite(pred, m[\'id\']+\' DINO raw\', probability=True)\n        if public_pred is not None: rsna_finite(public_pred,m[\'id\']+\' DINO retained raw\',probability=True)\n        rsna_save_predictions(\'dino_\'+str(m[\'id\']),ids,public_pred if public_pred is not None else pred,TARGETS)\n        if float(np.std(pred)) < 1e-09:\n            raise WeightsError(f"{m[\'id\']}: degenerate predictions")\n        with STATE_LOCK:\n            per_member.append({\'id\': m[\'id\'], \'fold\': m.get(\'fold\'), \'ids\': ids, \'pred\': pred, \'weight\': m.get(\'weight\', 1.0), \'target_weight\': m.get(\'target_weight\'), \'holdout\': m.get(\'holdout\')})\n            if public_pred is not None and len(starts) == len(starts_full):\n                if float(np.std(public_pred)) < 1e-09:\n                    raise WeightsError(f"{m[\'id\']}: degenerate public-frontier prediction")\n                public_frontier_members.append({\'id\': m[\'id\'], \'fold\': m.get(\'fold\'), \'ids\': ids, \'pred\': public_pred, \'soft_pred\': public_soft})\n            elif public_pred is not None:\n                raise WeightsError(f"{m[\'id\']}: incomplete public-frontier windows")\n            log(f"  banked {m[\'id\']} fold {m.get(\'fold\', \'?\')} ({len(starts)} window(s); {len(per_member)} member(s)")\n    for gi, (key, gm) in enumerate(groups.items(), 1):\n        cfg = json.loads(key)\n        adopt_config_globals(cfg)\n        log(f"decode group {gi}/{len(groups)}: {cfg[\'img\']}px x {cfg[\'slices\']} slices, crop {cfg[\'crop_mm\']} mm -> {len(gm)} member(s)")\n        st_te, Cte, Mte = build_cache(pick_slots(hte, plane_map), plane_map, lat_of(hte, \'test \'), f\'test g{gi}\')\n        idx = np.arange(len(st_te))\n        starts_full = window_starts(Cte.shape[2], GROUP)\n        if len(groups) == 1 and _shared_prefix_eligible(gm):\n            shared_results = _run_shared_dino_group(\n                path,\n                gm,\n                Cte,\n                Mte,\n                idx,\n                starts_full,\n            )\n            for member, predicted, jitter in shared_results:\n                prediction, public, soft = predicted\n                bank(\n                    member,\n                    st_te,\n                    prediction,\n                    starts_full,\n                    jitter,\n                    public,\n                    soft,\n                )\n            rsna_release_pixels(Cte)\n            del Cte, Mte, shared_results\n            gc.collect()\n            continue\n        pending = sorted(gm, key=lambda m: -(m.get(\'holdout\') or 0))\n        left_after = sum((len(g) for j, (_, g) in enumerate(groups.items(), 1) if j > gi))\n\n        def pop_next():\n            with STATE_LOCK:\n                if not pending:\n                    return (None, None, False)\n                rsna_deadline(\'DINO complete-member inference\')\n                starts, jit = starts_full, False\n                return (pending.pop(0), starts, jit)\n\n        def worker(dev):\n            while not abort.is_set():\n                m, starts, jit = pop_next()\n                if m is None:\n                    return\n                try:\n                    p, public_p, public_soft, (fs, ws) = _run_member(path, m, dev, Cte, Mte, idx, starts, jit)\n                    with STATE_LOCK:\n                        est[\'fixed\'], est[\'win\'] = (fs, ws)\n                    bank(m, st_te, p, starts, jit, public_p, public_soft)\n                except Exception as exc:\n                    with STATE_LOCK:\n                        failures.append((m[\'id\'], str(dev), type(exc).__name__, str(exc)))\n                    abort.set()\n                    return\n        threads = [threading.Thread(target=worker, args=(d,)) for d in DEVS]\n        for t in threads:\n            t.start()\n        for t in threads:\n            t.join()\n        if failures:\n            raise WeightsError(f\'DINOv2 member failure; no fallback permitted: {failures[0]}\')\n        rsna_release_pixels(Cte)\n        del Cte, Mte\n        gc.collect()\n    if _DINOV2_MATCHED_MEMBERS != EXPECTED_DINOV2_MEMBERS:\n        raise WeightsError(f\'fingerprint gate failed: {_DINOV2_MATCHED_MEMBERS} / {EXPECTED_DINOV2_MEMBERS}\')\n    if len(public_frontier_members) != EXPECTED_DINOV2_MEMBERS:\n        raise WeightsError(f\'public-frontier inference incomplete: {len(public_frontier_members)} / {EXPECTED_DINOV2_MEMBERS} members\')\n    log(f\'DINOv2 fail-closed gate PASS: {_DINOV2_MATCHED_MEMBERS}/{EXPECTED_DINOV2_MEMBERS} fingerprints and members\')\n    frontier_ids, frontier_acc = _combine(public_frontier_members)\n    sub = write_submission(frontier_acc, frontier_ids, test_df, \'_pipeline_stage.csv\')\n    log(f\'submission.csv = exact no-jitter public-frontier rank mean of {len(public_frontier_members)} member(s); {sub.shape}\')\n    return sub\n\ndef adopt_config_globals(cfg):\n    global IMG, CACHE_IMG, GROUP, CACHE_SLICES, N_GROUP, CROP_MM, SLICE_BAND, RULES\n    CACHE_IMG = IMG = int(cfg[\'img\'])\n    GROUP = int(cfg[\'group\'])\n    CACHE_SLICES = int(cfg[\'slices\'])\n    N_GROUP = max(CACHE_SLICES // GROUP, 1)\n    CROP_MM = float(cfg[\'crop_mm\'])\n    SLICE_BAND = tuple((float(x) for x in cfg[\'band\']))\n    rules = cfg.get(\'rules\') or RULES_NATIVE\n    unknown = {k: v for k, v in rules.items() if k not in RULES_NATIVE or v not in (RULES_NATIVE[k], RULES_LEGACY[k])}\n    if unknown:\n        raise WeightsError(f\'the members record pixel rules this pipeline cannot reproduce: {unknown}\')\n    RULES = {**RULES_NATIVE, **rules}\n    if [s[0] for s in SLOTS] != list(cfg[\'slots\']):\n        raise WeightsError(f"the members were fitted on slots {cfg[\'slots\']} and this pipeline defines {[s[0] for s in SLOTS]}; a weight would be read against the wrong slot")\n', globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(16, "def write_submission(pred, studies, test_df, path):\n    sub = pd.DataFrame(pd.DataFrame(pred).rank(pct=True).values, columns=TARGETS)\n    sub.insert(0, 'StudyInstanceUID', studies)\n    sub = test_df[['StudyInstanceUID']].merge(sub, on='StudyInstanceUID', how='left')\n    if sub[TARGETS].isna().any().any():\n        rsna_event('submission_neutral_fill', studies=int(sub[TARGETS].isna().any(axis=1).sum()))\n        sub[TARGETS] = sub[TARGETS].fillna(0.5)\n    sub.to_csv(path, index=False)\n    return sub\n\ndef _v37_validate_submission(path, test_df, tag):\n    frame = pd.read_csv(path)\n    expected = ['StudyInstanceUID'] + TARGETS\n    if list(frame.columns) != expected:\n        raise ValueError(f'{tag}: columns differ from the competition contract')\n    if len(frame) != len(test_df) or not frame['StudyInstanceUID'].is_unique:\n        raise ValueError(f'{tag}: row count or StudyInstanceUID uniqueness failed')\n    if set(frame['StudyInstanceUID'].astype(str)) != set(test_df['StudyInstanceUID'].astype(str)):\n        raise ValueError(f'{tag}: StudyInstanceUID set differs from test.csv')\n    values = frame[TARGETS].to_numpy(np.float64)\n    if not np.isfinite(values).all():\n        raise ValueError(f'{tag}: non-finite prediction')\n    return frame\n\ndef main():\n    pkg = find_weights()\n    if pkg is None:\n        raise WeightsError('required public checkpoint manifest not found')\n    infer_from_package(pkg, DEVS[0])\n    _v37_validate_submission('_pipeline_stage.csv', pd.read_csv(ROOT / 'test.csv'), 'public frontier')\n", globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(17, "rsna_phase('dinov2', 'START')\nmain()\nlog('done')\n\n_speed_dino_templates.clear()\ngc.collect()\n\nrsna_phase('dinov2', 'COMPLETE')\n", globals())


## Stage 2 — A5 attention pooling

Five folds pool variable-length slice features. Host preparation stays bounded
and overlaps the active GPU pass; unavailable acquisitions fall back to the
base branch, while model failures remain fatal.


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(19, 'if globals().get(\'_DINOV2_MATCHED_MEMBERS\') != 20:\n    raise RuntimeError(\'DINOv2 20/20 fingerprint gate did not pass\')\nrsna_phase(\'a5\', \'START\')\n_A5_SAVED = dict(globals())\nimport gc, os, time, warnings\nfrom concurrent.futures import ThreadPoolExecutor as A5DecodePool, as_completed\nfrom pathlib import Path\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport pydicom\nimport timm\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nwarnings.filterwarnings(\'ignore\')\ncv2.setNumThreads(1)\nCROP_MM = 130.0\nSIZE = 336\nSLICE_BAND = (0.12, 0.88)\nN_SLICE = 16\nINTENSITY = \'slice\'\nSLOTS = [(\'Sagittal\', 1), (\'Sagittal\', 0), (\'Coronal\', 1), (\'Coronal\', 0), (\'Axial\', 1), (\'Axial\', 0)]\nN_SLOT = len(SLOTS)\nLABELS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\', \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\n\ndef _one_a5_root(tag, candidates, marker):\n    hits = [Path(path) for path in candidates if Path(path).is_dir() and (Path(path) / marker).exists()]\n    if len(hits) != 1:\n        raise RuntimeError(f\'{tag}: expected one direct root, found {hits}\')\n    return hits[0]\n\nCOMP = Path(ROOT)\nCKPT = _one_a5_root(\'A5 weights\', [\n    \'/kaggle/input/datasets/mattiaangeli/knee-mri-fold-weights\',\n    \'/kaggle/input/knee-mri-fold-weights\',\n], \'m_f0.pt\')\nassert COMP is not None, \'competition data not attached\'\nassert CKPT is not None, \'fold weights not attached\'\nassert (COMP / \'sample_submission.csv\').exists(), f\'no competition data at {COMP}\'\nassert list(CKPT.glob(\'*_f*.pt\')), f\'no checkpoints at {CKPT}\'\nDEV = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n', globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(20, "SERIES_ROOT = COMP / 'test_series'\nif not SERIES_ROOT.is_dir():\n    raise FileNotFoundError('Missing test_series; train-series fallback is forbidden')\nprint('series root:', SERIES_ROOT)\n\ndef ordered_files(sdir, cap=64):\n    # Preserve the parent's InstanceNumber ordering; remove pre-sort truncation.\n    # The unused cap argument remains only for call compatibility.\n    keyed=[]; unreadable=[]\n    for f in sorted(sdir.glob('*.dcm')):\n        try:\n            ds=pydicom.dcmread(str(f),stop_before_pixels=True)\n            key=int(ds.InstanceNumber)\n        except Exception as exc:\n            unreadable.append(str(f)); continue\n        keyed.append((key,str(f)))\n    if unreadable:\n        rsna_event('a5_order_unreadable_dropped',series=str(sdir),unreadable=len(unreadable),readable=len(keyed),examples=unreadable[:3])\n    return [f for _,f in sorted(keyed)]\n\ndef series_side(path):\n    try:\n        return float(pydicom.dcmread(path, stop_before_pixels=True).ImagePositionPatient[0])\n    except Exception:\n        return 0.0\n\ndef read_crop(path):\n    try:\n        ds = pydicom.dcmread(path)\n        arr = ds.pixel_array.astype(np.float32)\n    except Exception:\n        return None\n    try:\n        ps = float(ds.PixelSpacing[0])\n    except Exception:\n        ps = CROP_MM / max(arr.shape)\n    half = int(round(CROP_MM / ps / 2))\n    cy, cx = (arr.shape[0] // 2, arr.shape[1] // 2)\n    y0, y1 = (max(0, cy - half), min(arr.shape[0], cy + half))\n    x0, x1 = (max(0, cx - half), min(arr.shape[1], cx + half))\n    crop = arr[y0:y1, x0:x1]\n    return None if crop.size == 0 else crop\n\ndef window(crop, lo, hi, flip):\n    c = np.clip((crop - lo) / max(hi - lo, 1e-06), 0, 1)\n    img = cv2.resize(c, (SIZE, SIZE), interpolation=cv2.INTER_AREA)\n    return img[:, ::-1].copy() if flip else img\n\ndef render(path, flip):\n    crop = read_crop(path)\n    if crop is None:\n        return None\n    lo, hi = np.percentile(crop[::4, ::4], [1, 99])\n    return window(crop, lo, hi, flip)\n\ndef build_study(args):\n    idx,study,recs=args\n    out=np.zeros((N_SLOT,N_SLICE,SIZE,SIZE),np.uint8);mask=np.zeros(N_SLOT,np.uint8)\n    rows=pd.DataFrame(recs)\n    if not len(rows):\n        rsna_event('a5_no_metadata',study=str(study)); return idx,out,mask\n    for s_i,(plane,fs) in enumerate(SLOTS):\n        sub=rows[(rows.Anatomical_Plane==plane)&(rows.Fat_Suppression==fs)]\n        if sub.empty: continue  # Actual acquisition absence, not a decoding failure.\n        series=str(sub.iloc[0].SeriesInstanceUID)\n        files=ordered_files(SERIES_ROOT/str(study)/series)\n        if not files:\n            rsna_event('a5_slot_no_files',study=str(study),series=series); continue\n        flip=plane!='Sagittal' and series_side(files[0])<0\n        i0=int(round(SLICE_BAND[0]*(len(files)-1)));i1=int(round(SLICE_BAND[1]*(len(files)-1)))\n        avail=list(range(i0,i1+1))\n        if len(avail)>=N_SLICE:\n            picks=[avail[int(round(t))] for t in np.linspace(0,len(avail)-1,N_SLICE)];off=0\n        else:picks,off=avail,(N_SLICE-len(avail))//2\n        rendered=0;errors=[]\n        if INTENSITY=='series':\n            crops=[read_crop(files[p]) for p in picks];got=[x for x in crops if x is not None]\n            if not got:\n                rsna_event('a5_slot_all_crops_failed',study=str(study),series=series); continue\n            samp=np.concatenate([x[::4,::4].ravel() for x in got]);lo_,hi_=np.percentile(samp,[1,99])\n            for c,x in enumerate(crops):\n                if x is None:x=read_crop(files[min(len(files)-1,picks[c]+1)])\n                if x is None:errors.append(files[picks[c]]);continue\n                img=window(x,lo_,hi_,flip)\n                rsna_finite(img,'A5 rendered pixels')\n                out[s_i,off+c]=(img*255).astype(np.uint8);rendered+=1\n        else:\n            for c,p in enumerate(picks):\n                img=render(files[p],flip)\n                if img is None:img=render(files[min(len(files)-1,p+1)],flip)\n                if img is None:errors.append(files[p]);continue\n                rsna_finite(img,'A5 rendered pixels')\n                out[s_i,off+c]=(img*255).astype(np.uint8);rendered+=1\n        if errors or rendered!=len(picks):\n            rsna_event('a5_decode_failure',study=str(study),series=series,failed=errors,rendered=int(rendered),wanted=int(len(picks)))\n            if not rendered: continue\n        mask[s_i]=rendered\n    if not mask.any():\n        rsna_event('a5_empty_study',study=str(study))\n        print(f'[a5] {study}: no acquired slots; all-zero mask kept (parent policy)', flush=True)\n    return idx,out,mask\nsub_df = pd.read_csv(COMP / 'sample_submission.csv',dtype={'StudyInstanceUID':str})\nser_csv = pd.read_csv(COMP / 'test_series.csv',dtype={'StudyInstanceUID':str,'SeriesInstanceUID':str})\nif set(sub_df.StudyInstanceUID)!=set(_RSNA_TEST_IDS):\n    raise RuntimeError('A5 test/sample UID mismatch')\nser_csv = ser_csv.loc[:, ~ser_csv.columns.duplicated()]\nstudies = sub_df.StudyInstanceUID.tolist()\nby = {s: g.to_dict('records') for s, g in ser_csv[ser_csv.StudyInstanceUID.isin(set(studies))].groupby('StudyInstanceUID')}\nprint(f'{len(studies):,} test studies, {len(by):,} with series metadata')\n", globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(21, "N_SLOT_TYPES, MASK_IDX = (6, 0)\n\ndef segment_softmax(scores, sidx, B):\n    T, K = scores.shape\n    idx = sidx.unsqueeze(1).expand(-1, K)\n    m = torch.full((B, K), float('-inf'), device=scores.device, dtype=scores.dtype)\n    m = m.scatter_reduce(0, idx, scores, reduce='amax', include_self=True)\n    e = (scores - m[sidx]).exp()\n    s = torch.zeros(B, K, device=scores.device, dtype=scores.dtype).index_add_(0, sidx, e)\n    return e / s[sidx].clamp(min=1e-06)\n\nclass MeanMaxPool(nn.Module):\n\n    def forward(self, f, sidx, B, slot=None, return_attn=False):\n        D = f.shape[1]\n        cnt = torch.zeros(B, device=f.device, dtype=f.dtype).index_add_(0, sidx, torch.ones(f.shape[0], device=f.device, dtype=f.dtype))\n        mean = torch.zeros(B, D, device=f.device, dtype=f.dtype).index_add_(0, sidx, f)\n        mean = mean / cnt.clamp(min=1).unsqueeze(1)\n        mx = torch.full((B, D), -10000.0, device=f.device, dtype=f.dtype)\n        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), f, reduce='amax', include_self=True)\n        return (torch.cat([mean, mx], 1), None)\n\nclass LabelAttentionPool(nn.Module):\n\n    def __init__(self, d, n_labels=12, n_heads=4, slot_bias=True):\n        super().__init__()\n        self.d, self.k, self.h = (d, n_labels, n_heads)\n        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)\n        self.key, self.val = (nn.Linear(d, d), nn.Linear(d, d))\n        self.slot_bias = nn.Parameter(torch.zeros(n_labels, N_SLOT_TYPES + 1)) if slot_bias else None\n\n    def forward(self, f, sidx, B, slot=None, return_attn=False):\n        scores = self.key(f) @ self.q.t() / self.d ** 0.5\n        if self.slot_bias is not None and slot is not None:\n            scores = scores + self.slot_bias.t()[slot]\n        a = segment_softmax(scores, sidx, B)\n        out = torch.zeros(B, self.k, self.d, device=f.device, dtype=f.dtype)\n        out = out.index_add_(0, sidx, a.unsqueeze(-1) * self.val(f).unsqueeze(1))\n        return (out, a)\n\nclass TokenXAttnPool(nn.Module):\n\n    def __init__(self, d, n_labels=12, n_heads=6, dropout=0.2):\n        super().__init__()\n        self.d, self.k = (d, n_labels)\n        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)\n        self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, d, padding_idx=0)\n        self.kv_norm = nn.LayerNorm(d)\n        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)\n\n    def forward(self, tok, sidx, B, slot=None, return_attn=False):\n        T, N, D = tok.shape\n        cnt = torch.bincount(sidx, minlength=B)\n        S = int(cnt.max().item())\n        starts = torch.cumsum(cnt, 0) - cnt\n        pos = torch.arange(T, device=tok.device) - starts[sidx]\n        kv = tok + self.slot_emb(slot).unsqueeze(1)\n        pad = tok.new_zeros(B, S, N, D)\n        pad[sidx, pos] = kv\n        keep = torch.zeros(B, S, dtype=torch.bool, device=tok.device)\n        keep[sidx, pos] = True\n        kpm = ~keep.repeat_interleave(N, dim=1)\n        pad = self.kv_norm(pad.reshape(B, S * N, D))\n        q = self.q.unsqueeze(0).expand(B, -1, -1)\n        att, w = self.attn(q, pad, pad, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)\n        cls = tok[:, 0]\n        mean = torch.zeros(B, D, device=tok.device, dtype=tok.dtype).index_add_(0, sidx, cls) / cnt.clamp(min=1).unsqueeze(1)\n        mx = torch.full((B, D), -10000.0, device=tok.device, dtype=tok.dtype)\n        mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), cls, reduce='amax', include_self=True)\n        base = torch.cat([mean, mx], 1).unsqueeze(1).expand(-1, self.k, -1)\n        return (torch.cat([att, base], -1), w)\n\nclass ViTSlotToken(nn.Module):\n\n    def __init__(self, vit, n_cat, dim=None):\n        super().__init__()\n        self.vit = vit\n        d = dim or vit.embed_dim\n        self.tok = nn.Embedding(n_cat + 1, d, padding_idx=MASK_IDX)\n        self.num_features = vit.num_features\n        self._orig_prefix = getattr(vit, 'num_prefix_tokens', 1)\n        vit.num_prefix_tokens = self._orig_prefix + 1\n        for blk in vit.blocks:\n            a = getattr(blk, 'attn', None)\n            if a is not None and hasattr(a, 'num_prefix_tokens'):\n                a.num_prefix_tokens = a.num_prefix_tokens + 1\n\n    @staticmethod\n    def _maybe(mod, x):\n        return x if mod is None else mod(x)\n\n    def forward_features(self, x, cat):\n        v = self.vit\n        x = v.patch_embed(x)\n        pos = v._pos_embed(x)\n        rope = None\n        if isinstance(pos, tuple):\n            x, rope = pos\n        else:\n            x = pos\n        x = self._maybe(getattr(v, 'patch_drop', None), x)\n        x = self._maybe(getattr(v, 'norm_pre', None), x)\n        npt = self._orig_prefix\n        tok = self.tok(cat).unsqueeze(1)\n        x = torch.cat([x[:, :npt], tok, x[:, npt:]], dim=1)\n        if rope is not None:\n            if getattr(v, 'rope_mixed', False):\n                for i, blk in enumerate(v.blocks):\n                    x = blk(x, rope=rope[i])\n            else:\n                for blk in v.blocks:\n                    x = blk(x, rope=rope)\n        else:\n            x = v.blocks(x)\n        return v.norm(x)\n\n    def forward_head(self, x, pre_logits=True):\n        return self.vit.forward_head(x, pre_logits=pre_logits)\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\n\nclass _GatedDepthBlock(nn.Module):\n\n    def __init__(self, n_slice, dropout=0.0, ls_init=0.1):\n        super().__init__()\n        self.norm = nn.GroupNorm(1, n_slice)\n        self.v = nn.Conv2d(n_slice, n_slice, 1)\n        self.g = nn.Conv2d(n_slice, n_slice, 1)\n        self.out = nn.Conv2d(n_slice, n_slice, 1)\n        self.gamma = nn.Parameter(torch.full((n_slice, 1, 1), ls_init))\n        self.drop = nn.Dropout2d(dropout) if dropout else nn.Identity()\n\n    def forward(self, x):\n        z = self.norm(x)\n        return x + self.gamma * self.drop(self.out(self.v(z) * F.silu(self.g(z))))\n\nclass DepthCompress(nn.Module):\n\n    def __init__(self, n_slice=16, out_ch=3, depth=1, dropout=0.0, ls_init=0.1, imagenet=True, proj_noise=0.25):\n        super().__init__()\n        self.imagenet = imagenet\n        self.blocks = nn.ModuleList([_GatedDepthBlock(n_slice, dropout, ls_init) for _ in range(depth)])\n        self.proj = nn.Conv2d(n_slice, out_ch, 1, bias=True)\n        if imagenet:\n            self.register_buffer('mu', torch.tensor(IMAGENET_MEAN).view(1, -1, 1, 1))\n            self.register_buffer('sd', torch.tensor(IMAGENET_STD).view(1, -1, 1, 1))\n\n    def forward(self, x):\n        keep = (x.amax(dim=1, keepdim=True) > 0).to(x.dtype)\n        z = x\n        for b in self.blocks:\n            z = b(z)\n        z = self.proj(z)\n        if self.imagenet:\n            z = (z - self.mu.to(z.dtype)) / self.sd.to(z.dtype)\n        return z * keep\nN_PLANE, N_CONTRAST = (3, 2)\n_PLANE_OF = lambda s: torch.clamp(s - 1, 0, 5) // 2\n_CONTRAST_OF = lambda s: torch.clamp(s - 1, 0, 5) % 2\n\nclass SlotDepthMixer(nn.Module):\n\n    def __init__(self, n_slice=16, ksize=5, alpha_max=0.25):\n        super().__init__()\n        self.n_slice, self.ksize, self.r = (n_slice, ksize, ksize // 2)\n        self.alpha_max = alpha_max\n        b = torch.tensor([1.0, 4.0, 6.0, 4.0, 1.0])\n        self.register_buffer('base', b.log()[self.r:])\n        n_u = self.r + 1\n        self.shared = nn.Parameter(torch.zeros(n_u))\n        self.plane_k = nn.Parameter(torch.zeros(N_PLANE, n_u))\n        self.contrast_k = nn.Parameter(torch.zeros(N_CONTRAST, n_u))\n        self.g0 = nn.Parameter(torch.zeros(()))\n        self.gate_p = nn.Parameter(torch.zeros(N_PLANE))\n        self.gate_c = nn.Parameter(torch.zeros(N_CONTRAST))\n        idx = torch.arange(n_slice)\n        self.register_buffer('off', idx[None, :] - idx[:, None])\n\n    def kernel(self, slot):\n        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))\n        half = self.base + self.shared + self.plane_k[p] + self.contrast_k[c]\n        full = torch.cat([half.flip(-1)[..., :self.r], half], dim=-1)\n        return F.softmax(full, dim=-1)\n\n    def alpha(self, slot):\n        p, c = (_PLANE_OF(slot), _CONTRAST_OF(slot))\n        return self.alpha_max * torch.tanh(self.g0 + self.gate_p[p] + self.gate_c[c])\n\n    def forward(self, x, slot, vmask):\n        T, S, H, W = x.shape\n        if vmask is None:\n            raise ValueError('stem=mixer requires the padding mask')\n        k = self.kernel(slot)\n        v = vmask.to(k.dtype)\n        d = self.off + self.r\n        inb = (d >= 0) & (d < self.ksize)\n        kk = k[:, d.clamp(0, self.ksize - 1)] * inb\n        M = kk * v[:, None, :]\n        den = M.sum(-1, keepdim=True)\n        eye = torch.eye(S, device=x.device, dtype=M.dtype).expand(T, S, S)\n        ok = (den > 1e-06) & v[:, :, None].bool()\n        M = torch.where(ok, M / den.clamp(min=1e-06), eye)\n        a = self.alpha(slot)[:, None, None]\n        Aop = ((1.0 - a) * eye + a * M).to(x.dtype)\n        if x.is_contiguous(memory_format=torch.channels_last) and (not x.is_contiguous()):\n            y = torch.bmm(x.permute(0, 2, 3, 1).reshape(T, H * W, S), Aop.transpose(1, 2))\n            return y.reshape(T, H, W, S).permute(0, 3, 1, 2)\n        return torch.bmm(Aop, x.reshape(T, S, H * W)).reshape(T, S, H, W)\n\ndef _seg_mean_max(v, sidx, B):\n    D = v.shape[1]\n    cnt = torch.zeros(B, device=v.device, dtype=v.dtype).index_add_(0, sidx, torch.ones(v.shape[0], device=v.device, dtype=v.dtype))\n    mean = torch.zeros(B, D, device=v.device, dtype=v.dtype).index_add_(0, sidx, v)\n    mean = mean / cnt.clamp(min=1).unsqueeze(1)\n    mx = torch.full((B, D), -10000.0, device=v.device, dtype=v.dtype)\n    mx = mx.scatter_reduce(0, sidx.unsqueeze(1).expand(-1, D), v, reduce='amax', include_self=True)\n    return torch.cat([mean, mx], 1)\n\ndef _pad_kv(x, sidx, B, norm):\n    T, P, D = x.shape\n    cnt = torch.bincount(sidx, minlength=B)\n    S = int(cnt.max().item())\n    starts = torch.cumsum(cnt, 0) - cnt\n    pos = torch.arange(T, device=x.device) - starts[sidx]\n    pad = x.new_zeros(B, S, P, D)\n    pad[sidx, pos] = x\n    keep = torch.zeros(B, S, dtype=torch.bool, device=x.device)\n    keep[sidx, pos] = True\n    return (norm(pad.reshape(B, S * P, D)), ~keep.repeat_interleave(P, dim=1))\n\nclass _GatedDelta(nn.Module):\n\n    def __init__(self, d, n_labels, n_heads, dropout):\n        super().__init__()\n        self.q = nn.Parameter(torch.randn(n_labels, d) * 0.02)\n        self.kv_norm = nn.LayerNorm(d)\n        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)\n        self.d_norm = nn.LayerNorm(d)\n        self.dw = nn.Parameter(torch.randn(n_labels, d) * (1.0 / d ** 0.5))\n        self.db = nn.Parameter(torch.zeros(n_labels))\n        self.gate = nn.Parameter(torch.zeros(n_labels))\n\n    def delta(self, pat, sidx, B, return_attn):\n        kv, kpm = _pad_kv(pat, sidx, B, self.kv_norm)\n        q = self.q.unsqueeze(0).expand(B, -1, -1)\n        att, w = self.attn(q, kv, kv, key_padding_mask=kpm, need_weights=return_attn, average_attn_weights=True)\n        return ((self.d_norm(att) * self.dw).sum(-1) + self.db, w)\n\nclass TokenResidualPool(_GatedDelta):\n\n    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):\n        super().__init__(d, n_labels, n_heads, dropout)\n        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))\n\n    def forward(self, tok, slot, sidx, B, pres, return_attn=False):\n        base = self.base(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), pres], 1))\n        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)\n        return (base + self.gate * d_, w)\n\nclass CodexResidualPool(_GatedDelta):\n\n    def __init__(self, d, n_labels=12, n_heads=6, pe=64, dropout=0.2):\n        super().__init__(d, n_labels, n_heads, dropout)\n        self.base = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(dropout), nn.Linear(2 * d + pe, n_labels))\n\n    def forward(self, tok, slot, sidx, B, pres, return_attn=False):\n        base = self.base(torch.cat([_seg_mean_max(tok[:, 0], sidx, B), pres], 1))\n        d_, w = self.delta(tok[:, 1:], sidx, B, return_attn)\n        return (base + self.gate * d_, w)\n\nclass ClsAddPool(nn.Module):\n\n    def __init__(self, d, n_labels=12, pe=64, dropout=0.2):\n        super().__init__()\n        self.net = nn.Sequential(nn.LayerNorm(4 * d + pe), nn.Dropout(dropout), nn.Linear(4 * d + pe, n_labels))\n\n    def forward(self, tok, slot, sidx, B, pres, return_attn=False):\n        return (self.net(torch.cat([_seg_mean_max(tok[:, 1:].mean(1), sidx, B), _seg_mean_max(tok[:, 0], sidx, B), pres], 1)), None)\n\nclass Readout(nn.Module):\n\n    def __init__(self, pool, d, n_labels=12, pe=64):\n        super().__init__()\n        self.pool_kind, self.k = (pool, n_labels)\n        self.pres_emb = nn.Embedding(N_SLOT_TYPES + 1, pe, padding_idx=0)\n        if pool in ('xres', 'clsadd', 'xcodex'):\n            self.pool = {'xres': TokenResidualPool, 'clsadd': ClsAddPool, 'xcodex': CodexResidualPool}[pool](d, n_labels, pe=pe)\n        elif pool in ('attn', 'xattn'):\n            if pool == 'xattn':\n                self.pool = TokenXAttnPool(d, n_labels)\n                wd = 3 * d + pe\n            else:\n                self.pool = LabelAttentionPool(d, n_labels)\n                wd = d + pe\n            self.norm = nn.LayerNorm(wd)\n            self.w = nn.Parameter(torch.randn(n_labels, wd) * (1.0 / wd ** 0.5))\n            self.b = nn.Parameter(torch.zeros(n_labels))\n        else:\n            self.pool = MeanMaxPool()\n            self.net = nn.Sequential(nn.LayerNorm(2 * d + pe), nn.Dropout(0.2), nn.Linear(2 * d + pe, n_labels))\n        self.drop = nn.Dropout(0.2)\n\n    def forward(self, f, slot, sidx, B, return_attn=False):\n        pe = self.pres_emb(slot)\n        pres = torch.zeros(B, pe.shape[1], device=f.device, dtype=f.dtype).index_add_(0, sidx, pe)\n        if self.pool_kind in ('xres', 'clsadd', 'xcodex'):\n            return self.pool(f, slot, sidx, B, pres)[0]\n        pooled, attn = self.pool(f, sidx, B, slot=slot, return_attn=return_attn)\n        if self.pool_kind in ('attn', 'xattn'):\n            x = torch.cat([pooled, pres.unsqueeze(1).expand(-1, self.k, -1)], -1)\n            x = self.drop(self.norm(x))\n            return (x * self.w).sum(-1) + self.b\n        return self.net(torch.cat([pooled, pres], 1))\n\nclass Net(nn.Module):\n\n    def __init__(self, enc, cond, n_meta=0, pool='mean_max', stem='native', n_slice=16):\n        super().__init__()\n        self.enc, self.cond = (enc, cond)\n        self.compress = DepthCompress(n_slice, 3) if stem == 'compress' else None\n        self.mixer = SlotDepthMixer(n_slice) if stem == 'mixer' else None\n        self.tokens = pool in ('xattn', 'xres', 'clsadd', 'xcodex')\n        D = enc.num_features\n        self.meta_mlp = nn.Sequential(nn.LayerNorm(n_meta), nn.Linear(n_meta, 128), nn.GELU(), nn.Linear(128, D)) if n_meta > 0 else None\n        self.readout = Readout(pool, D)\n        if cond == 'post':\n            self.slot_emb = nn.Embedding(N_SLOT_TYPES + 1, D, padding_idx=MASK_IDX)\n\n    def forward(self, im, slot, smeta, sidx, B, vm=None):\n        if self.mixer is not None:\n            im = self.mixer(im, slot, vm)\n        if self.compress is not None:\n            im = self.compress(im)\n        f = self.enc.forward_features(im, slot) if self.cond == 'token' else self.enc.forward_features(im)\n        if self.tokens:\n            inner = getattr(self.enc, 'vit', self.enc)\n            orig = getattr(self.enc, '_orig_prefix', getattr(inner, 'num_prefix_tokens', 1))\n            f = torch.cat([f[:, :1], f[:, orig:]], 1)\n        else:\n            f = self.enc.forward_head(f, pre_logits=True)\n            if f.dim() > 2:\n                f = f.flatten(1)\n        ex = (lambda v: v.unsqueeze(1)) if self.tokens else lambda v: v\n        if self.cond == 'post':\n            f = f + ex(self.slot_emb(slot))\n        if self.meta_mlp is not None and smeta.shape[1] > 0:\n            mt = self.meta_mlp(smeta)\n            f = torch.cat([f, mt.unsqueeze(1)], 1) if self.tokens else f + mt\n        return self.readout(f, slot, sidx, B)\nmodels=[]\n_A5_LOAD_RECEIPT=[]\n_a5_paths=[CKPT/f'm_f{i}.pt' for i in range(5)]\nif not all(p.is_file() for p in _a5_paths):\n    raise RuntimeError(f'A5 requires m_f0.pt..m_f4.pt; missing {[str(p) for p in _a5_paths if not p.is_file()]}')\n_a5_config=None\nfor ckpt_path in _a5_paths:\n    z=torch.load(ckpt_path,map_location='cpu',weights_only=False)\n    cfg=z['cfg'];fold=int(z['fold'])\n    if ckpt_path.name!=f'm_f{fold}.pt': raise RuntimeError('A5 fold/file identity mismatch')\n    fields={'backbone':cfg['backbone'],'cond':cfg['cond'],'pool':cfg['pool'],'img':int(cfg['img']),'n_slice':int(cfg.get('n_slice',16)),'n_meta':int(cfg.get('n_meta',0)),'meta':cfg.get('meta','none'),'stem':cfg.get('stem','native'),'norm':cfg.get('norm','none')}\n    if _a5_config is None:_a5_config=fields\n    if fields!=_a5_config:raise RuntimeError(f'A5 per-fold input/model contract differs: {ckpt_path}')\n    if fields['img']!=SIZE or fields['n_slice']!=N_SLICE or fields['n_meta']!=0:\n        raise RuntimeError(f'A5 checkpoint input contract incompatible with loader: {fields}')\n    if 'labels' in z and list(z['labels'])!=list(LABELS):raise RuntimeError('A5 label order differs')\n    _stem=fields['stem'];_in=3 if _stem=='compress' else fields['n_slice']\n    enc=timm.create_model(cfg['backbone'],pretrained=False,num_classes=0,in_chans=_in,**({'img_size':cfg['img']} if 'vit_' in cfg['backbone'] else {}))\n    if cfg['cond']=='token':enc=ViTSlotToken(enc,N_SLOT_TYPES)\n    m=Net(enc,cfg['cond'],cfg.get('n_meta',0),cfg['pool'],stem=_stem,n_slice=cfg.get('n_slice',16))\n    rsna_strict_load(m,z['state_dict'],f'A5 fold {fold}')\n    models.append(m.eval());_A5_LOAD_RECEIPT.append({'fold':fold,'file':str(ckpt_path),'sha256':rsna_sha(ckpt_path),'tensor_keys':len(z['state_dict']),'strict':True,'config':fields})\n    print(f'A5 strict-loaded {ckpt_path.name}; all encoder and head tensors present')\nCFG=dict(cfg)\nrsna_json('/kaggle/working/diagnostics/a5_loads.json',_A5_LOAD_RECEIPT)\n", globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(22, 'AMP_PREF = \'bf16\'  # BTKD precision retained, including T4 execution; no unvalidated AMP change.\n\ndef amp_for(dev):\n    if not str(dev).startswith(\'cuda\'):\n        return (torch.float32, False)\n    cc = torch.cuda.get_device_capability(dev)\n    if AMP_PREF == \'bf16\':\n        return (torch.bfloat16, True)\n    if AMP_PREF == \'fp16\':\n        return (torch.float16, True)\n    if AMP_PREF == \'fp32\':\n        return (torch.float32, False)\n    return (torch.bfloat16 if cc >= (8, 0) else torch.float16, True)\nAMP_DT, AMP_ON = amp_for(DEV)\nWORKERS = max(1, min(4, os.cpu_count() or 4))\nCHUNK = 48\nMICRO = 8\nimport copy as _speed_a5_copy\nfrom concurrent.futures import ThreadPoolExecutor as _SpeedA5Pool\nif torch.cuda.device_count() != 2:\n    raise RuntimeError(\'A5 optimized path requires exactly two GPUs\')\n_speed_a5_second = [_speed_a5_copy.deepcopy(m).to(\'cuda:1\').eval() for m in models]\n_speed_a5_models = [[m.to(\'cuda:0\').eval() for m in models], _speed_a5_second]\ndel _speed_a5_second\nmodels = _speed_a5_models[0]\n_speed_a5_pool = _SpeedA5Pool(max_workers=2)\nprint(f"device {DEV} | amp {str(AMP_DT).split(\'.\')[-1]} (on={AMP_ON}) | workers {WORKERS} | chunk {CHUNK} | micro {MICRO}")\n\ndef _norm_(im):\n    k = CFG.get(\'norm\', \'none\')\n    if k == \'zscore\':\n        m = (im > 0).float()\n        n = m.sum(dim=(1, 2, 3), keepdim=True).clamp(min=1.0)\n        mu = (im * m).sum(dim=(1, 2, 3), keepdim=True) / n\n        var = (((im - mu) * m) ** 2).sum(dim=(1, 2, 3), keepdim=True) / n\n        return (im - mu) / (var.sqrt() + 1e-06) * m\n    if k == \'imagenet\':\n        m = (im > 0).float()\n        return (im - 0.485) / 0.229 * m\n    return im\n\n@torch.no_grad()\ndef _speed_a5_micro(images, masks, *, dev, model_group, amp_dtype):\n    """Predict acquired rows only; NaN is a sentinel exclusively for absent inputs."""\n    active = np.flatnonzero((masks > 0).any(axis=1))\n    out = np.full((len(model_group), len(masks), len(LABELS)), np.nan, np.float32)\n    if not len(active):\n        return out\n    ims, slots, sidx, vms = [], [], [], []\n    for compact_index, original_index in enumerate(active):\n        present = np.flatnonzero(masks[original_index] > 0)\n        blk = images[original_index][present]\n        ims.append(torch.from_numpy(blk))\n        vms.append(torch.from_numpy(blk.reshape(blk.shape[0], blk.shape[1], -1).max(2) > 0))\n        slots.append(torch.from_numpy(present + 1).long())\n        sidx.append(torch.full((len(present),), compact_index, dtype=torch.long))\n    im = _norm_(torch.cat(ims).to(dev, non_blocking=True).float().div_(255.0))\n    sl = torch.cat(slots).to(dev)\n    si = torch.cat(sidx).to(dev)\n    vm = torch.cat(vms).to(dev)\n    sm = torch.zeros(len(sl), CFG.get(\'n_meta\', 0), device=dev)\n    per = torch.zeros(len(model_group), len(active), len(LABELS), device=dev, dtype=torch.float32)\n    def _a5_forward(enabled):\n        with torch.autocast(\'cuda\' if str(dev).startswith(\'cuda\') else \'cpu\',\n                            dtype=amp_dtype, enabled=enabled):\n            for fold_index, model in enumerate(model_group):\n                per[fold_index] = torch.sigmoid(model(im, sl, sm, si, len(active), vm=vm).float())\n        return per.cpu().numpy()\n    got = _a5_forward(AMP_ON)\n    if AMP_ON and not np.isfinite(got).all():\n        rsna_event(\'a5_fp16_nonfinite_retry_fp32\', rows=int(len(active)))\n        got = _a5_forward(False)\n    _bad = ~np.isfinite(got)\n    if _bad.any():\n        rsna_event(\'a5_nonfinite_neutral\', rows=int(_bad.any(axis=(0, 2)).sum()))\n        got = np.where(_bad, 0.5, got)\n    rsna_finite(got, \'A5 acquired-row probabilities\', probability=True)\n    out[:, active] = got\n    return out\n\n"""Two GPU owners, preserving every original eight-study microbatch."""\n\n\ndef speed_a5_ranges(n, micro=8, devices=2):\n    if n < 0 or micro < 1 or devices < 1:\n        raise ValueError((n, micro, devices))\n    return [[(start, min(start + micro, n))\n             for i, start in enumerate(range(0, n, micro)) if i % devices == d]\n            for d in range(devices)]\n\n\ndef speed_a5_predict(images, masks, models_by_device, executor, dtype=None):\n    """No shard-local ranks; output remains [fold, original study, finding]."""\n    count = len(masks)\n    assert len(images) == count\n    output = np.full((len(models_by_device[0]), count, len(LABELS)), np.nan, np.float32)\n    plans = speed_a5_ranges(count, MICRO, len(models_by_device))\n\n    def owner(device_index, ranges):\n        device = torch.device(f"cuda:{device_index}")\n        result = []\n        with torch.cuda.device(device):\n            for start, stop in ranges:\n                value = _speed_a5_micro(images[start:stop], masks[start:stop],\n                    dev=device, model_group=models_by_device[device_index],\n                    amp_dtype=AMP_DT if dtype is None else dtype)\n                result.append((start, stop, value))\n        return result\n\n    tasks = [executor.submit(owner, d, ranges) for d, ranges in enumerate(plans) if ranges]\n    for future in tasks:\n        for start, stop, value in future.result():\n            output[:, start:stop] = value\n    return output\n\ndef predict(images, masks):\n    return speed_a5_predict(images, masks, _speed_a5_models, _speed_a5_pool)\n\ndef _a5_validate_applicability(raw, eligible):\n    """Never confuse an absent input with failed inference on an acquired input."""\n    x = np.asarray(raw)\n    m = np.asarray(eligible, dtype=bool)\n    if x.ndim != 3 or x.shape[1] != len(m):\n        raise ValueError(\'A5 eligibility/raw shape mismatch\')\n    if m.any():\n        rsna_finite(x[:, m], \'A5 acquired-row complete tensor\', probability=True)\n    if (~m).any() and not np.isnan(x[:, ~m]).all():\n        raise RuntimeError(\'A5 absent rows must retain explicit NaN sentinels\')\n    return m\n\ndef _a5_rank_available(raw, eligible):\n    eligible = _a5_validate_applicability(raw, eligible)\n    result = np.full((len(eligible), raw.shape[-1]), .5, dtype=np.float64)\n    if eligible.any():\n        result[eligible] = 0.0\n        for fold in raw:\n            result[eligible] += rsna_rank01(fold[eligible])\n        result[eligible] /= raw.shape[0]\n    return result\n\ndef _a5_blend_available(parent, a5_values, eligible, weight):\n    """Parent-only on metadata-proven absence; no silent fallback on model error."""\n    base = np.asarray(parent, dtype=np.float64)\n    a5 = np.asarray(a5_values, dtype=np.float64)\n    mask = np.asarray(eligible, dtype=bool)\n    if base.shape != a5.shape or mask.shape != (len(base),):\n        raise ValueError(\'A5 merge shape mismatch\')\n    rsna_finite(base, \'A5 parent probabilities\', probability=True)\n    if not mask.any() or weight == 0:\n        return base.copy()\n    rsna_finite(a5[mask], \'A5 applicable ranks\', probability=True)\n    base_rank = pd.DataFrame(base).rank(method=\'average\', pct=True).to_numpy()\n    a5_rank = pd.DataFrame(a5[mask]).rank(method=\'average\', pct=True).to_numpy()\n    result = base_rank.copy()\n    result[mask] = (1-weight)*base_rank[mask] + weight*a5_rank\n    rsna_finite(result, \'A5 applicability-aware merge\', probability=True)\n    return result\n\n# Macro ROC-AUC depends on ordering, so combine fold orderings rather\n# than allowing a fold\'s probability scale to dominate the mean.\npreds = np.full((len(models), len(studies), len(LABELS)), np.nan, np.float32)\n_a5_eligible = np.zeros(len(studies), dtype=bool)\nt0, done = (time.time(), 0)\ndef submit_study_block(executor, start):\n    """Keep one bounded CPU decode block ahead of GPU inference."""\n    block = studies[start:start + CHUNK]\n    futures = [\n        executor.submit(\n            build_study,\n            (index, study, by.get(study, [])),\n        )\n        for index, study in enumerate(block)\n    ]\n    return start, block, futures\n\n# CPU decode threads avoid forking a process after CUDA/model initialization.\nwith A5DecodePool(max_workers=WORKERS) as ex:\n    pending_block = submit_study_block(ex, 0)\n    while pending_block is not None:\n        rsna_deadline(\'A5 full-cohort inference\')\n        c0, block, futs = pending_block\n        next_start = c0 + len(block)\n        pending_block = (\n            submit_study_block(ex, next_start)\n            if next_start < len(studies)\n            else None\n        )\n        imgs = np.zeros((len(block), N_SLOT, N_SLICE, SIZE, SIZE), np.uint8)\n        msks = np.zeros((len(block), N_SLOT), np.uint8)\n        _fut_index = {f: i for i, f in enumerate(futs)}\n        for f in as_completed(futs):\n            try:\n                i, a, k = f.result()\n                imgs[i], msks[i] = (a, k)\n            except Exception as e:\n                i = _fut_index[f]\n                rsna_event(\'a5_study_failed\', study=str(block[i]), error=f\'{type(e).__name__}: {str(e)[:500]}\')\n                print(f\'[a5] {block[i]}: build failed, treated as A5-absent (flagged): {type(e).__name__}: {str(e)[:200]}\', flush=True)\n                imgs[i] = 0; msks[i] = 0\n        _a5_eligible[c0:c0 + len(block)] = (msks > 0).any(axis=1)\n        preds[:, c0:c0 + len(block)] = predict(imgs, msks)\n        _a5_validate_applicability(preds[:, c0:c0 + len(block)], _a5_eligible[c0:c0 + len(block)])\n        done += len(block)\n        el = time.time() - t0\n        print(f\'  {done:,}/{len(studies):,}  {el / 60:.1f}m  eta {el / done * (len(studies) - done) / 60:.1f}m\', flush=True)\n        del imgs, msks\n        gc.collect()\nprint(f\'\\ninference done in {(time.time() - t0) / 60:.1f} min\')\nnp.savez_compressed(\'/kaggle/working/speed_a5_raw.npz\',\n    study_uids=np.asarray(studies), raw_probabilities=preds, applicable=_a5_eligible)\n_speed_a5_pool.shutdown(wait=True)\ndel _speed_a5_models, models, m, enc, z\ngc.collect()\nfor _speed_device in range(2):\n    with torch.cuda.device(_speed_device):\n        torch.cuda.empty_cache()\nA5_W = 0.52\nA5_LABELS = list(LABELS)\n_a5_ok = _a5_validate_applicability(preds, _a5_eligible)\n_a5_rank_mean = _a5_rank_available(preds, _a5_eligible)\nA5_PREDS = dict(zip(sub_df[\'StudyInstanceUID\'].astype(str), _a5_rank_mean.astype(np.float32)))\nA5_AVAILABLE = dict(zip(sub_df[\'StudyInstanceUID\'].astype(str), _a5_eligible.tolist()))\nrsna_save_predictions(\'a5_rank_mean\', studies, _a5_rank_mean, LABELS)\nrsna_json(\'/kaggle/working/diagnostics/a5_applicability.json\', {\n    \'unavailable_uids\':[u for u, ok in A5_AVAILABLE.items() if not ok],\n    \'available_count\':int(_a5_eligible.sum()), \'model_failure_fallback_allowed\':False,\n    \'rank_placeholder_for_unavailable\':0.5, \'placeholder_used_in_blend\':False})\nrsna_phase(\'a5\', \'COMPLETE\', applicable=int(_a5_eligible.sum()), absent=int((~_a5_eligible).sum()))\nfor _a5k, _a5v in _A5_SAVED.items():\n    globals()[_a5k] = _a5v\ndel _A5_SAVED, _a5k, _a5v\n', globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(23, "_a5_sub = pd.read_csv('/kaggle/working/_pipeline_stage.csv', dtype={'StudyInstanceUID':str})\nassert _a5_sub.columns.tolist()[1:] == A5_LABELS, 'submission schema drift'\nif A5_W > 0:\n    _a5_uid_order = _a5_sub.StudyInstanceUID.astype(str).tolist()\n    _a5_ours = np.stack([A5_PREDS[u] for u in _a5_uid_order])\n    _a5_presence = np.asarray([A5_AVAILABLE[u] for u in _a5_uid_order], dtype=bool)\n    _a5_sub[A5_LABELS] = _a5_blend_available(\n        _a5_sub[A5_LABELS].to_numpy(), _a5_ours, _a5_presence, A5_W)\n    assert np.isfinite(_a5_sub[A5_LABELS].to_numpy()).all()\n    _a5_sub.to_csv('/kaggle/working/_pipeline_stage.csv', index=False)\n", globals())


## Stage 3 — RadImageNet heads

One RadImageNet encoder feeds the E10/E13/E11 layouts. Version 7 uses the scored
0.942 routing: main alpha 0.55 and second E13 pass alpha 0.20.


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(25, 'from __future__ import annotations\nimport base64 as _rad_b64\nimport zlib as _rad_zlib\n_RAD_CAL_PAYLOAD = \'eNrtmk1vI8cRhv9KsJdcKKE/q6tzc4z4ZCMBcjQWhrCRDSG2ZEjaIEGQ/57n7RlRQ3KG4jqLJAcDS4o709NdXR9vvVU9/3z30+3N/bvffRuuawgxlm7eq2ePeffrpV8v/V9euvLraD3k6ilW6zn126vYd+U6eKmx9RiaF7OSx+X1weE67K7SdSgpVSauuaecUxr3rtp1LK0HyyV7y9Gmy/E6pBhTL61Zt2gWx+WtSew6JmOoM5AHutXp+ro8+ZrlsnvJOfDdfRL+Kl4zd2bqllNmga7rvrva2OzGNFuynGxpmnxrS/069hCtpt6T1dLLOQVsTbJ1fWunG2qXAX8NiU+7VK8rdqvNU89uwQwjpWyeLdTCnVy84ua9pthSjznHXLjDpZxbLe4WPRAQCT+LrSVcGGdLzNX0XKhesC2eV5uZ67lQPDlmSzUhS2+61ELtoTOyWGjdPudc73fvnj7c/Hg7ElpyzYXjdwIZ32m7X3INZ0crtvtc833ua6fyoUYUGf4H13rJpX+GK5aa5VbeWO9ye/03bLi97i8SpaSCl49nc4gdxI2p1dZyVmQnfL56jBH/j70GqSq6dyMROLHQGvCpcSnkYsyeohNErnVjbamVjs47wOpBa7BAsa61SXulDfSIwAbAkQqDmSK38WwImKK1kFJ3d11CpFqR1mPHlj1pWVAHeDfyU2hk0vGogz0GqCBbKmFMx9xIWEAbQ8I4PUvmAplQCeuij7GNXGMk3yhBsFIfEnvVE6HFYMHDtFuLzKwpya9gisaZduAkcyAvmw1ROjmd7OnBYytpOBqJjTyK4KzT0Pi0tQJYslc0nGqcNZV7dEaRvfcSewg9h4p41iwO+4CWMkNG1326VBmHFUoB7VoakzE+JAtYN/NknS9lLJ+9S5qR56Q2kLA4qgTuplGNmUJAkbXiGbrU0HCIsgtYaGOUVXbayNeykLfpGv8lulBfgiFEmx61xm3LTlrOPoa51Ng7IyJiT4tWxrE0ZmnJRnhawZsyLgjXIC9MRu0tRAgImkJ7bUyHKk1eUgIewtRjXGDJqO1mNJrHfNUty1HRW2SSoV3FAVIA//hrUXKANxhbRbEkgqAFKiCC43qOEXsPTaKtKCdkqx3/koMYsuP+mh4HrHoQtqSIw/h4DM7EpRL5zRirAcHiUDj2SUlr9fFAsElHBX/zWgJCQlw+83Qksw8Pt9+Ty0hmOBQBh9XQAr7fxjRQ2IkGTT/w8cAiBKwRc1jwcMh+WKtMgFShNaDGJWRAechNSOMUldXTynNIUPAaEKKkCv1cLFzxaowBOmX8vU8Pj1voWa5JTPFcGN4QXm+PpZPjA8QQYYp9Qab9rZiIAuLX8AA8f/jX4kHgI+BXCtBEeOTFvNPapIyC92YAEOmHypJcCSiFOpQi712M73IL8KiBX8Hz62pDNsAH8MK/rMR+tNSRIRJwAkIIY8Rn/fD2kB2V9I7BMX+KSJ/XJtNAbwFglmQZbu/90CZcMhAeZKudKAvlSBLwNwP1aA88BYpPdJRkbADGeBzWN2IOvySVELxyU0Lx0JHQHsFpEQRsCB9iO1qT3IOHGUjMDqeczX4BiIbvCjkiLD6t6G239p+gAML1KQuAdaLLdidDV6Y6uPR+9+3LT8KrlYgzAsjg7ok+VJakimvgAjn5qUFmtTGJKXGxSadg2UuLkWok4EvGJ0Hdsr+DmpVlGsUMZkxtuTI8vDAVaIsnSIDbK0DhVSpAFlu4iRtgrLYWRaArADOcFDAfErEdwBS8dWpNqHupW3o7nIukTmxlmASq4L/51ESrznqJTSkgCzslVSMncekb8659ljeyYttxK0oX9k50n3h2kDqoapToWt8S9/yO3tTXyteLu+19rkPViKkIblLnzJhJUhYENUJCtdfWKkEFe9WTsWin6azpqJFIfEwNyLNes+PZXFV3h+tIOXjb5mzIRnCLaWKuhnPtjsCVPAOwdkhQhBSMfWLVruTgwEM/WXt1FYgv5AN4IsyBiHrOSscCBjIomksgZCPFn9grqDrE9UXDQCSPfs5RXyeuQl3gwcEIdn5JzADhpG34s4uF5NQ24eh1GWISkEmYA6iFiNezYAbkKNGJhIk+l9XNXK3r7AgLT4YnRUuHicJS0NRLLG0KDrF1IbkaMzj2MoeKzAH/kTo9KsnWJe8gDZUuxs1iPi0CayABMdLIT4qQCbfENfCiAsujIsj9KMUQ1kXwXUBFZspHZm9Zj/dB/SrVy1qOJg9ApKAlmSQNpr6SjibnNgVoqZQCr8gllgsTiFAOCFSeDcYWuKOChZQXi5dPxouF5Oo6ogVTaxDwnVE8KfQFI6Zomwgb/oI4VG2Ae1MdsTQCwQuZp5ZROZPcdudWHrZRfwe4cPE1cv9L/FDuwRwGNYOttGllMRItS1K3sFDQDAwQMvgpfIzyIeQj2yTFOhomGLsBVPtAASGLqiXKnChASW/4ILUTGlftQlUVl8FDjifbKYIzLKXmco4anNPK6ZVl8MhVBs+D5YgTo8HXuIEDQDJF4wHEYL5PvEExn4PIt7x0JumDeUgjxHeFIZDt+6xN5mALCc6pMjTZ7oy8EomCUA09MTRvvuCLxyFC+sEWCGiqjE+HIAJ4g39Bi5sadSN9MJZaFWLbpubBNBvlmwrPAONJWS1DpaIgbyKuZSbKg7qZzgNKsqnwIX8RAOvINdgfRg0jPNJBWE+5xAQ9qDaAPh4nKInagjAL9otj26U+sAnbUERTF0QYjHV8j8OEA+mohtH9SLLuoUIh6uAWHI6yAw54uEsqL6zufMAAt3yW+6xSFLmKwEmYgIPlXHYbPAo3AvKovCk/0KR5m3fmKkpFz2W3ng58h7KzqVVHqVYUpfGlAJHvQ2dN7QKmXoQITggHUz9HGZhSty5smYHeSL4pFdvbXlXZcSJV7GCHKPpsePEmVAOm41WxbuW3Y+qkrhYhisbrSBqbGiGdAkBwd5gzNehxGUVhzu5JCKa2VAyL2pfYgYgUtcTwu3RKZzO2JublhfCDbO1Qqypu8SdTWO0501Z6iAAH68g21mECvjX8bZZ7yvpViqq9BlHCnhj7DFuiXm8qEwgrAkK9hpfbuU/nN/i4l7I/qQEnpR4qVHUxgDjblWuM2UZFhOpzp+apb8i4ua8gnyHrqPMHs839gK7gaaYDDtILqNTWOF8kxVYdMxlVDwE68/F8DR2CgpCSkkzkEvKY3x9GoRqjpn8ZLj62TyEoRVV1dxrV2GtNOMosPAjqSC6pm6yRKRJFM5wnixbhh6uLl9FdjElsUvX7WxR61T1gGohOkQaej27MxiR+DQjAndQ5SgL4Yb+VMKT2UncGK4se5hfeR7Jr6nygbTJcPGK7QSpXOoBJkuMXlFTYQX0aRsObYEQn22BNrDQdKwmZ1DDaZNcANxumdkGs3pZIJcaCcxBuPSlGziTf+UNZIgqmLMTWS1/XYNBxFowH5FchZb7uT5EQqervUK+Rc15pP55mBpKra6Wju5MCbZwiKOSS/HFuZyVRBFeeIoVrc35oNnVt1ARpKDaHF2BO1z3DLILOrOVZdhIG52ojyFxsVa1k6Bq+sOS7AA0upIqpiCt9Om8+1SsIoI4/ZYFKZq9tdwnhCzrVRpoCxpqSo+0ua1DNLCOMc3X19vGSZZ9znfEAMTUpymSGqYW/kpXU6FWV6+pah9OGLveaTkopOV1d3U9oWfAhMiyK46fRgPeL9LTSDAtUjj5cFN4D8S5zmaCjmaTjJ/Kvxb3MaFrHVI27QS2vRfcMCmeqcUXY+NX6652okyj19OEC3SaFrdWyeyJMxo7gPrANn17GjQ5RgtqvregNjzr1hYOOxATeos4b/IItGRxEFZC4aNruVL1ZbGx8ojpE5moLiGNavazB+baxPoXr/gK56zjI1VGbjtG8nA3Xg3SpFl2gtqzqsM/oGgb5Axj4w+XD1g48MBFrMAnR+TRxMci1+8h+uCC1e1nmC7XEWhf2zEdIw5SsCe3Tcc18XibsU/PKhJhd7a380s67RLEvQQWUw9KKjmps7qefVeub/IZQoLbK0qxnHRFP7qlOy8BURC7Fy2LDPk7N1F1VEm1lrZbSmSWVSoNk63DQXgvIohAjYcjXU7b/zI8u7RVvPiRChSqVjkiU6YQjrT1ImVAAN/WoKEp62V2aQGAdAuQKhSW5qouyIdQ4jNe5NS6I8v10hLIeqIIMqplDn71o0dYl4fEFkRZ4LmhFJNUSQqZCNlBT2fIWnKzJ9XWwG48bOzGZWEIffcVx/q23xAziBV83gf1cE4/+mvI8/EFV3QVmPW0a6rB7ZD314ploTllOSlFYqX3X4u7TJg6jmxLRWxit1FaPtArKoHfHeb3jwPmNGDqfvS+Iv9Ns14Vv6p9rfyft+HGiFqmJIQSU++rlbegPBqDumli9zoOGSptec8O6GAWqtaAgkOgqZ3P1WhQQ08aj2KGONuEIqdZetzuLFB6qQniid8HkZTnlyGsPGnA4lY5Qu1456WnbokG9IupwU0NoPhHTwRRUqynZU0HObV8QUyZuo/lb6tFhsdxb7bCoU9qWe8vLxBwj2laVzgS+bIZGMfee1Qaldl87gBZ5jjr3Y4rq2Xaf3LatohHwxzbqBKtv4d8bHnh1eUqfVRx07jc4zfzySrj8IGV9NMFX4mBKrq6FXc4Jz154/3737u7++fbxw+3Pz9N7eg0X0mHCeHfHbHrNhrCIpdXRA/LRRC4WR9sU/ZqJB46XJsJouwU1rPT+il6uKHqNAdbJ2InUJp0wVWBV7w8NuldU7uMyajZqh2N6UfeoM6ym1zLGizF6T6AnNQZiHO/KZMijZMOV2/yKQFKv0TSXq0Hb5teE9F4HLp50QKJ3OX64edZ7ie+++PLrd7t339z+5e7mx9/88Qt+f82dx5f//Omr6e8fvv/+49Pdwz0/f3/z19vH3z7x68uH++fpKhP+/Pjw/PDh4cfv+Hz86f5Jk99/93T7eHersfff/fnmh/H3y4fH8feLv9/x96ub5+l7vq9f0wj9msf8+HH6fhnDr3kMvzRGG3p8+PizVt3vaXy/ysjox5sPzx8fbxn+7cuWv7m9v3v68PFpsfH9pcWwLc1oyEI3f/7H/cPf7p7vnhZ6ev/+X/8GYIe3xg==\'\n# BTKD Rad graph: preserve its E13/E11 calibrated feature contract; remove unused twin work.\n#\n# The pinned reference Rad family is fused with correct-contract E13, then\n# the same E13 heads run on the E11 layout at 0.15. No twin/legacy wrapper\n# follows it, matching the branch that produced V48\'s visible submission.\n\nimport contextlib as _rad_contextlib\nimport gc as _rad_gc\nimport hashlib as _rad_hashlib\nimport json as _rad_json\nimport os as _rad_os\nimport re as _rad_re\nimport time as _rad_time\nfrom concurrent.futures import ThreadPoolExecutor as _RadThreadPool\nfrom pathlib import Path as _RadPath\n\nimport numpy as _rad_np\nimport pandas as _rad_pd\nimport pydicom as _rad_pydicom\nimport torch as _rad_torch\nimport torch.nn as _rad_nn\nimport torch.nn.functional as _rad_F\nfrom torchvision.models import resnet50 as _rad_resnet50\n\n_RAD_LABELS = [\n    \'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n    \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s",\n    \'Contusion\', \'Fracture\',\n]\n_RAD_ALPHA = 0.55\n_RAD_EXCLUDE = ("Baker\'s", \'Fracture\')\n_RAD_HEADS_SHA256 = \'54f657826b3458a7ba3d462e198ba380732f2b136246182312704929874a9a2c\'\n_RAD_REFERENCE_HEADS_SHA256 = \'0f465649799ecfbccaac1767844639e7ced44e1bc9babde6e4bac7c5d9b89eaa\'\n_RAD_ENCODER_SHA256 = \'08629f7e7bd3e29b8ee9522ca3f65ce4d010a7ddf74f0ea3c7e3f3d0bbab0734\'\n_RAD_E13_HEADS_SHA256 = \'ad9f19af73bfdf4e49263c0e45060dc3cb239e1195039b26dc8c0a3a6bcd1a8a\'\n_RAD_E13_MEMBER_WEIGHT = 0.50\n_RAD_V48_SECOND_ALPHA = 0.20\n_RAD_TWIN_ALT_WEIGHT = 0.500001\n_RAD_TOKEN_DIM, _RAD_HEAD_DIM = 2048, 512\n\n_RAD_E11_SLOTS = [\n    (\'SAG_NOFS\', \'Sagittal\', None, False),\n    (\'COR_NOFS\', \'Coronal\', None, False),\n    (\'AX_NOFS\', \'Axial\', None, False),\n    (\'SAG_FS\', \'Sagittal\', None, True),\n]\n_RAD_E11_CROP_MM = 130.0\n_RAD_E11_CACHE_SLICES = 8\n_RAD_E11_IMG = 224\n\n_RAD_E13_SLOTS = [\n    (\'SAG_FS\', \'Sagittal\', None, True),\n    (\'COR_FS\', \'Coronal\', None, True),\n    (\'AX_FS\', \'Axial\', None, True),\n    (\'SAG_NOFS\', \'Sagittal\', None, False),\n]\n_RAD_E13_CROP_MM = 130.0\n_RAD_E13_CACHE_SLICES = 8\n_RAD_E13_IMG = 224\n\n# Our independently trained five-fold family.  Its preprocessing and estimator\n# are preserved from V35: native DICOM geometry/fat-sat handling and a mean of\n# per-fold percentile ranks (rather than v15\'s rank of the probability mean).\n_OUR_N_SLOT, _OUR_N_SLICE, _OUR_IMG = 3, 8, 224\n\n# Exact V40/E10 test representation: three fat-suppressed planes, eight\n# acquired slices per plane, full frame, legacy ordering/laterality/fill.\nSLOTS = [\n    (\'SAG_FS\', \'Sagittal\', None, True),\n    (\'COR_FS\', \'Coronal\', None, True),\n    (\'AX_FS\', \'Axial\', None, True),\n]\nN_SLOT = len(SLOTS)\nCACHE_SLICES = 8\nIMG = CACHE_IMG = 224\nCROP_MM = 10_000.0\nSLICE_BAND = (0.2, 0.8)\nRULES = dict(RULES_LEGACY)\nTIME_BUDGET = 8.0 * 3600\n\n\ndef _rad_log(message):\n    print(f\'[Rad-dual5] {message}\', flush=True)\n\n\ndef _rad_sha256(path, chunk=8 << 20):\n    digest = _rad_hashlib.sha256()\n    with open(path, \'rb\') as handle:\n        for block in iter(lambda: handle.read(chunk), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _rad_find_file(name, expected_sha=None, explicit_env=None):\n    if explicit_env and _rad_os.environ.get(explicit_env):\n        candidates = [_RadPath(_rad_os.environ[explicit_env])]\n    else:\n        by_name = {\n            \'ResNet50.pt\': [\n                \'/kaggle/input/datasets/marwanmath/resnet-50-radimagenet-marwan/ResNet50.pt\',\n                \'/kaggle/input/resnet-50-radimagenet-marwan/ResNet50.pt\',\n            ],\n            \'v52_radimagenet_heads.pt\': [\n                \'/kaggle/input/datasets/prvsiyan/rsna-knee-v52-radimagenet-heads-20260812/v52_radimagenet_heads.pt\',\n                \'/kaggle/input/rsna-knee-v52-radimagenet-heads-20260812/v52_radimagenet_heads.pt\',\n                \'/kaggle/input/datasets/antoinegg1/rsna-knee-e9-radimagenet-heads-v15/v52_radimagenet_heads.pt\',\n                \'/kaggle/input/rsna-knee-e9-radimagenet-heads-v15/v52_radimagenet_heads.pt\',\n            ],\n            \'v52_e11_heads.pt\': [\n                \'/kaggle/input/notebooks/sofiaanjenje/rsna-knee-e13-train/rsna_rad_e11/v52_e11_heads.pt\',\n                \'/kaggle/input/rsna-knee-e13-train/rsna_rad_e11/v52_e11_heads.pt\',\n                \'/kaggle/input/notebooks/sofiaanjenje/rsna-knee-e11-train/rsna_rad_e11/v52_e11_heads.pt\',\n                \'/kaggle/input/rsna-knee-e11-train/rsna_rad_e11/v52_e11_heads.pt\',\n                \'/kaggle/input/datasets/antoinegg1/rsna-knee-e11-diverse-heads-v20/v52_e11_heads.pt\',\n                \'/kaggle/input/rsna-knee-e11-diverse-heads-v20/v52_e11_heads.pt\',\n            ],\n        }\n        if name not in by_name:\n            raise FileNotFoundError(f\'unpinned Rad artifact name: {name}\')\n        candidates = [_RadPath(path) for path in by_name[name]]\n    existing = [path for path in candidates if path.is_file()]\n    valid = [\n        path for path in existing\n        if expected_sha is None or _rad_sha256(path) == expected_sha\n    ]\n    if len(valid) != 1:\n        raise RuntimeError(\n            f\'expected one verified Rad artifact {name}, found {valid}; existing={existing}\'\n        )\n    return valid[0]\n\n\nclass _RadEncoder(_rad_nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.backbone = _rad_nn.Sequential(\n            *list(_rad_resnet50(weights=None).children())[:-2]\n        )\n\n    def forward(self, image):\n        return self.backbone(image).mean(dim=(2, 3))\n\n\nclass _RadHead(_rad_nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.project = _rad_nn.Sequential(\n            _rad_nn.LayerNorm(_RAD_TOKEN_DIM),\n            _rad_nn.Linear(_RAD_TOKEN_DIM, _RAD_HEAD_DIM),\n            _rad_nn.GELU(),\n        )\n        self.plane = _rad_nn.Parameter(_rad_torch.randn(N_SLOT, _RAD_HEAD_DIM) * .01)\n        self.position = _rad_nn.Parameter(_rad_torch.randn(CACHE_SLICES, _RAD_HEAD_DIM) * .01)\n        self.query = _rad_nn.Parameter(_rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * .02)\n        self.attn = _rad_nn.MultiheadAttention(\n            _RAD_HEAD_DIM, 8, dropout=.10, batch_first=True\n        )\n        self.fuse = _rad_nn.Sequential(\n            _rad_nn.LayerNorm(_RAD_HEAD_DIM * 4),\n            _rad_nn.Linear(_RAD_HEAD_DIM * 4, _RAD_HEAD_DIM),\n            _rad_nn.GELU(),\n            _rad_nn.Dropout(.15),\n        )\n        self.weight = _rad_nn.Parameter(\n            _rad_torch.randn(len(_RAD_LABELS), _RAD_HEAD_DIM) * .02\n        )\n        self.bias = _rad_nn.Parameter(_rad_torch.zeros(len(_RAD_LABELS)))\n\n    def forward(self, feature, mask):\n        token = self.project(feature.float())\n        token = token.view(len(token), self.plane.shape[0], self.position.shape[0], self.plane.shape[-1])\n        token = token + self.plane[None, :, None] + self.position[None, None]\n        token = token.flatten(1, 2)\n        key_padding = mask <= 0\n        all_empty = key_padding.all(1)\n        if all_empty.any():\n            key_padding = key_padding.clone()\n            key_padding[all_empty, 0] = False\n        query = self.query.unsqueeze(0).expand(len(token), -1, -1)\n        attended = query + self.attn(\n            query, token, token, key_padding_mask=key_padding, need_weights=False\n        )[0]\n        denominator = mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)\n        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / denominator\n        mean = mean.expand(-1, len(_RAD_LABELS), -1)\n        fused = self.fuse(_rad_torch.cat(\n            [attended, mean, _rad_torch.abs(attended - mean), attended * mean], dim=-1\n        ))\n        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias\n\n\ndef _rad_load_public_heads(device, expected_sha):\n    heads_path = _rad_find_file(\'v52_radimagenet_heads.pt\', expected_sha)\n    payload = _rad_torch.load(heads_path, map_location=\'cpu\', weights_only=True)\n    expected = {\n        \'version\': \'v52-radimagenet-resnet50-official-1\',\n        \'targets\': _RAD_LABELS,\n        \'encoder_sha256\': _RAD_ENCODER_SHA256,\n        \'encoder_source_commit\': \'0ce16f7375db4236e646829d1eca61cdb4282133\',\n        \'img\': 224,\n        \'slices_per_plane\': 8,\n        \'feature\': \'global_average_pool\',\n    }\n    for key, value in expected.items():\n        if payload.get(key) != value:\n            raise RuntimeError(f\'public-v15 head contract drift for {key}\')\n    folds = payload.get(\'folds\')\n    if not isinstance(folds, list) or len(folds) != 5:\n        raise RuntimeError(\'public-v15 bundle requires exactly five heads\')\n    if sorted(int(record.get(\'fold\', -1)) for record in folds) != list(range(5)):\n        raise RuntimeError(\'public-v15 fold identity drift\')\n    heads = []\n    for record in folds:\n        head = _RadHead().to(device).eval()\n        head.load_state_dict(record[\'state_dict\'], strict=True)\n        heads.append(head)\n    return heads, str(heads_path)\n\n\ndef _rad_load_e13_heads(device):\n    # V48 used an unqualified filename shared by E11 and E13. Resolve the\n    # intended E13 bundle by content and validate its complete pixel contract.\n    heads_path = _rad_find_file(\'v52_e11_heads.pt\', _RAD_E13_HEADS_SHA256)\n    payload = _rad_torch.load(heads_path, map_location=\'cpu\', weights_only=False)\n    expected = {\n        \'version\': \'e11-radimagenet-resnet50-diverse-1\',\n        \'targets\': _RAD_LABELS,\n        \'encoder_sha256\': _RAD_ENCODER_SHA256,\n        \'slots\': [list(slot) for slot in _RAD_E13_SLOTS],\n        \'crop_mm\': _RAD_E13_CROP_MM,\n        \'img\': _RAD_E13_IMG,\n        \'slices_per_plane\': _RAD_E13_CACHE_SLICES,\n        \'feature\': \'global_average_pool\',\n    }\n    for key, value in expected.items():\n        if payload.get(key) != value:\n            raise RuntimeError(f\'E13 head contract drift for {key}\')\n    folds = payload.get(\'folds\')\n    if not isinstance(folds, list) or len(folds) != 5:\n        raise RuntimeError(\'E13 bundle requires exactly five heads\')\n    if sorted(int(record.get(\'fold\', -1)) for record in folds) != list(range(5)):\n        raise RuntimeError(\'E13 fold identity drift\')\n    heads = []\n    for record in folds:\n        head = _RadHead().to(device).eval()\n        head.load_state_dict(record[\'state_dict\'], strict=True)\n        heads.append(head)\n    return heads, str(heads_path)\n\n\ndef _rad_load_models(device):\n    """Two persistent encoder replicas; unused alternative heads are not loaded."""\n    import copy\n    path = _rad_find_file(\'ResNet50.pt\', _RAD_ENCODER_SHA256, explicit_env=\'RSNA_RAD_WEIGHT_PATH\')\n    encoder = _RadEncoder().eval()\n    rsna_strict_load(encoder, _rad_torch.load(path,map_location=\'cpu\',weights_only=True),\'Rad encoder\')\n    if sum(p.numel() for p in encoder.parameters())!=23508032:\n        raise RuntimeError(\'Rad encoder parameter count changed\')\n    replicas=[encoder.to(\'cuda:0\'), copy.deepcopy(encoder).to(\'cuda:1\')]\n    for m in replicas:\n        m.requires_grad_(False)\n    heads,hpath=_rad_load_public_heads(device,_RAD_REFERENCE_HEADS_SHA256)\n    return replicas,heads,str(path),hpath\n\n\n@_rad_torch.inference_mode()\ndef _rad_encode(encoder, pixels, slot_mask, device):\n    """Preserve parent\'s <=96 images/device chunks without DataParallel replication."""\n    n,slots,slices,height,width=pixels.shape\n    token_mask=_rad_np.repeat(slot_mask[:,:,None],slices,axis=2).reshape(n,-1)\n    _rad_empty=int((token_mask.sum(1)==0).sum())\n    if _rad_empty:\n        print(f\'[rad] {_rad_empty} study(ies) with no slot evidence encoded as all-zero rows (parent policy)\', flush=True)\n    valid=_rad_np.flatnonzero(token_mask.reshape(-1)>0)\n    flat=pixels.reshape(-1,height,width)\n    features=_rad_np.zeros((n,slots*slices,_RAD_TOKEN_DIM),_rad_np.float16)\n    def run(indices,d):\n        dev=_rad_torch.device(f\'cuda:{d}\')\n        def _rad_forward(_amp):\n            with _rad_torch.cuda.device(dev), _rad_torch.inference_mode(), _rad_torch.autocast(\'cuda\',dtype=_rad_torch.float16,enabled=_amp):\n                image=_rad_torch.from_numpy(flat[indices]).to(dev).float().div_(127.5).sub_(1.)\n                image=image.unsqueeze(1).expand(-1,3,-1,-1).contiguous()\n                return encoder[d](image).float().cpu().numpy()\n        value=_rad_forward(True)\n        half=value.astype(_rad_np.float16)\n        if not (_rad_np.isfinite(value).all() and _rad_np.isfinite(half).all()):\n            rsna_event(\'rad_fp16_nonfinite_retry_fp32\',tokens=int(len(indices)))\n            value=_rad_forward(False); half=_rad_np.clip(value,-65000,65000).astype(_rad_np.float16)\n        if not _rad_np.isfinite(half).all():\n            rsna_event(\'rad_nonfinite_zeroed\',tokens=int((~_rad_np.isfinite(half).reshape(len(half),-1).all(axis=1)).sum()))\n            half=_rad_np.where(_rad_np.isfinite(half),half,0).astype(_rad_np.float16)\n        return indices,half\n    with _RadThreadPool(max_workers=2) as pool:\n        for start in range(0,len(valid),192):\n            rsna_deadline(\'Rad full-cohort inference\')\n            block=valid[start:start+192]; cut=(len(block)+1)//2\n            jobs=[pool.submit(run,ii,d) for d,ii in enumerate((block[:cut],block[cut:])) if len(ii)]\n            for job in jobs:\n                indices,value=job.result(); features.reshape(-1,_RAD_TOKEN_DIM)[indices]=value\n    return features,token_mask.astype(_rad_np.float32)\n\n\n@_rad_torch.inference_mode()\ndef _rad_predict_head(head, features, masks, device, batch=64):\n    predictions = []\n    for start in range(0, len(features), batch):\n        image = _rad_torch.from_numpy(features[start:start + batch]).to(device)\n        mask = _rad_torch.from_numpy(masks[start:start + batch]).to(device)\n        def _head_forward(_amp):\n            amp = (_rad_torch.autocast(\'cuda\', enabled=_amp)\n                   if device.type == \'cuda\' else _rad_contextlib.nullcontext())\n            with amp:\n                return _rad_torch.sigmoid(head(image if _amp else image.float(), mask)).float().cpu()\n        block = _head_forward(True)\n        if not bool(_rad_torch.isfinite(block).all()):\n            rsna_event(\'rad_head_fp16_nonfinite_retry_fp32\', rows=int(len(image)))\n            block = _head_forward(False)\n        if not bool(_rad_torch.isfinite(block).all()):\n            rsna_event(\'rad_head_nonfinite_neutral\', rows=int((~_rad_torch.isfinite(block)).any(dim=1).sum()))\n            block = _rad_torch.where(_rad_torch.isfinite(block), block, _rad_torch.full_like(block, 0.5))\n        predictions.append(block)\n    return _rad_torch.cat(predictions).numpy()\n\n\ndef _rad_rank_columns(values):\n    return rsna_rankpct(values)\n\n\ndef _rad_validate(frame, expected_ids):\n    if frame.columns.tolist() != [\'StudyInstanceUID\', *_RAD_LABELS]:\n        raise RuntimeError(\'V36 submission schema drift\')\n    ids = frame[\'StudyInstanceUID\'].astype(str).tolist()\n    if ids != list(map(str, expected_ids)) or len(ids) != len(set(ids)):\n        raise RuntimeError(\'V36 submission study identity/order drift\')\n    values = frame[_RAD_LABELS].to_numpy(_rad_np.float64)\n    if not _rad_np.isfinite(values).all() or values.min() < 0 or values.max() > 1:\n        rsna_event(\'rad_invalid_values_repaired\', nonfinite=int((~_rad_np.isfinite(values)).sum()), out_of_range=int(((values < 0) | (values > 1)).sum()))\n        frame[_RAD_LABELS] = _rad_np.clip(_rad_np.where(_rad_np.isfinite(values), values, 0.5), 0, 1)\n\n\nclass _RadLayoutPrefetch:\n    """One CPU layout builder overlaps GPU work; GPU head shapes are model-owned."""\n    def __init__(self, prepare, configurations, release):\n        from concurrent.futures import ThreadPoolExecutor\n        self.prepare = prepare\n        self.configurations = list(configurations)\n        self.release = release\n        self.executor = ThreadPoolExecutor(max_workers=1)\n        self.pending = None\n        self.next_index = 0\n        self.acquired = None\n\n    def _submit(self):\n        if self.next_index < len(self.configurations):\n            config = self.configurations[self.next_index]\n            self.next_index += 1\n            self.pending = self.executor.submit(self.prepare, *config)\n\n    def __enter__(self):\n        self._submit()\n        return self\n\n    def take(self):\n        if self.acquired is not None:\n            raise RuntimeError(\'release the current Rad layout before taking another\')\n        if self.pending is None:\n            raise StopIteration(\'all Rad layouts consumed\')\n        future, self.pending = self.pending, None\n        try:\n            self.acquired = future.result()\n        except BaseException:\n            self.executor.shutdown(wait=True, cancel_futures=True)\n            raise\n        self._submit()\n        return self.acquired\n\n    def release_current(self):\n        if self.acquired is not None:\n            self.release(self.acquired[1])  # (study IDs, pixels, masks)\n            self.acquired = None\n\n    def __exit__(self, exc_type, exc, tb):\n        self.release_current()\n        pending, self.pending = self.pending, None\n        self.executor.shutdown(wait=True, cancel_futures=True)\n        if pending is not None and not pending.cancelled():\n            try:\n                extra = pending.result()\n                self.release(extra[1])\n            except BaseException:\n                if exc_type is None:\n                    raise\n        return False\n\n\ndef _rad_main():\n    """BTKD Rad/calibrator graph unchanged; diagnostic-only twin work removed.\n\n    BTKD\'s E13-on-E11 second input is intentional in its fitted calibrator.\n    Keep that pair together; do not claim standalone E13 train/serve parity.\n    """\n    started=_rad_time.time();work=_RadPath(\'/kaggle/working\');primary=work/\'_pipeline_stage.csv\'\n    test=_rad_pd.read_csv(ROOT/\'test.csv\',dtype={\'StudyInstanceUID\':str})\n    ids=test.StudyInstanceUID.tolist()\n    baseline=rsna_frame(_rad_pd.read_csv(primary,dtype={\'StudyInstanceUID\':str}),ids,_RAD_LABELS,\'Rad parent\')\n    device=_rad_torch.device(\'cuda:0\')\n    encoder,reference_heads,encoder_path,reference_heads_path=_rad_load_models(device)\n    tser=_rad_pd.read_csv(ROOT/\'test_series.csv\',dtype={\'StudyInstanceUID\':str,\'SeriesInstanceUID\':str})\n    plane=dict(zip(tser.SeriesInstanceUID,tser.Anatomical_Plane))\n    # One annotation pass. All three layouts use the identical legacy annotation rules.\n    headers=annotate(walk(\'test_series\'))\n    side_map=lat_of(headers,\'rad common\')\n    def prepare_layout(layout,crop,tag):\n        globals().update(SLOTS=list(layout),N_SLOT=len(layout),CACHE_SLICES=8,IMG=224,CACHE_IMG=224,CROP_MM=float(crop),RULES=dict(RULES_LEGACY))\n        value=build_cache(pick_slots(headers,plane),plane,side_map,tag)\n        if value[0]!=ids:\n            rsna_release_pixels(value[1])\n            raise RuntimeError(\'Rad cache order changed\')\n        return value\n    def encode_layout(prefetch):\n        studies,pixels,mask=prefetch.take()\n        try:\n            f,tm=_rad_encode(encoder,pixels,mask,device)\n            return f,tm,int(tm.sum())\n        finally:\n            prefetch.release_current()\n    def predict_heads(heads,f,mask,tag):\n        if len(heads)!=5: raise RuntimeError(tag+\': five heads required\')\n        probs=_rad_np.stack([_rad_predict_head(h,f,mask,device) for h in heads])\n        rsna_finite(probs,tag,probability=True);rsna_save_predictions(tag,ids,probs,_RAD_LABELS)\n        return probs.mean(0)\n    first=[(\'SAG_FS\',\'Sagittal\',None,True),(\'COR_FS\',\'Coronal\',None,True),(\'AX_FS\',\'Axial\',None,True)]\n    # E13 has four slots; construct its model before the producer mutates globals.\n    globals().update(SLOTS=list(_RAD_E13_SLOTS),N_SLOT=4,CACHE_SLICES=8)\n    e13_heads,e13_path=_rad_load_e13_heads(device)\n    configurations=[(first,10000.,\'rad_e10\'),(_RAD_E13_SLOTS,130.,\'rad_e13\'),(_RAD_E11_SLOTS,130.,\'rad_legacy_second\')]\n    with _RadLayoutPrefetch(prepare_layout,configurations,rsna_release_pixels) as prefetch:\n        f,m,tokens0=encode_layout(prefetch)\n        reference_rank=_rad_rank_columns(predict_heads(reference_heads,f,m,\'rad_e10_raw\'))\n        del f,m,reference_heads\n        f,m,tokens1=encode_layout(prefetch)\n        e13_rank=_rad_rank_columns(predict_heads(e13_heads,f,m,\'rad_e13_raw\'))\n        reference_rank=_rad_rank_columns((1-_RAD_E13_MEMBER_WEIGHT)*reference_rank+_RAD_E13_MEMBER_WEIGHT*e13_rank)\n        del f,m,e13_rank\n        baseline_rank=_rad_rank_columns(baseline[_RAD_LABELS].to_numpy())\n        candidate=baseline.copy()\n        for j,label in enumerate(_RAD_LABELS):\n            if label not in _RAD_EXCLUDE:\n                candidate[label]=(1-_RAD_ALPHA)*baseline_rank[:,j]+_RAD_ALPHA*reference_rank[:,j]\n        _rad_validate(candidate,ids)\n        f,m,tokens2=encode_layout(prefetch)\n        second_rank=_rad_rank_columns(predict_heads(e13_heads,f,m,\'rad_legacy_second_raw\'))\n        del f,m,e13_heads\n    branch=candidate.copy()\n    branch[_RAD_LABELS]=_rad_rank_columns((1-_RAD_V48_SECOND_ALPHA)*_rad_rank_columns(candidate[_RAD_LABELS].to_numpy())+_RAD_V48_SECOND_ALPHA*second_rank)\n    cal=_rad_json.loads(_rad_zlib.decompress(_rad_b64.b64decode(_RAD_CAL_PAYLOAD)).decode())\n    protocol=_rad_pd.DataFrame(index=_rad_pd.Index(ids,name=\'StudyInstanceUID\'))\n    protocol[\'n_series\']=tser.groupby(\'StudyInstanceUID\').size().reindex(ids).fillna(0)\n    for pl in (\'Sagittal\',\'Coronal\',\'Axial\'):\n        part=tser[tser.Anatomical_Plane.astype(str)==pl]\n        protocol[f\'n_{pl[:3]}\']=part.groupby(\'StudyInstanceUID\').size().reindex(ids).fillna(0)\n    for flag in (\'Fat_Suppression\',\'Fluid_Sensitive\'):\n        marked=tser[_rad_pd.to_numeric(tser[flag],errors=\'coerce\').fillna(0)>0]\n        protocol[flag[:3]]=marked.groupby(\'StudyInstanceUID\').size().reindex(ids).fillna(0)\n        for pl in (\'Sagittal\',\'Coronal\',\'Axial\'):\n            part=marked[marked.Anatomical_Plane.astype(str)==pl]\n            protocol[f\'{flag[:3]}_{pl[:3]}\']=part.groupby(\'StudyInstanceUID\').size().reindex(ids).fillna(0)\n    if list(protocol.columns)!=list(cal[\'protocol_columns\']): raise RuntimeError(\'calibrator protocol drift\')\n    mean=(baseline_rank+reference_rank+second_rank)/3.\n    blocks=[baseline_rank,reference_rank,second_rank,reference_rank-baseline_rank,second_rank-baseline_rank,mean]\n    for group in cal[\'groups\']:\n        blocks.append(mean[:,[_RAD_LABELS.index(t) for t in group]].mean(axis=1,keepdims=True))\n    blocks.append(protocol.to_numpy(_rad_np.float64))\n    x=_rad_np.concatenate(blocks,axis=1); center=_rad_np.asarray(cal[\'mean\']);spread=_rad_np.asarray(cal[\'scale\']);coef=_rad_np.asarray(cal[\'coef\']);bias=_rad_np.asarray(cal[\'intercept\'])\n    if x.shape[1]!=coef.shape[1] or not bool((spread>0).all()): raise RuntimeError(\'calibrator shape/scale drift\')\n    adjusted=_rad_rank_columns(((x-center)/spread)@coef.T+bias)\n    values=branch[_RAD_LABELS].to_numpy(_rad_np.float64).copy()\n    for j,label in enumerate(_RAD_LABELS):\n        if label in set(cal[\'gate\']): values[:,j]=.6*values[:,j]+.4*adjusted[:,j]\n    final=branch.copy();final[_RAD_LABELS]=_rad_rank_columns(values)\n    _rad_validate(final,ids);rsna_save_predictions(\'btkd_parent_after_rad\',ids,final[_RAD_LABELS].to_numpy(),_RAD_LABELS)\n    tmp=primary.with_suffix(\'.tmp\');final.to_csv(tmp,index=False);_rad_os.replace(tmp,primary)\n    globals()[\'V18_CALIBRATOR_APPLIED\']=True\n    rsna_json(work/\'v558_rad_receipt.json\',{\'status\':\'COMPLETE\',\'graph\':\'BTKD legacy Rad plus its matched calibrator preserved\',\'reference_heads_sha256\':_RAD_REFERENCE_HEADS_SHA256,\'e13_sha256\':_RAD_E13_HEADS_SHA256,\'encoder_sha256\':_RAD_ENCODER_SHA256,\'legacy_cross_layout_pass_preserved\':True,\'standalone_cross_layout_train_parity_claimed\':False,\'unused_alternate_family_removed\':True,\'encoder_replicas\':2,\'layout_cpu_prefetch\':True,\'model_owned_head_shape\':True,\'tokens\':[tokens0,tokens1,tokens2],\'seconds\':_rad_time.time()-started})\n    del encoder;_rad_gc.collect()\n    for d in range(2):\n        with _rad_torch.cuda.device(d): _rad_torch.cuda.empty_cache()\n    _rad_log(\'BTKD reference Rad/calibrator complete; unused alternate-head branch removed\')\n\n\nrsna_phase(\'radimagenet\', \'START\')\n_rad_main()\nrsna_phase(\'radimagenet\', \'COMPLETE\')\n', globals())


## Stages 4–5 — Raptor and complementary CoAtNet readers

Four Raptor views run first. Small cohorts overlap residual + D4 and
Global96 + Repair-v1; large cohorts run the readers one at a time. When all
four CoAt readers succeed, their probability arrays are averaged 25% each and
ranked, then blended 40% into Raptor. Missing arms use the scored upstream
fallback; outer routing is unchanged.


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(27, '# Reader threads share diagnostic JSON paths; serialize atomic writes.\nimport threading as _v9_threading\n_v9_json_lock = _v9_threading.RLock()\n_v9_original_json = rsna_json\ndef rsna_json(*args, **kwargs):\n    with _v9_json_lock:\n        return _v9_original_json(*args, **kwargs)\n\nif globals().get(\'_DINOV2_MATCHED_MEMBERS\') != 20:\n    raise RuntimeError(\'DINOv2 20/20 fingerprint gate did not pass\')\nimport gc as _ke_gc\nimport os as _ke_os\nimport time as _ke_time\nfrom concurrent.futures import ThreadPoolExecutor as _KeThreadPool\nimport numpy as _ke_np\nimport pandas as _ke_pd\nfrom pathlib import Path as _KePath\n\n_ke_primary = _KePath(\'/kaggle/working/_pipeline_stage.csv\')\n_ke_ours = _ke_pd.read_csv(_ke_primary, dtype={\'StudyInstanceUID\': str})\n_KE_LAB = [c for c in _ke_ours.columns if c != \'StudyInstanceUID\']\n\n_KE_SRC = r\'\'\'\nimport os, glob, time, gc, hashlib\nos.environ.setdefault(\'HF_HUB_OFFLINE\', \'1\')\nos.environ.setdefault(\'TRANSFORMERS_OFFLINE\', \'1\')\nos.environ.setdefault(\'HF_HUB_DISABLE_TELEMETRY\', \'1\')\nimport numpy as np\nimport torch, torch.nn as nn, torch.nn.functional as F\nimport timm\ntorch.backends.cudnn.benchmark = False  # ragged 94-window chunks: no repeated autotuning\ntorch.backends.cuda.matmul.allow_tf32 = True\nIMG = 336\nCROP_MM = 140.0\nSPAN_LO, SPAN_HI = 0.02, 0.98\nSLOTS = [("Sagittal", 1, 18), ("Sagittal", 0, 14),\n         ("Coronal", 1, 12), ("Coronal", 0, 8), ("Axial", -1, 12)]\nMAXS = sum(slot[2] for slot in SLOTS)\nK_EVAL = 62\nNORM = "imagenet"\nLAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",\n       "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker\'s",\n       "Contusion", "Fracture"]\n_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)\n_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)\n_SLOTS64 = [("Sagittal", 1, 18), ("Sagittal", 0, 14),\n            ("Coronal", 1, 12), ("Coronal", 0, 8), ("Axial", -1, 12)]\n_SLOTS44 = [("Sagittal", 1, 12), ("Sagittal", 0, 10),\n            ("Coronal", 1, 8), ("Coronal", 0, 6), ("Axial", -1, 8)]\nARMS = [\n    {"name": "maxspan-v5", "file": "raptor_ft_coatnet_v5_full_swa.pt",\n     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,\n     "img": 336, "slots": _SLOTS64, "span": (0.02, 0.98), "k_eval": 62,\n     "reverse": False, "w": 0.60},\n    {"name": "native384dense-v10", "file": "raptor_ft_coatnet_v10_full.pt",\n     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,\n     "img": 384, "slots": _SLOTS64, "span": (0.02, 0.98), "k_eval": 62,\n     "reverse": False, "w": 0.10},\n    {"name": "maxspan-v5-reverse", "file": "raptor_ft_coatnet_v5_full_swa.pt",\n     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,\n     "img": 336, "slots": _SLOTS64, "span": (0.02, 0.98), "k_eval": 62,\n     "reverse": True, "w": 0.10},\n    {"name": "native384-v8", "file": "raptor_ft_coatnet_v8_full_swa.pt",\n     "arch": "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "res": 384,\n     "img": 384, "slots": _SLOTS44, "span": (0.06, 0.94), "k_eval": 42,\n     "reverse": False, "w": 0.20},\n]\n\ndef build_backbone(arch, pretrained=False):\n    hybrid = arch.startswith((\'maxvit\', \'maxxvit\', \'coatnet\', \'coat_\', \'convnext\'))\n    is_vit = not hybrid and any((k in arch for k in (\'vit\', \'deit\', \'dinov2\', \'eva\', \'beit\')))\n    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)\n    if is_vit:\n        kw.update(global_pool=\'token\', dynamic_img_size=True)\n    else:\n        kw.update(global_pool=\'avg\')\n    return timm.create_model(arch, **kw)\n\nclass RaptorClassifier(nn.Module):\n\n    def __init__(self, backbone, F_dim=768, n=12, drop=0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n\n    def encode(self, x):\n        B, K = x.shape[:2]\n        f = self.backbone(x.flatten(0, 1))\n        return f.view(B, K, -1)\n\n    def head(self, feats):\n        h = self.norm(feats)\n        a = self.att(h)\n        a = torch.softmax(a, dim=1)\n        pooled = torch.einsum(\'bkn,bkf->bnf\', a, h)\n        logits = (pooled * self.clsW).sum(-1) + self.clsb\n        return logits\n\n    def forward(self, x):\n        return self.head(self.encode(x))\n\ndef load_model(pt_path, arch_default, res_default, device, ngpu=1):\n    ck = torch.load(pt_path, map_location=\'cpu\', weights_only=False)\n    arch = ck.get(\'arch\', arch_default)\n    if arch != arch_default:\n        raise RuntimeError(f\'Raptor architecture drift: {arch} != {arch_default}\')\n    ck_res = int(ck.get(\'res\', res_default))\n    bb = build_backbone(arch, pretrained=False)\n    model = RaptorClassifier(bb, F_dim=bb.num_features)\n    rsna_strict_load(model,ck[\'model\'],str(pt_path))\n    rsna_event(\'raptor_checkpoint\',file=str(pt_path),sha256=rsna_sha(pt_path),arch=arch,res=ck_res)\n    model.eval().to(device)\n    del ck\n    gc.collect()\n    return (model, ck_res)\n\n\n\ndef _eval_centers(mask, D, k):\n    valid = np.where(mask > 0)[0]\n    if len(valid) < 3:\n        valid = np.arange(min(3, D))\n    lo, hi = (int(valid.min()), int(valid.max()))\n    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]\n    if not cs:\n        cs = [max(1, min((lo + hi) // 2, D - 2))]\n    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)\n    return [cs[i] for i in idx]\n\ndef eval_windows(vol, mask, k, res, norm=NORM):\n    """Resize each selected source plane once, then gather the unchanged RGB triplets."""\n    volume = np.asarray(vol)\n    depth = int(volume.shape[0])\n    centers = np.asarray(_eval_centers(mask, depth, k), dtype=np.int64)\n    centers = np.clip(centers, 1, depth - 2)\n    if tuple(volume.shape[-2:]) == (res, res):\n        # Native384 needs no interpolation; the existing small per-triplet copy\n        # avoids a slower large advanced-index gather and preserves its exact math.\n        wins = np.empty((len(centers), 3, res, res), np.float32)\n        for j, center in enumerate(centers):\n            wins[j] = np.stack([volume[center - 1], volume[center],\n                                volume[center + 1]], axis=0).astype(np.float32) / 255.0\n        x = torch.from_numpy(wins)\n        if norm == \'imagenet\':\n            x = (x - _MEAN) / _STD\n        return x\n    triplets = centers[:, None] + np.asarray([-1, 0, 1], dtype=np.int64)\n    # Use only planes referenced by the original eval-center recipe.\n    unique, inverse = np.unique(triplets.reshape(-1), return_inverse=True)\n    source = volume[unique].astype(np.float32) / 255.0\n    planes = torch.from_numpy(source)\n    if tuple(planes.shape[-2:]) != (res, res):\n        resized = torch.empty((len(planes), res, res), dtype=torch.float32)\n        for start in range(0, len(planes), 16):\n            resized[start:start+16] = F.interpolate(\n                planes[start:start+16, None], size=(res, res),\n                mode=\'bilinear\', align_corners=False)[:, 0]\n        planes = resized\n    x = planes[torch.from_numpy(inverse.reshape(-1, 3))]\n    if norm == \'imagenet\':\n        x = (x - _MEAN) / _STD\n    return x\n\n\n\n\n\ndef rankpct(x):\n    return rsna_rank01(x)\n\ndef _make_reader():\n    import pydicom, cv2\n    from pydicom.pixel_data_handlers.util import apply_modality_lut\n\n    def order_and_meta(sdir):\n        fs = glob.glob(sdir + \'/*.dcm\')\n        recs = []\n        ps_list = []\n        for f in fs:\n            try:\n                h = pydicom.dcmread(f, stop_before_pixels=True)\n                iop = getattr(h, \'ImageOrientationPatient\', None)\n                ipp = getattr(h, \'ImagePositionPatient\', None)\n                if iop is not None and ipp is not None and (len(iop) == 6):\n                    r = np.array(iop[:3], float)\n                    c = np.array(iop[3:], float)\n                    n = np.cross(r, c)\n                    pos = float(np.dot(np.array(ipp, float), n))\n                else:\n                    instance = getattr(h, \'InstanceNumber\', None)\n                    if instance is None:\n                        rsna_event(\'raptor_order_index_fallback\', file=str(f)); instance = len(recs)\n                    pos = float(instance)\n                if not np.isfinite(pos):\n                    rsna_event(\'raptor_order_nonfinite_fallback\', file=str(f)); pos = float(len(recs))\n                ps = getattr(h, \'PixelSpacing\', None)\n                ps = float(ps[0]) if ps is not None else 0.5\n                ps_list.append(ps)\n                recs.append((pos, f, ps))\n            except Exception as exc:\n                rsna_event(\'raptor_order_unreadable_skipped\', file=str(f), error=f\'{type(exc).__name__}: {exc}\'); continue\n        recs.sort(key=lambda x: (x[0], x[1]))\n        med_ps = float(np.median(ps_list)) if ps_list else 0.5\n        return ([(f, ps) for _, f, ps in recs], med_ps)\n\n    def read_px(f):\n        d = pydicom.dcmread(f)\n        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n        if str(getattr(d, \'PhotometricInterpretation\', \'\')) == \'MONOCHROME1\':\n            a = a.max() - a\n        return a\n\n    def mm_crop_resize(a, ps):\n        h, w = a.shape\n        cpx = int(round(CROP_MM / max(ps, 0.001)))\n        cpx = min(cpx, min(h, w))\n        y0 = (h - cpx) // 2\n        x0 = (w - cpx) // 2\n        a = a[y0:y0 + cpx, x0:x0 + cpx]\n        return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)\n    return (order_and_meta, read_px, mm_crop_resize)\n\ndef _pick_series_for_slot(rows, plane, fluid, used):\n    """An unknown protocol flag is not false and must not reach int(NaN)."""\n    def flag(value):\n        if value is None:\n            return None\n        if isinstance(value, str) and value.strip().lower() in (\'\', \'nan\', \'none\', \'<na>\', \'unknown\'):\n            return None\n        try:\n            number = float(value)\n        except (TypeError, ValueError):\n            text = str(value).strip().lower()\n            if text in (\'true\', \'yes\', \'y\', \'t\'):\n                return 1\n            if text in (\'false\', \'no\', \'n\', \'f\'):\n                return 0\n            return None  # unknown code: neither false nor fatal; the plane-level fallback applies\n        if not np.isfinite(number):\n            return None\n        if number not in (0.0, 1.0):\n            return None\n        return int(number)\n    candidates = [r for r in rows if r[\'Anatomical_Plane\'] == plane\n                  and r[\'SeriesInstanceUID\'] not in used]\n    if fluid in (0, 1):\n        preferred = [r for r in candidates if flag(r.get(\'Fluid_Sensitive\')) == fluid]\n        if preferred:\n            return preferred[0]\n    return candidates[0] if candidates else None\n\n\n\ndef find_test_root():\n    from pathlib import Path\n    for value in [os.environ.get(\'RSNA_COMP_ROOT\', \'\'),\n                  \'/kaggle/input/competitions/rsna-knee-abnormality-detection\',\n                  \'/kaggle/input/rsna-knee-abnormality-detection\']:\n        if value and (Path(value)/\'test.csv\').is_file():\n            return value\n    raise FileNotFoundError(\'explicit competition root absent\')\n\ndef find_weight_file(fname):\n    return str(_asset_find_asset(fname))\n\'\'\'\n_KE_NS = {\'__name__\': \'_ke_raptor\', \'_asset_find_asset\': _asset_find_asset, \'rsna_rank01\':rsna_rank01, \'rsna_strict_load\':rsna_strict_load, \'rsna_event\':rsna_event, \'rsna_sha\':rsna_sha}\nexec(compile(_KE_SRC, \'<raptor>\', \'exec\'), _KE_NS)\nfrom concurrent.futures import Future as _KeFuture\nimport threading as _ke_threading\n\n_ke_base_make_reader = _KE_NS[\'_make_reader\']\n_ke_order_cache = {}\nfor _ke_series_dir, _ke_header_value in globals().get(\'RAPTOR_HEADER_CACHE\', {}).items():\n    try:\n        _ke_cached_paths = [item[0] for item in _ke_header_value[0]]\n        _ke_current_paths = _KE_NS[\'glob\'].glob(str(_ke_series_dir) + \'/*.dcm\')\n        if len(_ke_cached_paths) != len(_ke_current_paths) or set(_ke_cached_paths) != set(_ke_current_paths):\n            continue\n        _ke_future = _KeFuture()\n        _ke_future.set_result(_ke_header_value)\n        _ke_order_cache[str(_ke_series_dir)] = _ke_future\n    except (OSError, TypeError, ValueError):\n        continue\nfrom collections import OrderedDict as _KeOrderedDict\n_ke_pixel_cache = _KeOrderedDict()\n_ke_order_cache_lock = _ke_threading.Lock()\n\n\ndef _ke_make_cached_reader():\n    """Share immutable DICOM ordering metadata across Raptor views."""\n    base_order, read_pixel, crop_resize = _ke_base_make_reader()\n\n    def cached_order(series_dir):\n        key = str(series_dir)\n        owner = False\n        with _ke_order_cache_lock:\n            future = _ke_order_cache.get(key)\n            if future is None:\n                future = _KeFuture()\n                _ke_order_cache[key] = future\n                owner = True\n        if owner:\n            try:\n                future.set_result(base_order(series_dir))\n            except BaseException as error:\n                future.set_exception(error)\n                with _ke_order_cache_lock:\n                    _ke_order_cache.pop(key, None)\n                raise\n        return future.result()\n\n    def cached_pixel(path):\n        global _ke_pixel_cache_bytes\n        from collections import OrderedDict\n        key=str(path)\n        with _ke_order_cache_lock:\n            got=_ke_pixel_cache.get(key)\n            if got is not None:\n                _ke_pixel_cache.move_to_end(key);return got\n        value=read_pixel(path)\n        if value.ndim!=2 or not _ke_np.isfinite(value).all():\n            raise RuntimeError(f\'Raptor invalid pixels {path}\')\n        value.setflags(write=False)\n        with _ke_order_cache_lock:\n            previous = _ke_pixel_cache.pop(key, None)\n            if previous is not None:\n                _ke_pixel_cache_bytes -= previous.nbytes\n            _ke_pixel_cache[key] = value\n            _ke_pixel_cache_bytes += value.nbytes\n            while _ke_pixel_cache_bytes > (192 << 20):\n                _, removed = _ke_pixel_cache.popitem(last=False)\n                _ke_pixel_cache_bytes -= removed.nbytes\n        return value\n    return cached_order, cached_pixel, crop_resize\n\n\n_KE_NS[\'_make_reader\'] = _ke_make_cached_reader\nif _ke_os.environ.get(\'RSNA_COMP_ROOT\'):\n    _KE_NS[\'find_test_root\'] = lambda: _ke_os.environ[\'RSNA_COMP_ROOT\']\n\n\ndef _ke_prepare_windows(arm, study_uid, series, series_root, reader):\n\n\n\n\n\n\n\n    order_and_meta, read_px, _ = reader\n    image_size = int(arm[\'img\'])\n    slots = list(arm[\'slots\'])\n    span_lo, span_hi = map(float, arm[\'span\'])\n    used, pools = set(), []\n    rows = series.get(study_uid, [])\n    for plane, fluid, count in slots:\n        record = _KE_NS[\'_pick_series_for_slot\'](rows, plane, fluid, used)\n        if record is None:\n            pools.append(None)\n            continue\n        used.add(record[\'SeriesInstanceUID\'])\n        files, median_spacing = order_and_meta(\n            f"{series_root}/{study_uid}/{record[\'SeriesInstanceUID\']}")\n        if not files:\n            rsna_event(\'raptor_acquisition_empty\', study=str(study_uid), series=str(record[\'SeriesInstanceUID\'])); pools.append(None); continue\n        lo = int(len(files) * span_lo)\n        hi = max(int(len(files) * span_hi) - 1, lo)\n        original = _ke_np.linspace(lo, hi, count).round().astype(int)\n        pools.append((files, median_spacing, lo, hi, original))\n    capacities = [0 if p is None else p[3] - p[2] + 1 for p in pools]\n    quotas = _dense_allocate(capacities, [s[2] for s in slots], 96)\n    volume, sources = [], []\n    import cv2\n    for pool, quota in zip(pools, quotas):\n        if pool is None or not quota:\n            continue\n        files, median_spacing, lo, hi, original = pool\n        picks = lo + _dense_unique_linspace(hi - lo + 1, int(quota))\n        arrays = {}\n        # Decode union once. Failure is explicit, not an unreported 0.5 score.\n        for i in sorted(set(original.tolist()) | set(picks.tolist())):\n            arrays[i] = read_px(files[i][0])\n        all_pixels = _ke_np.concatenate([arrays[int(i)].ravel() for i in original])\n        low, high = _ke_np.percentile(all_pixels, [2.0, 98.0])\n        for i in picks:\n            path, spacing = files[int(i)]\n            a = arrays[int(i)]\n            a = _ke_np.clip((a-low)/(high-low+1e-6), 0, 1)\n            spacing = spacing if spacing > 0 else median_spacing\n            h, w = a.shape\n            c = min(int(round(140.0/max(spacing, .001))), min(h, w))\n            y, x = (h-c)//2, (w-c)//2\n            a = cv2.resize(a[y:y+c, x:x+c], (image_size,image_size),\n                           interpolation=cv2.INTER_AREA)\n            volume.append((a*255).astype(_ke_np.uint8))\n            sources.append(str(path))\n    if len(sources) != len(set(sources)):\n        raise RuntimeError(\'capacity-aware sampling selected duplicate source files\')\n    if len(volume) < 3:\n        raise RuntimeError(f\'{study_uid}: fewer than three unique source slices\')\n    volume = _ke_np.stack(volume)\n    # Presence means an acquired source, not an intensity test. A genuinely\n    # black acquired slice is not interchangeable with padding.\n    mask = _ke_np.ones(len(volume), dtype=_ke_np.uint8)\n    windows = _KE_NS[\'eval_windows\'](\n        volume, mask, k=len(volume)-2, res=int(arm[\'res\']), norm=_KE_NS[\'NORM\'])\n    if len(windows) != min(96, sum(capacities))-2:\n        raise RuntimeError(\'capacity-aware sampling Raptor cardinality drift\')\n    import hashlib, json\n    signature = hashlib.sha256(json.dumps(sources, ensure_ascii=False,\n                                separators=(\',\', \':\')).encode()).hexdigest()\n    audit = {\'arm\':arm[\'name\'], \'uid\':str(study_uid),\n             \'capacities\':capacities, \'quotas\':quotas.tolist(),\n             \'source_count\':len(sources), \'source_list_sha256\':signature,\n             \'windows\':len(windows), \'duplicate_sources\':0}\n    identity = (arm[\'name\'], str(study_uid))\n    with _ke_input_lock:\n        if identity in _ke_input_ids:\n            raise RuntimeError(f\'Duplicate Raptor preparation: {identity}\')\n        _ke_input_ids.add(identity)\n        with _ke_input_audit_path.open(\'a\') as handle:\n            handle.write(json.dumps(audit, sort_keys=True, allow_nan=False) + \'\\n\')\n    return windows\n\n\ndef _ke_infer_input(model, xwins, device):\n    """Bound backbone workspace; keep all window features for one head call."""\n    torch = _KE_NS[\'torch\']\n    def _forward(_amp):\n        with torch.inference_mode(), torch.autocast(\'cuda\', dtype=torch.float16, enabled=_amp):\n            features = [model.backbone(xwins[i:i+8].to(device))\n                        for i in range(0,len(xwins),8)]\n            feats = torch.cat(features,dim=0).unsqueeze(0)\n            return torch.sigmoid(model.head(feats).float())[0].cpu().numpy()\n    probabilities = _forward(True)\n    if not _ke_np.isfinite(probabilities).all():\n        rsna_event(\'raptor_fp16_nonfinite_retry_fp32\')\n        probabilities = _forward(False)\n    if not _ke_np.isfinite(probabilities).all():\n        raise RuntimeError(\'nonfinite Raptor prediction\')\n    return probabilities\n\n\ndef _ke_prefetched_windows(arm, test_ids, series, series_root, reader):\n    """Bound host memory while decoding one study ahead of its GPU forward."""\n    depth = int(_ke_os.environ.get(\'RSNA_RAPTOR_PREFETCH\', \'2\'))\n    if not 1 <= depth <= 4: raise ValueError(\'Raptor prefetch must be 1..4\')\n    with _KeThreadPool(max_workers=1) as executor:\n        pending = {}\n        submit_at = 0\n        while submit_at < min(depth, len(test_ids)):\n            pending[submit_at] = executor.submit(\n                _ke_prepare_windows,\n                arm,\n                test_ids[submit_at],\n                series,\n                series_root,\n                reader,\n            )\n            submit_at += 1\n        for study_index, study_uid in enumerate(test_ids):\n            future = pending.pop(study_index)\n            if submit_at < len(test_ids):\n                pending[submit_at] = executor.submit(\n                    _ke_prepare_windows,\n                    arm,\n                    test_ids[submit_at],\n                    series,\n                    series_root,\n                    reader,\n                )\n                submit_at += 1\n            yield study_index, study_uid, future\n\n\ndef _ke_run_raptor_arms():\n    """Run the four unchanged Raptor arms across exactly two GPUs."""\n    torch = _KE_NS[\'torch\']\n    if not torch.cuda.is_available() or torch.cuda.device_count() != 2:\n        raise RuntimeError(\n            f"optimized Raptor requires exactly two GPUs, got {torch.cuda.device_count()}"\n        )\n    started = _ke_time.time()\n    root = _KE_NS[\'find_test_root\']()\n    series_root = root + \'/test_series\'\n    if not _ke_os.path.isdir(series_root):\n        series_root = root + \'/test_images\'\n    test = _ke_pd.read_csv(root + \'/test.csv\')\n    test[\'StudyInstanceUID\'] = test[\'StudyInstanceUID\'].astype(str)\n    test_ids = test[\'StudyInstanceUID\'].tolist()\n    test_series = _ke_pd.read_csv(root + \'/test_series.csv\')\n    test_series[\'StudyInstanceUID\'] = test_series[\'StudyInstanceUID\'].astype(str)\n    test_series[\'SeriesInstanceUID\'] = test_series[\'SeriesInstanceUID\'].astype(str)\n    series = {\n        key: frame.to_dict(\'records\')\n        for key, frame in test_series.groupby(\'StudyInstanceUID\')\n    }\n    sample = root + \'/sample_submission.csv\'\n    columns = [\'StudyInstanceUID\', *_KE_NS[\'LAB\']]\n    if _ke_os.path.exists(sample):\n        columns = list(_ke_pd.read_csv(sample, nrows=1).columns)\n    arms = list(_KE_NS[\'ARMS\'])\n    outputs = [\n        _ke_np.full((len(test_ids), len(_KE_NS[\'LAB\'])), np.nan, _ke_np.float32)\n        for _ in arms\n    ]\n    print(\n        f"[raptor-fast] {len(test_ids)} studies; balanced arm groups "\n        f"cuda:0=[0,2], cuda:1=[1,3]",\n        flush=True,\n    )\n\n    def run_single(arm_index, device):\n        arm = arms[arm_index]\n        reader = _KE_NS[\'_make_reader\']()\n        weight_path = _KE_NS[\'find_weight_file\'](arm[\'file\'])\n        model, resolution = _KE_NS[\'load_model\'](\n            weight_path, arm[\'arch\'], arm[\'res\'], device\n        )\n        if int(resolution) != int(arm[\'res\']):\n            raise RuntimeError(\n                f"{arm[\'name\']} checkpoint resolution {resolution} != {arm[\'res\']}"\n            )\n        for study_index, study_uid, future in _ke_prefetched_windows(\n            arm, test_ids, series, series_root, reader\n        ):\n            rsna_deadline(\'Raptor full-cohort inference\')\n            try:\n                windows = future.result()\n                outputs[arm_index][study_index] = _KE_NS[\'infer_probs\'](\n                    model, windows, device\n                )\n                del windows\n            except Exception as error:\n                rsna_event(\'raptor_study_failed\', arm=str(arm[\'name\']), study=str(study_uid), error=f\'{type(error).__name__}: {str(error)[:500]}\')\n                with _ke_input_lock:\n                    _ke_input_ids.add((arm[\'name\'], str(study_uid)))\n        del model\n        _ke_gc.collect()\n        with torch.cuda.device(device):\n            torch.cuda.empty_cache()\n\n    def run_shared_maxspan(device):\n        first, reverse = arms[0], arms[2]\n        comparable = (\'file\', \'arch\', \'res\', \'img\', \'slots\', \'span\', \'k_eval\')\n        if any(first[key] != reverse[key] for key in comparable):\n            raise RuntimeError(\'MaxSpan forward/reverse arms no longer share preprocessing\')\n        reader = _KE_NS[\'_make_reader\']()\n        weight_path = _KE_NS[\'find_weight_file\'](first[\'file\'])\n        model, resolution = _KE_NS[\'load_model\'](\n            weight_path, first[\'arch\'], first[\'res\'], device\n        )\n        if int(resolution) != int(first[\'res\']):\n            raise RuntimeError(\n                f"MaxSpan checkpoint resolution {resolution} != {first[\'res\']}"\n            )\n        for study_index, study_uid, future in _ke_prefetched_windows(\n            first, test_ids, series, series_root, reader\n        ):\n            rsna_deadline(\'Raptor full-cohort inference\')\n            try:\n                windows = future.result()\n                outputs[0][study_index] = _KE_NS[\'infer_probs\'](\n                    model, windows, device\n                )\n                outputs[2][study_index] = _KE_NS[\'infer_probs\'](\n                    model, windows.flip(1).contiguous(), device\n                )\n                del windows\n            except Exception as error:\n                rsna_event(\'raptor_study_failed\', arm=str(first[\'name\']), study=str(study_uid), error=f\'{type(error).__name__}: {str(error)[:500]}\')\n                with _ke_input_lock:\n                    _ke_input_ids.add((first[\'name\'], str(study_uid))); _ke_input_ids.add((reverse[\'name\'], str(study_uid)))\n        del model\n        _ke_gc.collect()\n        with torch.cuda.device(device):\n            torch.cuda.empty_cache()\n\n    def gpu_zero():\n        with torch.cuda.device(0):\n            run_shared_maxspan(torch.device(\'cuda:0\'))\n\n    def gpu_one():\n        with torch.cuda.device(1):\n            run_single(1, torch.device(\'cuda:1\'))\n            run_single(3, torch.device(\'cuda:1\'))\n\n    with _KeThreadPool(max_workers=2) as executor:\n        workers = [executor.submit(gpu_zero), executor.submit(gpu_one)]\n        for worker in workers:\n            worker.result()\n\n    _ke_np.savez_compressed(\'/kaggle/working/raptor_raw.npz\',\n        study_uids=_ke_np.asarray(test_ids), raw_probabilities=_ke_np.stack(outputs))\n    for _ai, _arr in enumerate(outputs):\n        _bad = ~_ke_np.isfinite(_arr).all(axis=1)\n        if _bad.any():\n            _fill = _ke_np.nanmean(_ke_np.where(_ke_np.isfinite(_arr), _arr, _ke_np.nan), axis=0) if (~_bad).any() else _ke_np.full(_arr.shape[1], .5, _arr.dtype)\n            _arr[_bad] = _ke_np.where(_ke_np.isfinite(_fill), _fill, .5)\n            rsna_event(\'raptor_neutral_fill\', arm=str(arms[_ai][\'name\']), studies=int(_bad.sum()))\n    rsna_finite(_ke_np.stack(outputs),\'all four Raptor views\',probability=True)\n    weights = _ke_np.asarray([float(arm[\'w\']) for arm in arms], _ke_np.float64)\n    weights /= weights.sum()\n    probability_blend = _ke_np.tensordot(\n        weights,\n        _ke_np.stack([_ke_np.clip(value, 0, 1) for value in outputs]),\n        axes=(0, 0),\n    )\n    ranks = _KE_NS[\'rankpct\'](probability_blend)\n    rsna_finite(ranks,\'Raptor ranked ensemble\',probability=True)\n    submission = _ke_pd.DataFrame(ranks.astype(_ke_np.float32), columns=_KE_NS[\'LAB\'])\n    submission.insert(0, \'StudyInstanceUID\', test_ids)\n    submission = submission[columns]\n    if submission[\'StudyInstanceUID\'].tolist() != test_ids:\n        raise RuntimeError(\'Raptor study order drift\')\n    if not _ke_np.isfinite(submission[_KE_NS[\'LAB\']].to_numpy()).all():\n        raise RuntimeError(\'Raptor produced non-finite predictions\')\n    output = _KePath(\'/kaggle/working/_raptor.csv\')\n    submission.to_csv(output, index=False)\n    print(\n        f"[raptor-fast] wrote {output}; elapsed {_ke_time.time() - started:.1f}s",\n        flush=True,\n    )\n\n\n_ke_input_lock = _ke_threading.Lock()\n_ke_input_ids = set()\n_ke_pixel_cache_bytes = 0\n_ke_input_audit_path = _KePath(\'/kaggle/working/diagnostics/raptor_inputs.jsonl\')\n_ke_input_audit_path.unlink(missing_ok=True)\n_KE_NS[\'infer_probs\'] = _ke_infer_input\nfor _arm in _KE_NS[\'ARMS\']:\n    _arm[\'parent_k_eval\'] = _arm[\'k_eval\']\n    _arm[\'k_eval\'] = 94\nrsna_phase(\'public_raptor\', \'START\')\n_ke_run_raptor_arms()\nrsna_phase(\'public_raptor\', \'COMPLETE\')\n_ke_pixel_cache.clear()\n_ke_order_cache.clear()\n_ke_gc.collect()\n_KePath(\'/kaggle/working/raptor_input_before_coat.csv\').write_bytes(_KePath(\'/kaggle/working/_raptor.csv\').read_bytes())\nimport json as _asset_json\n_KePath(\'/kaggle/working/raptor_input_summary.json\').write_text(\n    _asset_json.dumps({\'studies\':len(_RSNA_TEST_IDS), \'preparations\':len(_ke_input_ids),\n                    \'view_names\':[a[\'name\'] for a in _KE_NS[\'ARMS\']],\n                    \'shared_preparation_views\':[\'maxspan-v5\',\'maxspan-v5-reverse\'],\n                    \'records\':str(_ke_input_audit_path)},indent=2))\n\n\n\ndef _coat_substitute():\n    import hashlib as _h, os as _o, subprocess as _sp, sys as _sy\n    from pathlib import Path as _P\n    import pandas as _pd\n\n    import numpy as _np\n    _P(\'/kaggle/working/_raptor_public.csv\').write_bytes(\n        _P(\'/kaggle/working/_raptor.csv\').read_bytes())\n    _np.savez_compressed(\'/kaggle/working/btk_v32_rest_input.npz\',\n        study_uids=_ke_ours.StudyInstanceUID.to_numpy().astype(str),\n        labels=_np.asarray(_KE_LAB), values=_ke_ours[_KE_LAB].to_numpy())\n    MAN_SHA = \'98511a8fdeb9da0e6e70c78d013ff636e1476f31c80b5dc134d294b18c3f284e\'\n    WHL_SHA = \'236c8df54a90f4d02076e6f9c1cc763d794542e886c576a6fee46ec8ff75a7a9\'\n    raptor = _P(\'/kaggle/working/_raptor.csv\')\n\n    def sha(p):\n        d = _h.sha256()\n        with _P(p).open(\'rb\') as f:\n            for b in iter(lambda: f.read(8 << 20), b\'\'):\n                d.update(b)\n        return d.hexdigest()\n\n    def find(name, want):\n        return _asset_find_asset(name, want)\n\n    man = find(\'coat_resgated_ep10_top3_manifest.json\', MAN_SHA)\n    if man is None:\n        raise RuntimeError(\'coat manifest absent or hash mismatch\')\n\n    art = man.parent\n\n    whl = find(\'opencv_python_headless-4.12.0.88-*.whl\', WHL_SHA)\n    envd = _P(\'/kaggle/working/_coat_env\')\n    _sp.run(\n        [\n            _sy.executable,\n            \'-m\',\n            \'pip\',\n            \'install\',\n            \'--no-deps\',\n            \'--quiet\',\n            \'--target\',\n            str(envd),\n            str(whl),\n        ],\n        check=True,\n    )\n\n    out = _P(\'/kaggle/working/_coat_arm.csv\')\n\n    import inspect\n    runtime_path = _P(\'/kaggle/working/input_resgated_runtime.py\')\n    helper_source = \'import numpy as np\\n\' + \'\\n\'.join(inspect.getsource(f) for f in\n        (_dense_allocate, _dense_unique_linspace, _dense_coat_specs))\n    runtime_sha = _asset_write_coat_runtime(art,runtime_path,helper_source)\n\n    child = (\n        "import sys, json, os\\n"\n        f"sys.path.insert(0, {str(envd)!r})\\n"\n        f"sys.path.insert(0, {str(art)!r})\\n"\n        "import cv2; assert cv2.__version__ == \'4.12.0\', cv2.__version__\\n"\n        "import torch; assert torch.cuda.device_count() == 2\\n"\n        "sys.path.insert(0, \'/kaggle/working\')\\n"\n        "import input_resgated_runtime as rt\\n"\n        "assert rt.base.cv2.__version__ == \'4.12.0\'\\n"\n        "from pathlib import Path\\n"\n        "r = rt.run_submission("\n        "competition_root=Path(os.environ[\'RSNA_COMP_ROOT\']) "\n        "if os.environ.get(\'RSNA_COMP_ROOT\') "\n        "else rt.base.find_competition_root(),\\n"\n        f"    artifact_root=Path({str(art)!r}), "\n        f"output_path=Path({str(out)!r}),\\n"\n        "    gpu_batch_studies=2, backbone_micro_images=8)\\n"\n        "assert r[\'status\'] == rt.SUBMISSION_STATUS\\n"\n        "assert r[\'models\'] == 3\\n"\n        "assert [i[\'epoch\'] for i in r[\'checkpoints\']] == [4, 6, 8]\\n"\n        "print(\'[coat-child] fallback_studies\', r[\'fallback_studies\'], flush=True)\\n"\n        "Path(\'/kaggle/working/_coat_arm_receipt.json\').write_text("\n        "json.dumps(r, indent=2))\\n"\n    )\n\n    env = dict(_o.environ)\n    env[\'CUDNN_CONV_WSCAP_DBG\'] = \'1024\'\n    env[\'RSNA_COMP_ROOT\'] = _KE_NS[\'find_test_root\']()\n    env[\'PYTHONPATH\'] = f"{envd}:{art}:" + env.get(\'PYTHONPATH\', \'\')\n\n    import json as _j\n    _coat_resgated_ok = _coat_g96_ok = _coat_d4_ok = _coat_rv1_ok = False\n    _coat_receipt = _g96_receipt = _rv1_receipt = None\n    g96_out = d4_out = rv1_out = None\n    def _run_residual_reader():\n        nonlocal _coat_resgated_ok, _coat_g96_ok, _coat_d4_ok, _coat_rv1_ok, _coat_receipt, _g96_receipt, _rv1_receipt, g96_out, d4_out, rv1_out\n        rsna_phase(\'residual_coat\', \'START\')\n        try:\n            _run_required_child([_sy.executable, \'-c\', child], env,\n                                _P(\'/kaggle/working/resgated_runtime.log\'))\n            _coat_receipt = _j.loads(_P(\'/kaggle/working/_coat_arm_receipt.json\').read_text())\n            if int(_coat_receipt.get(\'fallback_studies\', 0) or 0):\n                rsna_event(\'coat_resgated_fallback_studies\', count=int(_coat_receipt[\'fallback_studies\']), failures=list(_coat_receipt.get(\'failures\', []))[:20])\n            _coat_resgated_ok = True\n        except Exception as _coat_exc:\n            rsna_event(\'coat_resgated_child_failed\', error=f\'{type(_coat_exc).__name__}: {str(_coat_exc)[:1500]}\')\n            print(f\'[coat-arm] residual CoAt child FAILED; continuing without it (flagged): {type(_coat_exc).__name__}\', flush=True)\n            _coat_resgated_ok = False\n        rsna_phase(\'residual_coat\', \'COMPLETE\', ok=_coat_resgated_ok)\n    def _run_global96_reader():\n        nonlocal _coat_resgated_ok, _coat_g96_ok, _coat_d4_ok, _coat_rv1_ok, _coat_receipt, _g96_receipt, _rv1_receipt, g96_out, d4_out, rv1_out\n        G96_MAN_SHA = \'015f09030e76f86274cadae40777b3c3cf5c19f8c835d5ad88e15db41903779e\'\n        G96_TIMM_SHA = \'888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0\'\n        g96_out = _P(\'/kaggle/working/g96_input/_coat_g96.csv\')\n        g96_out.parent.mkdir(exist_ok=True)\n        rsna_phase(\'global96_coat\', \'START\')\n        try:\n            g96_art = find(\'coatnet_global96_top3_manifest.json\', G96_MAN_SHA).parent\n            g96_timm = g96_art / \'timm-1.0.22-py3-none-any.whl\'\n            if sha(g96_timm) != G96_TIMM_SHA:\n                raise RuntimeError(\'Global96 timm wheel changed\')\n            g96_env = _P(\'/kaggle/working/_g96_env\')\n            _sp.run(\n                [_sy.executable, \'-m\', \'pip\', \'install\', \'--no-deps\', \'--quiet\',\n                 \'--target\', str(g96_env), str(g96_timm)],\n                check=True,\n            )\n            g96_child = (\n                "import sys, json, os\\n"\n                f"sys.path.insert(0, {str(g96_env)!r})\\n"\n                f"sys.path.insert(0, {str(envd)!r})\\n"\n                f"sys.path.insert(0, {str(g96_art)!r})\\n"\n                "import cv2; assert cv2.__version__ == \'4.12.0\', cv2.__version__\\n"\n                "import timm; assert timm.__version__ == \'1.0.22\', timm.__version__\\n"\n                "import torch; assert torch.cuda.device_count() == 2\\n"\n                "import coatnet_global96_baseline_top3_inference as rt\\n"\n                "assert rt.base.cv2.__version__ == \'4.12.0\'\\n"\n                "assert rt.MAX_EVAL_WINDOWS == 94, rt.MAX_EVAL_WINDOWS\\n"\n                "from pathlib import Path\\n"\n                "r = rt.run_submission("\n                "competition_root=Path(os.environ[\'RSNA_COMP_ROOT\']) "\n                "if os.environ.get(\'RSNA_COMP_ROOT\') "\n                "else rt.base.find_competition_root(),\\n"\n                f"    artifact_root=Path({str(g96_art)!r}), "\n                f"output_path=Path({str(g96_out)!r}),\\n"\n                "    gpu_batch_studies=2, backbone_micro_images=8)\\n"\n                "assert r[\'status\'] == rt.SUBMISSION_STATUS\\n"\n                "assert r[\'models\'] == 3\\n"\n                "assert [i[\'epoch\'] for i in r[\'checkpoints\']] == [16, 23, 18]\\n"\n                "print(\'[coat-child] global96 fallback_studies\', r[\'fallback_studies\'], flush=True)\\n"\n                "Path(\'/kaggle/working/_coat_g96_receipt.json\').write_text("\n                "json.dumps(r, indent=2))\\n"\n            )\n            g96_env_vars = dict(_o.environ)\n            g96_env_vars[\'CUDNN_CONV_WSCAP_DBG\'] = \'1024\'\n            g96_env_vars[\'RSNA_COMP_ROOT\'] = _KE_NS[\'find_test_root\']()\n            g96_env_vars[\'PYTHONPATH\'] = f"{g96_env}:{envd}:{g96_art}:" + g96_env_vars.get(\'PYTHONPATH\', \'\')\n            _run_required_child([_sy.executable, \'-c\', g96_child], g96_env_vars,\n                                _P(\'/kaggle/working/global96_runtime.log\'))\n            _g96_receipt = _j.loads(_P(\'/kaggle/working/_coat_g96_receipt.json\').read_text())\n            if int(_g96_receipt.get(\'fallback_studies\', 0) or 0):\n                rsna_event(\'coat_global96_fallback_studies\', count=int(_g96_receipt[\'fallback_studies\']))\n            _coat_g96_ok = True\n        except Exception as _g96_exc:\n            rsna_event(\'coat_global96_child_failed\', error=f\'{type(_g96_exc).__name__}: {str(_g96_exc)[:1500]}\')\n            print(f\'[coat-arm] Global96 CoAt child FAILED; continuing without it (flagged): {type(_g96_exc).__name__}\', flush=True)\n            _coat_g96_ok = False\n        rsna_phase(\'global96_coat\', \'COMPLETE\', ok=_coat_g96_ok)\n    def _run_d4_reader():\n        nonlocal _coat_resgated_ok, _coat_g96_ok, _coat_d4_ok, _coat_rv1_ok, _coat_receipt, _g96_receipt, _rv1_receipt, g96_out, d4_out, rv1_out\n        rsna_phase(\'d4\', \'START\')\n        d4_out = _P(\'/kaggle/working/d4_input\')/\'d4.csv\'\n        try:\n            d4_manifest = _asset_find_asset(\'coatnet_pairfilm_manifest.json\',\n                \'7ada0605bca6b0530569c6454e988ace479606a3328ed591d090e5764fea661d\')\n            timm_whl = d4_manifest.parent/\'timm-1.0.22-py3-none-any.whl\'\n            if sha(timm_whl) != \'888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0\':\n                raise RuntimeError(\'D4 timm wheel changed\')\n            d4_env = _P(\'/kaggle/working/_d4_env\')\n            _sp.run([_sy.executable,\'-m\',\'pip\',\'install\',\'--no-deps\',\'--quiet\',\n                \'--target\',str(d4_env),str(timm_whl)],check=True)\n            d4_dir = _P(\'/kaggle/working/d4_input\')\n            d4_dir.mkdir(exist_ok=True)\n            _asset_run_d4(d4_manifest.parent,d4_env,_KE_NS[\'find_test_root\'](),d4_out)\n            _coat_d4_ok = True\n        except Exception as _d4_exc:\n            rsna_event(\'coat_d4_child_failed\', error=f\'{type(_d4_exc).__name__}: {str(_d4_exc)[:1500]}\')\n            print(f\'[coat-arm] D4 CoAt child FAILED; continuing without it (flagged): {type(_d4_exc).__name__}\', flush=True)\n            _coat_d4_ok = False\n        rsna_phase(\'d4\', \'COMPLETE\', ok=_coat_d4_ok)\n    def _run_repairv1_reader():\n        nonlocal _coat_resgated_ok, _coat_g96_ok, _coat_d4_ok, _coat_rv1_ok, _coat_receipt, _g96_receipt, _rv1_receipt, g96_out, d4_out, rv1_out\n        RV1_MAN_SHA = \'99288a9a42f36ba544d7b0ad2fe645e3b1fc9c507c23802c16ca0d2e5085f684\'\n        rv1_out = _P(\'/kaggle/working/_coat_repairv1.csv\')\n        rsna_phase(\'repairv1_coat\', \'START\')\n        try:\n            rv1_art = find(\'coatnet_repairv1_top3_manifest.json\', RV1_MAN_SHA).parent\n            rv1_child = (\n                "import sys, json, os\\n"\n                f"sys.path.insert(0, {str(envd)!r})\\n"\n                f"sys.path.insert(0, {str(rv1_art)!r})\\n"\n                "import cv2; assert cv2.__version__ == \'4.12.0\', cv2.__version__\\n"\n                "import torch; assert torch.cuda.device_count() == 2\\n"\n                "import coatnet_repairv1_top3_inference as rt\\n"\n                "assert rt.base.cv2.__version__ == \'4.12.0\'\\n"\n                "from pathlib import Path\\n"\n                "r = rt.run_submission("\n                "competition_root=Path(os.environ[\'RSNA_COMP_ROOT\']) "\n                "if os.environ.get(\'RSNA_COMP_ROOT\') "\n                "else rt.base.find_competition_root(),\\n"\n                f"    artifact_root=Path({str(rv1_art)!r}), "\n                f"output_path=Path({str(rv1_out)!r}),\\n"\n                "    gpu_batch_studies=2, backbone_micro_images=8)\\n"\n                "assert r[\'status\'] == rt.SUBMISSION_STATUS\\n"\n                "assert r[\'models\'] == 3\\n"\n                "assert [i[\'epoch\'] for i in r[\'checkpoints\']] == [12, 7, 11]\\n"\n                "print(\'[coat-child] repairv1 fallback_studies\', r.get(\'fallback_studies\', 0), flush=True)\\n"\n                "Path(\'/kaggle/working/_coat_repairv1_receipt.json\').write_text("\n                "json.dumps(r, indent=2))\\n"\n            )\n            rv1_env_vars = dict(_o.environ)\n            rv1_env_vars[\'CUDNN_CONV_WSCAP_DBG\'] = \'1024\'\n            rv1_env_vars[\'RSNA_COMP_ROOT\'] = _KE_NS[\'find_test_root\']()\n            rv1_env_vars[\'PYTHONPATH\'] = f"{envd}:{rv1_art}:" + rv1_env_vars.get(\'PYTHONPATH\', \'\')\n            _run_required_child([_sy.executable, \'-c\', rv1_child], rv1_env_vars,\n                                _P(\'/kaggle/working/repairv1_runtime.log\'))\n            _rv1_receipt = _j.loads(_P(\'/kaggle/working/_coat_repairv1_receipt.json\').read_text())\n            if int(_rv1_receipt.get(\'fallback_studies\', 0) or 0):\n                rsna_event(\'coat_repairv1_fallback_studies\', count=int(_rv1_receipt[\'fallback_studies\']))\n            _coat_rv1_ok = True\n        except Exception as _rv1_exc:\n            rsna_event(\'coat_repairv1_child_failed\', error=f\'{type(_rv1_exc).__name__}: {str(_rv1_exc)[:1500]}\')\n            print(f\'[coat-arm] Repair-v1 CoAt child FAILED; continuing without it (flagged): {type(_rv1_exc).__name__}\', flush=True)\n            _coat_rv1_ok = False\n        rsna_phase(\'repairv1_coat\', \'COMPLETE\', ok=_coat_rv1_ok)\n\n\n    _parallel_pairs = _o.environ.get(\n        \'RSNA_PARALLEL_COAT_READERS\',\n        \'1\' if len(_RSNA_TEST_IDS) <= 48 else \'0\',\n    ) == \'1\'\n    if _parallel_pairs:\n        for _pair in ((_run_residual_reader, _run_d4_reader),\n                      (_run_global96_reader, _run_repairv1_reader)):\n            with _KeThreadPool(max_workers=2) as _pool:\n                _jobs = [_pool.submit(_reader) for _reader in _pair]\n                for _job in _jobs:\n                    _job.result()\n    else:\n        for _reader in (_run_residual_reader, _run_global96_reader,\n                        _run_d4_reader, _run_repairv1_reader):\n            _reader()\n    pub = _pd.read_csv(raptor, dtype={\'StudyInstanceUID\': str})\n    pub = rsna_frame(pub,_RSNA_TEST_IDS,_KE_LAB,\'public Raptor\')\n    lab = [c for c in pub.columns if c != \'StudyInstanceUID\']\n    import numpy as _np\n    # Retain BTKD inner blend coefficients; input/new family scores are unvalidated.\n    private_alpha = 0.4\n    public_rank = pub[lab].rank(method=\'average\', pct=True)\n    _members = []\n    if _coat_resgated_ok:\n        try:\n            ours = _pd.read_csv(out, dtype={\'StudyInstanceUID\': str})\n            ours = rsna_frame(ours,_RSNA_TEST_IDS,_KE_LAB,\'resgated CoAt\')\n            if list(ours.columns) != list(pub.columns):\n                raise RuntimeError(\'coat arm column drift\')\n            ours = ours.set_index(\'StudyInstanceUID\').reindex(pub.StudyInstanceUID.astype(str).tolist()).reset_index()\n            if ours[lab].isna().any().any():\n                raise RuntimeError(\'coat arm does not cover every study\')\n            rsna_save_predictions(\'coat_resgated_rank\',_RSNA_TEST_IDS,ours[lab].to_numpy(),lab)\n            _members.append((\'resgated_top3\', ours[lab].rank(method=\'average\', pct=True)))\n        except Exception as _exc:\n            rsna_event(\'coat_resgated_output_rejected\', error=f\'{type(_exc).__name__}: {str(_exc)[:500]}\')\n    if _coat_g96_ok:\n        try:\n            g96 = _pd.read_csv(g96_out, dtype={\'StudyInstanceUID\': str})\n            g96 = rsna_frame(g96,_RSNA_TEST_IDS,_KE_LAB,\'Global96 CoAt\')\n            if list(g96.columns) != list(pub.columns):\n                raise RuntimeError(\'global96 arm column drift\')\n            g96 = g96.set_index(\'StudyInstanceUID\').reindex(pub.StudyInstanceUID.astype(str).tolist()).reset_index()\n            if g96[lab].isna().any().any():\n                raise RuntimeError(\'global96 arm does not cover every study\')\n            rsna_save_predictions(\'coat_global96_rank\',_RSNA_TEST_IDS,g96[lab].to_numpy(),lab)\n            _members.append((\'global96_top3\', g96[lab].rank(method=\'average\', pct=True)))\n        except Exception as _exc:\n            rsna_event(\'coat_global96_output_rejected\', error=f\'{type(_exc).__name__}: {str(_exc)[:500]}\')\n    if _coat_d4_ok:\n        try:\n            d4 = _pd.read_csv(d4_out,dtype={\'StudyInstanceUID\':str})\n            d4 = rsna_frame(d4,_RSNA_TEST_IDS,_KE_LAB,\'D4 CoAt\')\n            rsna_save_predictions(\'coat_d4_rank\',_RSNA_TEST_IDS,d4[lab].to_numpy(),lab)\n            if d4.StudyInstanceUID.duplicated().any() or list(d4.columns)!=list(pub.columns):\n                raise RuntimeError(\'D4 schema/UID drift\')\n            d4 = d4.set_index(\'StudyInstanceUID\').reindex(pub.StudyInstanceUID.astype(str))\n            if d4[lab].isna().any().any():\n                raise RuntimeError(\'D4 missing predictions\')\n            _members.append((\'d4_swa3\', d4[lab].reset_index(drop=True).rank(method=\'average\',pct=True)))\n        except Exception as _exc:\n            rsna_event(\'coat_d4_output_rejected\', error=f\'{type(_exc).__name__}: {str(_exc)[:500]}\')\n    if _coat_rv1_ok:\n        try:\n            rv1 = _pd.read_csv(rv1_out, dtype={\'StudyInstanceUID\': str})\n            rv1 = rsna_frame(rv1,_RSNA_TEST_IDS,_KE_LAB,\'Repair-v1 CoAt\')\n            if list(rv1.columns) != list(pub.columns):\n                raise RuntimeError(\'repairv1 arm column drift\')\n            rv1 = rv1.set_index(\'StudyInstanceUID\').reindex(pub.StudyInstanceUID.astype(str).tolist()).reset_index()\n            if rv1[lab].isna().any().any():\n                raise RuntimeError(\'repairv1 arm does not cover every study\')\n            rsna_save_predictions(\'coat_repairv1_rank\',_RSNA_TEST_IDS,rv1[lab].to_numpy(),lab)\n            _members.append((\'repairv1_top3\', rv1[lab].rank(method=\'average\', pct=True)))\n        except Exception as _exc:\n            rsna_event(\'coat_repairv1_output_rejected\', error=f\'{type(_exc).__name__}: {str(_exc)[:500]}\')\n    if not _members:\n        rsna_event(\'coat_family_unavailable\', note=\'public Raptor kept as the CoAt/Raptor input (flagged)\')\n        print(\'[coat-arm] no CoAt family member available; public Raptor retained (flagged)\', flush=True)\n        return 0\n    # Family reduction: rank of the members\' PROBABILITY mean, not the mean of their ranks.\n    # Each child already writes its probability_mean (or the raw per-checkpoint probabilities)\n    # beside its rank CSV, in the submission\'s own label order.  If any member\'s probabilities\n    # are unavailable, fall back to the previous rank mix rather than drop the member.\n    _PROB_FILES = {\n        \'resgated_top3\': _P(\'/kaggle/working/coat_resgated_ep10_top3_predictions.npz\'),\n        \'global96_top3\': _P(\'/kaggle/working/g96_input/coatnet_global96_top3_predictions.npz\'),\n        \'d4_swa3\': _P(\'/kaggle/working/d4_input/coatnet_d4_depthzone_swa3_predictions.npz\'),\n        \'repairv1_top3\': _P(\'/kaggle/working/coatnet_repairv1_top3_predictions.npz\'),\n    }\n\n    def _coat_member_probability(name):\n        with _np.load(_PROB_FILES[name], allow_pickle=False) as z:\n            uids = [str(u) for u in z[\'study_uids\'].astype(str)]\n            if \'probability_mean\' in z.files:\n                prob = _np.asarray(z[\'probability_mean\'], dtype=_np.float64)\n            else:\n                prob = _np.asarray(z[\'raw_probabilities\'], dtype=_np.float64).mean(axis=0)\n        if prob.shape != (len(uids), len(lab)):\n            raise RuntimeError(f\'{name} probability shape drift {prob.shape}\')\n        frame = _pd.DataFrame(prob, columns=lab, index=uids)\n        frame = frame.reindex(pub.StudyInstanceUID.astype(str).tolist())\n        values = frame.to_numpy(_np.float64)\n        if frame.isna().any().any():\n            raise RuntimeError(f\'{name} probabilities do not cover every study\')\n        if not _np.isfinite(values).all() or (values < 0).any() or (values > 1).any():\n            raise RuntimeError(f\'{name} probabilities are not finite in [0,1]\')\n        return frame.reset_index(drop=True)\n\n    _probs = {}\n    for _name, _ in _members:\n        try:\n            _probs[_name] = _coat_member_probability(_name)\n        except Exception as _exc:\n            rsna_event(\'coat_member_probability_unavailable\', member=_name,\n                       error=f\'{type(_exc).__name__}: {str(_exc)[:300]}\')\n    if len(_members) >= 2 and len(_probs) == len(_members):\n        _mean = sum(_probs[_n] for _n, _ in _members) / float(len(_members))\n        private_rank = _mean.rank(method=\'average\', pct=True)\n        _family_reduction = \'rank_of_member_probability_mean\'\n    elif len(_members) >= 2:\n        private_rank = _asset_half_rank_mix(*[_member for _, _member in _members])\n        _family_reduction = \'rank_sum_fallback_probabilities_unavailable\'\n    else:\n        rsna_event(\'coat_family_partial\', member=_members[0][0])\n        private_rank = _members[0][1]\n        _family_reduction = \'single_member_rank\'\n    print(\'[coat-arm] family reduction: \' + _family_reduction, flush=True)\n    print(\'[coat-arm] CoAt family members: \' + \', \'.join(m for m, _ in _members), flush=True)\n    family = pub.copy()\n    family[lab] = private_rank\n    family.to_csv(\'/kaggle/working/_coat_family_rank.csv\', index=False)\n    hybrid = pub.copy()\n    hybrid[lab] = (1.0 - private_alpha) * public_rank + private_alpha * private_rank\n    if not _np.isfinite(hybrid[lab].to_numpy(_np.float64)).all():\n        raise RuntimeError(\'CoAt/Raptor hybrid contains non-finite values\')\n    tmp = raptor.with_name(\'.raptor_coat_hybrid.csv\')\n    hybrid.to_csv(tmp, index=False)\n    _o.replace(tmp, raptor)\n    raptor.with_name(\'_coat_raptor_blend_receipt.json\').write_text(_j.dumps({\n        \'contract\': \'public_raptor_private_residual_coat_v32_no_inner_rerank_v1\', \'inner_rerank\': False, \'private_alpha\': private_alpha,\n        \'within_coat\': {m: 1.0 / len(_members) for m, _ in _members} if len(_members) >= 2 else {_members[0][0]: 1.0}, \'public_raptor_alpha\': 1.0 - private_alpha,\n        \'study_count\': len(pub), \'finding_specific_weights\': False, \'members\': [m for m, _ in _members], \'family_reduction\': _family_reduction}, indent=2, sort_keys=True) + \'\\n\')\n    return len(pub)\n\n_coat_n = _coat_substitute()\nprint(f\'[coat-arm] equal-rank CoAt family; BTKD32 no-inner-rerank; {_coat_n} studies\',flush=True)\n\n_ke_theirs = _ke_pd.read_csv(\'/kaggle/working/_raptor.csv\',\n                             dtype={\'StudyInstanceUID\': str})\nassert list(_ke_theirs.columns) == list(_ke_ours.columns), \'column drift\'\n_ke_theirs = _ke_theirs.set_index(\'StudyInstanceUID\').reindex(\n    _ke_ours[\'StudyInstanceUID\']).reset_index()\nassert _ke_theirs[_KE_LAB].notna().all().all(), \'study identity drift\'\n\n\n_ke_tr = _ke_ours[_KE_LAB].rank(method=\'average\', pct=True)\n_ke_cr = _ke_theirs[_KE_LAB].copy()\n_blend_transformer = _ke_ours.copy()\n_blend_coatnet = _ke_theirs.copy()\n_blend_labels = list(_KE_LAB)\n_blend_tr = _ke_tr.copy()\n_blend_cr = _ke_cr.copy()\n_coatnet_weight = {label: 0.60 for label in _blend_labels}\n_coatnet_weight.update({\n    \'ACL\': 0.75,\n    \'Medial Meniscus\': 0.80,\n    \'Lateral Meniscus\': 1.00,\n    \'Lateral OA\': 0.75,\n    \'Fracture\': 0.75,\n})\n_blend_output = _blend_transformer.copy()\nfor _blend_label in _blend_labels:\n    _blend_w = float(_coatnet_weight[_blend_label])\n    _blend_output[_blend_label] = (\n        (1.0 - _blend_w) * _blend_tr[_blend_label]\n        + _blend_w * _blend_cr[_blend_label]\n    )\n_blend_output[_blend_labels] = _blend_output[_blend_labels].rank(\n    method=\'average\', pct=True\n)\nassert _ke_np.isfinite(\n    _blend_output[_blend_labels].to_numpy(_ke_np.float64)\n).all()\n_blend_output.to_csv(_ke_primary, index=False)\n', globals())


## Final checks

The notebook publishes `submission.csv` only after schema, study order, finite
values, checkpoint identities, model counts, and fallback counts are recorded.


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.run_cell(29, '# Publish submission.csv only after the entire BTKD + CoAt family graph succeeds.\nimport platform\n_release_path=Path(\'/kaggle/working/_pipeline_stage.csv\')\n_release_df=rsna_frame(pd.read_csv(_release_path,dtype={\'StudyInstanceUID\':str}),_RSNA_TEST_IDS,_RSNA_LABELS,\'FINAL\')\nif _DINOV2_MATCHED_MEMBERS!=20:raise RuntimeError(\'final DINO member count mismatch\')\nif not globals().get(\'V18_CALIBRATOR_APPLIED\'):raise RuntimeError(\'BTKD calibrator was not applied\')\n_expected_preparations={(name,uid) for name in (\'maxspan-v5\',\'native384dense-v10\',\'native384-v8\') for uid in _RSNA_TEST_IDS}\nif _ke_input_ids != _expected_preparations:\n    rsna_event(\'raptor_preparation_identities_incomplete\', missing=len(_expected_preparations - _ke_input_ids), extra=len(_ke_input_ids - _expected_preparations))\nrsna_save_predictions(\'final\',_RSNA_TEST_IDS,_release_df[_RSNA_LABELS].to_numpy(),_RSNA_LABELS)\nimport json as _coat_audit_json\n_coat_receipt_path = Path(\'/kaggle/working/_coat_raptor_blend_receipt.json\')\n_coat_blend_receipt = (_coat_audit_json.loads(_coat_receipt_path.read_text()) if _coat_receipt_path.exists() else {})\n_coat_members = _coat_blend_receipt.get(\'members\', [])\n_coat_weights = _coat_blend_receipt.get(\'within_coat\', {})\n_RSNA_AUDIT.update(status=\'COMPLETE\',dino_members=20,a5_folds=5,raptor_views=4,coats={\'resgated_epochs\':[4,6,8],\'d4_parent_swa_epochs\':[15,16],\'d4_adapter_swa_epochs\':[11,9,5],\'global96_epochs\':[16,23,18],\'repairv1_epochs\':[12,7,11]},family_mix=[_coat_weights[m] for m in _coat_members],inner_public_coat_mix=([.6,.4] if _coat_members else [1.0,0.0]),inner_rerank=False,coat_readers_parallel_pairs=(os.environ.get(\'RSNA_PARALLEL_COAT_READERS\', \'1\' if len(_RSNA_TEST_IDS) <= 48 else \'0\') == \'1\'),coat_family_members=len(_coat_members),family_reduction=_coat_blend_receipt.get(\'family_reduction\', \'public_raptor_only\'),outer_weights=_coatnet_weight,bt_input_sha256=\'eb9c51cf04ddbd4e923f1a5c278e7b019036db9e67ecc20bea6661bd536bb303\',v555_input_sha256=\'22d71a1311ef8282e26a0ca498bed57100da5152352deae7cbdaeca4ab8539c9\',elapsed_seconds=time.time()-T0,score_recovery_claimed=False)\nfor cache_file in list(_RSNA_CACHE_FILES):Path(cache_file).unlink(missing_ok=True)\n_final=Path(\'/kaggle/working/submission.csv\');_tmp=_final.with_suffix(\'.csv.tmp\')\nwith _tmp.open(\'w\') as _out_handle:\n    _release_df.to_csv(_out_handle,index=False)\n    _out_handle.flush();os.fsync(_out_handle.fileno())\n_RSNA_AUDIT[\'submission_sha256\']=rsna_sha(_tmp)\nos.replace(_tmp,_final)\nrsna_json(\'/kaggle/working/btkd_v559_complete.json\',_RSNA_AUDIT)\nprint(f\'COMPLETE: {_final}, {len(_release_df)} studies, SHA256={_RSNA_AUDIT["submission_sha256"]}\')\n', globals())


In [ ]:
import builtins as _p003_builtins
_p003_builtins._rsna_p003_guard.finish(globals())
